# PCG Baseline — Focused Measurement Notebook

**Purpose:** verify the archived PCG experiment artifacts and measure single-sample trajectory-predictor latency on the current Colab GPU.

This is a trimmed measurement-only notebook. It deliberately excludes the old exploratory and reproduction workflow.

## What this notebook does
1. Checks the active Colab runtime.
2. Mounts Google Drive and verifies the existing repository/data/artifact locations.
3. Records the runtime and artifact fingerprints.
4. Reconstructs the retrospective measurements from the original notebook's saved outputs.
5. Optionally benchmarks the archived epoch-4 predictor checkpoint.
6. Saves plots and a hash-indexed output manifest in a new, timestamped Drive folder.

## What it does NOT do
- Install or upgrade Python packages.
- Create or activate a separate Python environment.
- Retrain the PCG model.
- Run the historical `test.py` evaluation.
- Modify, move, or overwrite the archived checkpoints, configs, datasets, or predictions.

## Execution order
Run cells from the top through **Artifact integrity manifest** first. Review the manifest. If the epoch-4 checkpoint and experiment configuration match their recorded SHA-256 hashes, change `RUN_INFERENCE_BENCHMARK` to `True` in the dedicated control cell, then run the benchmark, visualisation, and output-index cells.

**Do not run the benchmark if a critical checkpoint/configuration hash is missing or mismatched.**


In [1]:
import sys, platform, os, importlib.util
import torch

print("Python:", sys.version)
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("Torch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime is not enabled. In Colab choose Runtime > Change runtime type > T4 GPU, then reconnect.")

gpu = torch.cuda.get_device_properties(0)
print("GPU:", gpu.name)
print("VRAM (MiB):", round(gpu.total_memory / 2**20))
print("Compute capability:", f"{gpu.major}.{gpu.minor}")
if "T4" not in gpu.name:
    print("WARNING: This is not a Tesla T4. The benchmark will still record the actual GPU.")
try:
    import psutil
    print("System RAM (GiB):", round(psutil.virtual_memory().total / 2**30, 2))
except Exception:
    print("psutil is unavailable; RAM details will be omitted from the fingerprint.")


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Executable: /usr/bin/python3
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
PyTorch: 2.14.1+cu130
Torch CUDA build: 13.0
CUDA available: True
GPU: Tesla T4
VRAM (MiB): 14913
Compute capability: 7.5
System RAM (GiB): 12.67


In [2]:
import os
from google.colab import drive

if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
else:
    print("Google Drive is already mounted.")

assert os.path.isdir("/content/drive/MyDrive"), "Google Drive mount was not found."
print("Drive mount ready.")


Mounted at /content/drive
Drive mount ready.


In [3]:
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Golchoubian/PolarCollisionGrid-PedestrianTrajectoryPrediction.git"
EXPECTED_COMMIT = "0735a0f3b9f2186cb4e2be614eaf9c98d25f0de5"
REPO = "/content/PolarCollisionGrid-PedestrianTrajectoryPrediction"

PROJECT_DIR = "/content/drive/MyDrive/PCG-Research"
DRIVE_RESULTS = os.path.join(PROJECT_DIR, "runs", "pcg_baseline_01", "Store_Results")
DRIVE_BACKUP = "/content/drive/MyDrive/PCG_Experiments/PCG_T4_5_Epoch_Run"
# Read the archived Store_Results directly from Drive; do not create a symlink or move files.
STORE = DRIVE_RESULTS

def run_checked(args):
    result = subprocess.run(args, text=True, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed: {' '.join(args)}\n"
            f"stdout:\n{result.stdout}\nstderr:\n{result.stderr}"
        )
    return result.stdout.strip()

if not os.path.isdir(os.path.join(REPO, ".git")):
    if os.path.exists(REPO):
        raise RuntimeError(f"{REPO} exists but is not a Git repository. Inspect it; this notebook will not delete or replace it.")
    run_checked(["git", "clone", REPO_URL, REPO])
    run_checked(["git", "-C", REPO, "checkout", "--detach", EXPECTED_COMMIT])

actual_commit = run_checked(["git", "-C", REPO, "rev-parse", "HEAD"])
if actual_commit != EXPECTED_COMMIT:
    raise RuntimeError(
        "Repository commit differs from the archived measurement baseline.\n"
        f"Expected: {EXPECTED_COMMIT}\nActual:   {actual_commit}\n"
        "No automatic reset or checkout was performed. Inspect the repository before continuing."
    )

required_repo_files = [
    "train.py", "test.py", "utils/DataLoader.py", "utils/helper.py",
    "utils/Interaction.py", "utils/grid.py", "model/model_CollisionGrid.py",
    "Data/HBS/hbs.csv", "Data/train/trajectories.cpkl",
    "Data/train/trajectories_train.cpkl", "Data/test/trajectories_test.cpkl",
]
missing_repo = [f for f in required_repo_files if not os.path.isfile(os.path.join(REPO, f))]
if missing_repo:
    raise FileNotFoundError("Required repository/data files are missing:\n  - " + "\n  - ".join(missing_repo))

required_store = [
    "model/SOCIALLSTM_lstm_model_4.tar",
    "model/config.pkl",
    "model/CollisionGrid/SOCIALLSTM_lstm_model_4.tar",
    "model/CollisionGrid/config.pkl",
    "plot/test/test_results.pkl",
    "log/log_curve.txt",
]
required_backup = [
    "SOCIALLSTM_lstm_model_4.tar",
    "config.pkl",
    "CollisionGrid_original_config.pkl",
    "SOCIALLSTM_lstm_model_181.tar",
    "test_results_epoch4.pkl",
]
if not os.path.isdir(STORE):
    raise FileNotFoundError(f"Archived Store_Results directory not found: {STORE}")
if not os.path.isdir(DRIVE_BACKUP):
    raise FileNotFoundError(f"Archived Drive backup directory not found: {DRIVE_BACKUP}")

missing_store = [f for f in required_store if not os.path.isfile(os.path.join(STORE, f))]
missing_backup = [f for f in required_backup if not os.path.isfile(os.path.join(DRIVE_BACKUP, f))]
if missing_store or missing_backup:
    msg = []
    if missing_store:
        msg.append("Missing Store_Results files:\n  - " + "\n  - ".join(missing_store))
    if missing_backup:
        msg.append("Missing Drive backup files:\n  - " + "\n  - ".join(missing_backup))
    raise FileNotFoundError("\n".join(msg))

os.chdir(REPO)
print("Repository:", REPO)
print("Pinned commit:", actual_commit)
print("Working tree:", run_checked(["git", "-C", REPO, "status", "--short"]) or "clean")
print("Dataset/source preflight: PASS")
print("Drive Store_Results:", STORE)
print("Separate checkpoint backup:", DRIVE_BACKUP)
print("Archived artifact path preflight: PASS")


Repository: /content/PolarCollisionGrid-PedestrianTrajectoryPrediction
Pinned commit: 0735a0f3b9f2186cb4e2be614eaf9c98d25f0de5
Working tree: clean
Dataset/source preflight: PASS
Drive Store_Results: /content/drive/MyDrive/PCG-Research/runs/pcg_baseline_01/Store_Results
Separate checkpoint backup: /content/drive/MyDrive/PCG_Experiments/PCG_T4_5_Epoch_Run
Archived artifact path preflight: PASS


## Measurement setup

The next cell writes helper scripts and creates a **new, timestamped output directory**. It does not install packages or modify experiment artifacts.

In [4]:
# ==============================================================================================
# MEASUREMENT SETUP - no training, no package installation, no artifact modification.
# Repository and Drive paths are established by the preceding preflight cell.
# ==============================================================================================
import os, sys, json, time, pickle, platform, subprocess, shutil, hashlib, datetime, pathlib

# ---- benchmark settings -------------------------------------------------------------------------
PYTHON_FOR_NEW_RUNS = None       # recorded for provenance; benchmark runs in this notebook kernel
BENCH_WARMUP = 20                # warm-up trajectories excluded from timing
BENCH_MAX_TRAJECTORIES = None    # None -> all available HBS test trajectories
BENCH_SEED = 12345
TRAINING_OUTPUT_DIR = None       # historical training telemetry is not rerun in this notebook

# ---- fresh output directory for THIS measurement session -----------------------------------------
MEAS_RUN_ID = "meas_" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT_PARENT = "/content/drive/MyDrive/PCG_Experiments/measurements" if os.path.isdir("/content/drive/MyDrive") else "/content/pcg_measurements"
MEAS_DIR = os.path.join(OUT_PARENT, MEAS_RUN_ID)
WORK_ROOT = os.path.join("/content/pcg_runs", MEAS_RUN_ID)       # local disk (fast) for any new run's working directories
os.makedirs(OUT_PARENT, exist_ok=True)
os.makedirs(MEAS_DIR, exist_ok=False)                             # fails if it exists: never reuse or overwrite
TOOLS_DIR = os.path.join(MEAS_DIR, "tools")
os.makedirs(TOOLS_DIR, exist_ok=False)

# ---- helper sources (embedded so this notebook is self-contained; written exclusively, never over an existing file)
HELPER_SOURCES = {'measurement_lib.py': '"""Shared, dependency-light helpers for the PCG measurement notebook (stdlib only; '
                       'numpy optional).\n'
                       '\n'
                       'Nothing here trains a model or touches an experiment artifact. Everything that '
                       'writes uses exclusive creation\n'
                       '(`make_fresh_dir`, `write_new_json`) so an existing file or directory is never '
                       'overwritten.\n'
                       '"""\n'
                       '\n'
                       'import csv\n'
                       'import hashlib\n'
                       'import json\n'
                       'import math\n'
                       'import os\n'
                       'import re\n'
                       'import time\n'
                       'from datetime import datetime, timezone\n'
                       'from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple\n'
                       '\n'
                       'LIB_VERSION = "pcg-measurement-lib-1.0"\n'
                       '\n'
                       '\n'
                       '# ------------------------------------------------------------------ hashing / file '
                       'records\n'
                       'def sha256_file(path: str, chunk: int = 1 << 20) -> str:\n'
                       '    h = hashlib.sha256()\n'
                       '    with open(path, "rb") as fh:\n'
                       '        for block in iter(lambda: fh.read(chunk), b""):\n'
                       '            h.update(block)\n'
                       '    return h.hexdigest()\n'
                       '\n'
                       '\n'
                       'def utc_now_iso() -> str:\n'
                       '    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")\n'
                       '\n'
                       '\n'
                       'def file_record(path: str, label: str = "", expected_sha256: Optional[str] = None, '
                       'expected_source: Optional[str] = None) -> Dict[str, Any]:\n'
                       '    """Read-only description of one file. ``matches_previous_hash`` is None when no '
                       'previous hash is known."""\n'
                       '    rec: Dict[str, Any] = {"label": label, "path": path, "exists": '
                       'os.path.isfile(path)}\n'
                       '    try:\n'
                       '        rec["resolved_path"] = os.path.realpath(path)\n'
                       '    except OSError:\n'
                       '        rec["resolved_path"] = None\n'
                       '    if rec["exists"]:\n'
                       '        st = os.stat(path)\n'
                       '        rec["size_bytes"] = st.st_size\n'
                       '        rec["modified_utc"] = datetime.fromtimestamp(st.st_mtime, '
                       'timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")\n'
                       '        rec["sha256"] = sha256_file(path)\n'
                       '    else:\n'
                       '        rec["size_bytes"] = rec["modified_utc"] = rec["sha256"] = None\n'
                       '    rec["previous_sha256"] = expected_sha256\n'
                       '    rec["previous_hash_source"] = expected_source if expected_sha256 else None\n'
                       '    rec["matches_previous_hash"] = None if (expected_sha256 is None or not '
                       'rec["exists"]) else (rec["sha256"].lower() == expected_sha256.lower())\n'
                       '    return rec\n'
                       '\n'
                       '\n'
                       '# ----------------------------------------------------------------------- safe '
                       'writing\n'
                       'def make_fresh_dir(path: str) -> str:\n'
                       '    """Create ``path`` (and parents). Refuses an existing non-empty directory; an '
                       'existing EMPTY one is accepted."""\n'
                       '    if os.path.exists(path):\n'
                       '        if not os.path.isdir(path):\n'
                       '            raise FileExistsError("output path exists and is not a directory: %s" % '
                       'path)\n'
                       '        if os.listdir(path):\n'
                       '            raise FileExistsError("output directory is not empty (nothing is '
                       'overwritten or deleted): %s" % path)\n'
                       '    os.makedirs(path, exist_ok=True)\n'
                       '    return path\n'
                       '\n'
                       '\n'
                       'def write_new_json(path: str, obj: Any) -> str:\n'
                       '    """Exclusive create (\'x\'): raises FileExistsError instead of replacing a file. '
                       'Returns the SHA-256."""\n'
                       '    text = json.dumps(obj, indent=2, sort_keys=True, allow_nan=False, default=str) + '
                       '"\\n"\n'
                       '    with open(path, "x", encoding="utf-8", newline="\\n") as fh:\n'
                       '        fh.write(text)\n'
                       '    return hashlib.sha256(text.encode("utf-8")).hexdigest()\n'
                       '\n'
                       '\n'
                       'def write_new_csv(path: str, rows: Sequence[Mapping[str, Any]], fieldnames: '
                       'Sequence[str]) -> None:\n'
                       '    with open(path, "x", newline="", encoding="utf-8") as fh:\n'
                       '        w = csv.DictWriter(fh, fieldnames=list(fieldnames))\n'
                       '        w.writeheader()\n'
                       '        for r in rows:\n'
                       '            w.writerow({k: r.get(k) for k in fieldnames})\n'
                       '\n'
                       '\n'
                       '# ------------------------------------------------------------------------- '
                       'statistics\n'
                       'def percentile_nearest_rank(values: Sequence[float], q: float) -> float:\n'
                       '    if not values:\n'
                       '        raise ValueError("percentile of an empty sequence")\n'
                       '    if not 0.0 <= q <= 100.0:\n'
                       '        raise ValueError("q must be in [0, 100]")\n'
                       '    s = sorted(values)\n'
                       '    rank = max(1, int(math.ceil(q / 100.0 * len(s))))\n'
                       '    return s[min(rank, len(s)) - 1]\n'
                       '\n'
                       '\n'
                       'def describe(values: Sequence[float]) -> Dict[str, Any]:\n'
                       '    """count, mean, std (population), min, p50, p90, p95, p99, max. Percentiles are '
                       'NEAREST-RANK (no interpolation)."""\n'
                       '    v = [float(x) for x in values if x is not None and math.isfinite(float(x))]\n'
                       '    if not v:\n'
                       '        return {"count": 0}\n'
                       '    n = len(v)\n'
                       '    mean = sum(v) / n\n'
                       '    std = math.sqrt(sum((x - mean) ** 2 for x in v) / n)\n'
                       '    return {"count": n, "mean": mean, "std": std, "min": min(v), "p50": '
                       'percentile_nearest_rank(v, 50), "p90": percentile_nearest_rank(v, 90),\n'
                       '            "p95": percentile_nearest_rank(v, 95), "p99": percentile_nearest_rank(v, '
                       '99), "max": max(v), "percentile_method": "nearest_rank"}\n'
                       '\n'
                       '\n'
                       '# --------------------------------------------------------------------------- log '
                       'parsing\n'
                       '_TRAIN_RE = '
                       're.compile(r"^\\s*(\\d+)/(\\d+)\\s+\\(epoch\\s+(\\d+)\\),\\s*train_loss\\s*=\\s*(-?[\\d.]+),\\s*time/batch\\s*=\\s*([\\d.]+)\\s*$")\n'
                       '_GRID_RE = re.compile(r"grid calculation time for all the '
                       'data:\\s*([\\d.]+)\\s*seconds")\n'
                       '_TOTAL_RE = re.compile(r"The whole trainig time for\\s+(\\d+)\\s+iteraction '
                       'was\\s+([\\d.]+)\\s+seconds")\n'
                       '_TEST_RE = re.compile(r"Processed trajectory number\\s*:\\s*(\\d+)\\s+out '
                       'of\\s+(\\d+)\\s+trajectories in time\\s+([\\d.]+)")\n'
                       '\n'
                       '\n'
                       'def parse_train_log(text: str) -> Dict[str, Any]:\n'
                       '    """Parse train.py stdout. Only values that are literally printed are returned; '
                       'nothing is inferred."""\n'
                       '    batches = []\n'
                       '    for line in text.splitlines():\n'
                       '        m = _TRAIN_RE.match(line)\n'
                       '        if m:\n'
                       '            batches.append({"global_batch": int(m.group(1)), "total_batches": '
                       'int(m.group(2)), "epoch": int(m.group(3)),\n'
                       '                            "train_loss": float(m.group(4)), "time_per_batch_s": '
                       'float(m.group(5))})\n'
                       '    g, t = _GRID_RE.search(text), _TOTAL_RE.search(text)\n'
                       '    return {"batches": batches, "grid_calculation_s": float(g.group(1)) if g else '
                       'None,\n'
                       '            "total_training_s": float(t.group(2)) if t else None, "epochs_reported": '
                       'int(t.group(1)) if t else None}\n'
                       '\n'
                       '\n'
                       'def parse_test_log(text: str) -> Dict[str, Any]:\n'
                       '    rows = [{"trajectory": int(m.group(1)), "of": int(m.group(2)), "time_s": '
                       'float(m.group(3))} for m in _TEST_RE.finditer(text)]\n'
                       '    return {"trajectories": rows}\n'
                       '\n'
                       '\n'
                       'def per_epoch_from_batches(batches: Sequence[Mapping[str, Any]]) -> List[Dict[str, '
                       'Any]]:\n'
                       '    by: Dict[int, List[Mapping[str, Any]]] = {}\n'
                       '    for b in batches:\n'
                       '        by.setdefault(int(b["epoch"]), []).append(b)\n'
                       '    out = []\n'
                       '    for e in sorted(by):\n'
                       '        bs = by[e]\n'
                       '        t = [x["time_per_batch_s"] for x in bs]\n'
                       '        l = [x["train_loss"] for x in bs]\n'
                       '        out.append({"epoch": e, "n_batches": len(bs), "mean_loss": sum(l) / len(l), '
                       '"sum_batch_time_s": sum(t), "batch_time": describe(t)})\n'
                       '    return out\n',
 'inference_benchmark.py': '"""Single-sample PCG trajectory-predictor inference benchmark (notebook '
                           'companion; test.py is NOT modified or run).\n'
                           '\n'
                           'SCOPE (written into every output): this measures the PCG trajectory-PREDICTOR '
                           'code path only, one prediction sample per input\n'
                           '(``sample_size = 1``). It is NOT end-to-end video performance (no detection, '
                           'tracking or video decoding) and it is NOT a\n'
                           'deployment-valid latency: the baseline ``sample`` function and the preparation '
                           'steps still consume the future actor information\n'
                           'that the historical test path uses (see the Phase 2 Task 4 audit). The numbers '
                           'characterise this code path on the declared hardware.\n'
                           '\n'
                           'Relation to ``test.py`` (read at the commit recorded in the output): the '
                           'per-trajectory steps are REPLICATED here, step for step,\n'
                           'from ``test.py`` ("dense vector creation" ... "revert position change"), and the '
                           'baseline ``sample`` function is imported from\n'
                           '``test.py`` unchanged. Left out on purpose: the ground-truth scoring '
                           '(ADE/FDE/MHD/speed/heading/collision), the best-of-N\n'
                           'selection (best-of-N needs ground truth: that is the oracle) and the result '
                           'list. Because ``test.py`` never calls ``net.eval()``,\n'
                           'this benchmark does not either unless ``eval_mode=True`` is requested; the model '
                           'contains ``nn.Dropout`` so the baseline\n'
                           'prediction path runs with dropout ACTIVE. Both facts are recorded.\n'
                           '\n'
                           'Timing method: ``torch.cuda.synchronize()`` before and after every timed stage '
                           'when CUDA is used; ``time.perf_counter``.\n'
                           'Two separate passes over the same trajectories with the same seed:\n'
                           '  pass 1 "pipeline": stages prep (CPU: dense arrays + interaction grids + '
                           'position change), transfer (to GPU), rollout (the baseline\n'
                           '                     ``sample`` call: model forwards + Gaussian sampling + '
                           'per-step grid work), post (revert position change);\n'
                           '                     pipeline latency = the whole per-trajectory span;\n'
                           '  pass 2 "model_only": forward hooks on the network time every ``net(...)`` call '
                           '(synchronised) and sum them per trajectory.\n'
                           'Warm-up trajectories are processed first in each pass and excluded from all '
                           'statistics.\n'
                           '"""\n'
                           '\n'
                           'import argparse\n'
                           'import csv\n'
                           'import hashlib\n'
                           'import importlib.util\n'
                           'import json\n'
                           'import math\n'
                           'import os\n'
                           'import pickle\n'
                           'import platform\n'
                           'import random\n'
                           'import shutil\n'
                           'import subprocess\n'
                           'import sys\n'
                           'import time\n'
                           'from datetime import datetime, timezone\n'
                           'from typing import Any, Dict, List, Optional, Sequence\n'
                           '\n'
                           'BENCH_VERSION = "pcg-inference-benchmark-1.0"\n'
                           'SCOPE = ("PCG trajectory-predictor-only, single-sample (sample_size=1), baseline '
                           'code path. NOT end-to-end video performance. NOT deployment-valid: "\n'
                           '         "the baseline path still uses future actor information as in the '
                           'historical test. Ground-truth scoring and oracle selection are excluded.")\n'
                           '\n'
                           '\n'
                           'def _sha(path: str) -> str:\n'
                           '    h = hashlib.sha256()\n'
                           '    with open(path, "rb") as fh:\n'
                           '        for b in iter(lambda: fh.read(1 << 20), b""):\n'
                           '            h.update(b)\n'
                           '    return h.hexdigest()\n'
                           '\n'
                           '\n'
                           'def _utc() -> str:\n'
                           '    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")\n'
                           '\n'
                           '\n'
                           'def _pct(v: Sequence[float], q: float) -> float:\n'
                           '    s = sorted(v)\n'
                           '    return s[min(len(s), max(1, int(math.ceil(q / 100.0 * len(s))))) - 1]\n'
                           '\n'
                           '\n'
                           'def describe(values: Sequence[float]) -> Dict[str, Any]:\n'
                           '    v = [float(x) for x in values if x is not None and math.isfinite(float(x))]\n'
                           '    if not v:\n'
                           '        return {"count": 0}\n'
                           '    m = sum(v) / len(v)\n'
                           '    return {"count": len(v), "mean_s": m, "std_s": math.sqrt(sum((x - m) ** 2 '
                           'for x in v) / len(v)), "min_s": min(v), "p50_s": _pct(v, 50), "p90_s": _pct(v, '
                           '90),\n'
                           '            "p95_s": _pct(v, 95), "p99_s": _pct(v, 99), "max_s": max(v), '
                           '"percentile_method": "nearest_rank"}\n'
                           '\n'
                           '\n'
                           'def preflight(repo: str, workdir: str, checkpoint: str, config: str, required: '
                           'Sequence[str] = ("test.py", "utils/DataLoader.py", '
                           '"Data/test/trajectories_test.cpkl")) -> List[str]:\n'
                           '    problems = []\n'
                           '    for rel in required:\n'
                           '        if not os.path.isfile(os.path.join(repo, rel)):\n'
                           '            problems.append("repository file missing: %s" % rel)\n'
                           '    for label, p in (("checkpoint", checkpoint), ("config", config)):\n'
                           '        if not os.path.isfile(p):\n'
                           '            problems.append("%s file missing: %s" % (label, p))\n'
                           '    rr, wr = os.path.realpath(repo), os.path.realpath(workdir)\n'
                           '    if wr == rr or wr.startswith(rr + os.sep):\n'
                           '        problems.append("work directory must be OUTSIDE the repository: %s" % '
                           'wr)\n'
                           '    if os.path.exists(workdir) and (not os.path.isdir(workdir) or '
                           'os.listdir(workdir)):\n'
                           '        problems.append("work directory exists and is not an empty directory: '
                           '%s" % workdir)\n'
                           '    return problems\n'
                           '\n'
                           '\n'
                           'def _gpu_driver() -> Optional[str]:\n'
                           '    try:\n'
                           '        return subprocess.run(["nvidia-smi", "--query-gpu=driver_version", '
                           '"--format=csv,noheader"], capture_output=True, text=True,\n'
                           '                              timeout=5).stdout.strip().splitlines()[0]\n'
                           '    except Exception:\n'
                           '        return None\n'
                           '\n'
                           '\n'
                           'def run_benchmark(repo: str, workdir: str, run_id: str, checkpoint: str, config: '
                           'str, *, epoch: int, method: int = 4, warmup: int = 20,\n'
                           '                  max_trajectories: Optional[int] = None, seed: int = 12345, '
                           'use_cuda: Optional[bool] = None, eval_mode: bool = False,\n'
                           '                  expected_checkpoint_sha256: Optional[str] = None, '
                           'expected_config_sha256: Optional[str] = None,\n'
                           '                  hardware_label: Optional[str] = None) -> Dict[str, Any]:\n'
                           '    problems = preflight(repo, workdir, checkpoint, config)\n'
                           '    if problems:\n'
                           '        raise RuntimeError("benchmark NOT started:\\n  - " + "\\n  - '
                           '".join(problems))\n'
                           '    repo, workdir = os.path.realpath(repo), os.path.realpath(workdir)\n'
                           '    os.makedirs(workdir, exist_ok=True)\n'
                           '    out = os.path.join(workdir, "measurements")\n'
                           '    os.makedirs(out)\n'
                           '    model_dir = os.path.join(workdir, "Store_Results", "model", "CollisionGrid" '
                           'if method == 4 else ("SocialLSTM" if method == 1 else "VanillaLSTM"))\n'
                           '    os.makedirs(model_dir)\n'
                           '    ck_dst = os.path.join(model_dir, "SOCIALLSTM_lstm_model_%d.tar" % epoch)\n'
                           '    cf_dst = os.path.join(model_dir, "config.pkl")\n'
                           '    shutil.copy2(checkpoint, ck_dst)                                  # COPIES: '
                           'the originals are only read\n'
                           '    shutil.copy2(config, cf_dst)\n'
                           '    shutil.copytree(os.path.join(repo, "Data"), os.path.join(workdir, "Data"))\n'
                           '    hashes = {"checkpoint_sha256": _sha(checkpoint), "checkpoint_copy_sha256": '
                           '_sha(ck_dst), "config_sha256": _sha(config), "config_copy_sha256": '
                           '_sha(cf_dst),\n'
                           '              "test_py_sha256": _sha(os.path.join(repo, "test.py")), '
                           '"test_data_sha256": _sha(os.path.join(repo, "Data", "test", '
                           '"trajectories_test.cpkl"))}\n'
                           '    if hashes["checkpoint_sha256"] != hashes["checkpoint_copy_sha256"] or '
                           'hashes["config_sha256"] != hashes["config_copy_sha256"]:\n'
                           '        raise RuntimeError("copied checkpoint/config hashes differ from the '
                           'originals; aborting")\n'
                           '    hashes["checkpoint_matches_expected"] = None if not '
                           'expected_checkpoint_sha256 else hashes["checkpoint_sha256"].lower() == '
                           'expected_checkpoint_sha256.lower()\n'
                           '    hashes["config_matches_expected"] = None if not expected_config_sha256 else '
                           'hashes["config_sha256"].lower() == expected_config_sha256.lower()\n'
                           '\n'
                           '    old_cwd = os.getcwd()\n'
                           '    sys.path.insert(0, repo)\n'
                           '    try:\n'
                           '        os.chdir(workdir)\n'
                           '        import numpy as np\n'
                           '        import torch\n'
                           '        cuda = torch.cuda.is_available() if use_cuda is None else '
                           'bool(use_cuda)\n'
                           '        if cuda and not torch.cuda.is_available():\n'
                           '            raise RuntimeError("use_cuda requested but CUDA is not available")\n'
                           '        spec = importlib.util.spec_from_file_location("pcg_repo_test_module", '
                           'os.path.join(repo, "test.py"))     # not `import test` (stdlib name clash)\n'
                           '        tmod = importlib.util.module_from_spec(spec)\n'
                           '        spec.loader.exec_module(tmod)                                  # defines '
                           'main()/sample(); nothing runs\n'
                           '        from utils.DataLoader import DataLoader\n'
                           '        from utils.helper import get_model, position_change_seq, '
                           'revert_postion_change_seq2\n'
                           '        from utils.Interaction import getSequenceInteractionGridMask\n'
                           '        import argparse as _ap\n'
                           '\n'
                           '        def seed_all():\n'
                           '            random.seed(seed)\n'
                           '            np.random.seed(seed)\n'
                           '            torch.manual_seed(seed)\n'
                           '            if cuda:\n'
                           '                torch.cuda.manual_seed_all(seed)\n'
                           '\n'
                           '        def sync():\n'
                           '            if cuda:\n'
                           '                torch.cuda.synchronize()\n'
                           '\n'
                           '        with open(cf_dst, "rb") as fh:\n'
                           '            saved_args = pickle.load(fh)\n'
                           '        cpu_override = False\n'
                           '        if not cuda and getattr(saved_args, "use_cuda", False):\n'
                           '            saved_args.use_cuda = False             # in-memory only (the model '
                           'calls .cuda() when this is True); the config FILE is not changed\n'
                           '            cpu_override = True\n'
                           '        sample_args = _ap.Namespace(obs_length=6, pred_length=6, use_cuda=cuda, '
                           'method=method, epoch=epoch, sample_size=1)\n'
                           '        t = time.perf_counter()\n'
                           '        dataloader = DataLoader(1, sample_args.obs_length + '
                           'sample_args.pred_length, infer=True, filtering=True)\n'
                           '        dataloader.reset_batch_pointer()\n'
                           '        net = get_model(method, saved_args, True)\n'
                           '        if cuda:\n'
                           '            net = net.cuda()\n'
                           '        try:\n'
                           '            ck = torch.load(ck_dst, map_location=("cuda" if cuda else "cpu"), '
                           'weights_only=False)\n'
                           '        except TypeError:\n'
                           '            ck = torch.load(ck_dst, map_location=("cuda" if cuda else "cpu"))\n'
                           '        net.load_state_dict(ck["state_dict"])\n'
                           '        if eval_mode:\n'
                           '            net.eval()\n'
                           '        setup_s = time.perf_counter() - t\n'
                           '        t = time.perf_counter()\n'
                           '        whole = dataloader.batch_creater(False, method, suffle=False)\n'
                           '        batch_creation_s = time.perf_counter() - t\n'
                           '        x_W, nP_W, P_W, xv_W, nV_W, V_W = whole[0], whole[1], whole[2], '
                           'whole[3], whole[4], whole[5]\n'
                           '        n_total = dataloader.num_batches\n'
                           '        n_use = n_total if max_trajectories is None else min(n_total, '
                           'max_trajectories)\n'
                           '        if n_use <= warmup:\n'
                           '            raise RuntimeError("need more trajectories (%d) than warm-up '
                           'iterations (%d)" % (n_use, warmup))\n'
                           '        obs = sample_args.obs_length\n'
                           '        timers: Dict[str, float] = {"fwd": 0.0, "calls": 0}\n'
                           '\n'
                           '        def pre_hook(mod, inp):\n'
                           '            sync()\n'
                           '            timers["t"] = time.perf_counter()\n'
                           '\n'
                           '        def post_hook(mod, inp, outp):\n'
                           '            sync()\n'
                           '            timers["fwd"] += time.perf_counter() - timers["t"]\n'
                           '            timers["calls"] += 1\n'
                           '\n'
                           '        def one(b: int, stage_times: Optional[Dict[str, float]]) -> Dict[str, '
                           'Any]:\n'
                           '            x, nP, P = x_W[b], nP_W[b], P_W[b]\n'
                           '            xv, nV, V = xv_W[b], nV_W[b], V_W[b]\n'
                           '            x_seq, numPedsList_seq, PedsList_seq = x[0], nP[0], P[0]\n'
                           '            x_seq_veh, numVehsList_seq, VehsList_seq = xv[0], nV[0], V[0]\n'
                           '            sync()\n'
                           '            t0 = time.perf_counter()\n'
                           '            # --- prep (replicates test.py: dense vector creation, grid masks on '
                           'CPU, position change)\n'
                           '            x_seq, lookup_seq = dataloader.convert_proper_array(x_seq, '
                           'numPedsList_seq, PedsList_seq)\n'
                           '            x_seq_veh, lookup_seq_veh = '
                           'dataloader.convert_proper_array(x_seq_veh, numVehsList_seq, VehsList_seq, '
                           'veh_flag=True)\n'
                           '            orig_x_seq = x_seq.clone()\n'
                           '            grid_seq, grid_TTC_seq = getSequenceInteractionGridMask(x_seq, '
                           'PedsList_seq, x_seq, PedsList_seq, saved_args.TTC, saved_args.D_min,\n'
                           '                                                                    '
                           'saved_args.num_sector, False, lookup_seq, lookup_seq)\n'
                           '            grid_seq_veh_in_ped, grid_TTC_veh_seq = '
                           'getSequenceInteractionGridMask(x_seq, PedsList_seq, x_seq_veh, VehsList_seq, '
                           'saved_args.TTC_veh,\n'
                           '                                                                                    '
                           'saved_args.D_min_veh, saved_args.num_sector, False, lookup_seq,\n'
                           '                                                                                    '
                           'lookup_seq_veh, is_heterogeneous=True, is_occupancy=False)\n'
                           '            x_seq, first_values_dict = position_change_seq(x_seq, PedsList_seq, '
                           'lookup_seq)\n'
                           '            t1 = time.perf_counter()\n'
                           '            # --- transfer\n'
                           '            if cuda:\n'
                           '                x_seq = x_seq.cuda()\n'
                           '                x_seq_veh = x_seq_veh.cuda()\n'
                           '            sync()\n'
                           '            t2 = time.perf_counter()\n'
                           '            # --- rollout: the unchanged baseline sample() with ONE sample\n'
                           '            ret_x_seq, dist_param_seq = tmod.sample(x_seq[:obs], '
                           'PedsList_seq[:obs], sample_args, net, x_seq, PedsList_seq, saved_args, '
                           'dataloader,\n'
                           '                                                    lookup_seq, numPedsList_seq, '
                           'saved_args.gru, first_values_dict, orig_x_seq, grid_seq[:obs], x_seq_veh,\n'
                           '                                                    VehsList_seq, '
                           'lookup_seq_veh, grid_seq_veh_in_ped[:obs], grid_TTC_seq[:obs], '
                           'grid_TTC_veh_seq[:obs])\n'
                           '            sync()\n'
                           '            t3 = time.perf_counter()\n'
                           '            # --- post (revert position change)\n'
                           '            ret_x_seq = revert_postion_change_seq2(ret_x_seq.cpu(), '
                           'PedsList_seq, lookup_seq, first_values_dict, orig_x_seq, obs, infer=True)\n'
                           '            dist_param_seq[:, :, 0:2] = '
                           'revert_postion_change_seq2(dist_param_seq[:, :, 0:2].cpu(), PedsList_seq, '
                           'lookup_seq, first_values_dict, orig_x_seq,\n'
                           '                                                                   obs, '
                           'infer=True)\n'
                           '            sync()\n'
                           '            t4 = time.perf_counter()\n'
                           '            finite = bool(torch.isfinite(ret_x_seq).all().item())\n'
                           '            return {"n_peds": len(lookup_seq), "n_vehicles": '
                           'len(lookup_seq_veh), "prep_s": t1 - t0, "transfer_s": t2 - t1, "rollout_s": t3 - '
                           't2,\n'
                           '                    "post_s": t4 - t3, "pipeline_s": t4 - t0, "output_finite": '
                           'finite, "output_shape": list(ret_x_seq.shape)}\n'
                           '\n'
                           '        # ---------------- pass 1: pipeline latency\n'
                           '        seed_all()\n'
                           '        for b in range(warmup):\n'
                           '            one(b, None)\n'
                           '        sync()\n'
                           '        if cuda:\n'
                           '            torch.cuda.reset_peak_memory_stats()\n'
                           '        rows: List[Dict[str, Any]] = []\n'
                           '        sync()\n'
                           '        loop_t0 = time.perf_counter()\n'
                           '        for b in range(warmup, n_use):\n'
                           '            r = one(b, None)\n'
                           '            r["trajectory_index"] = b\n'
                           '            rows.append(r)\n'
                           '        sync()\n'
                           '        loop_wall = time.perf_counter() - loop_t0\n'
                           '        peak_alloc = torch.cuda.max_memory_allocated() / 2 ** 20 if cuda else '
                           'None\n'
                           '        peak_res = torch.cuda.max_memory_reserved() / 2 ** 20 if cuda else None\n'
                           '        # ---------------- pass 2: model-only (forward hooks, same trajectories, '
                           'same seed)\n'
                           '        h1, h2 = net.register_forward_pre_hook(pre_hook), '
                           'net.register_forward_hook(post_hook)\n'
                           '        try:\n'
                           '            seed_all()\n'
                           '            for b in range(warmup):\n'
                           '                one(b, None)\n'
                           '            fwd = []\n'
                           '            calls = []\n'
                           '            for i, b in enumerate(range(warmup, n_use)):\n'
                           '                timers["fwd"], timers["calls"] = 0.0, 0\n'
                           '                one(b, None)\n'
                           '                rows[i]["model_forward_s"], rows[i]["n_forward_calls"] = '
                           'timers["fwd"], timers["calls"]\n'
                           '                fwd.append(timers["fwd"])\n'
                           '                calls.append(timers["calls"])\n'
                           '        finally:\n'
                           '            h1.remove()\n'
                           '            h2.remove()\n'
                           '        fields = ["trajectory_index", "n_peds", "n_vehicles", "prep_s", '
                           '"transfer_s", "rollout_s", "post_s", "pipeline_s", "model_forward_s", '
                           '"n_forward_calls",\n'
                           '                  "output_finite"]\n'
                           '        with open(os.path.join(out, "inference_latency_samples.csv"), "x", '
                           'newline="", encoding="utf-8") as fh:\n'
                           '            w = csv.DictWriter(fh, fieldnames=fields)\n'
                           '            w.writeheader()\n'
                           '            for r in rows:\n'
                           '                w.writerow({k: r.get(k) for k in fields})\n'
                           '        n = len(rows)\n'
                           '        summary = {\n'
                           '            "benchmark_version": BENCH_VERSION, "run_id": run_id, '
                           '"finished_utc": _utc(), "scope": SCOPE,\n'
                           '            "environment": {"python_executable": sys.executable, '
                           '"python_version": sys.version, "platform": platform.platform(), "torch": '
                           'torch.__version__,\n'
                           '                            "torch_cuda": torch.version.cuda, "numpy": '
                           'np.__version__, "cuda": cuda,\n'
                           '                            "gpu_name": torch.cuda.get_device_name(0) if cuda '
                           'else None,\n'
                           '                            "gpu_total_vram_mib": '
                           'torch.cuda.get_device_properties(0).total_memory / 2 ** 20 if cuda else None,\n'
                           '                            "gpu_driver": _gpu_driver(), "hardware_label": '
                           'hardware_label,\n'
                           '                            "cudnn_deterministic": '
                           'torch.backends.cudnn.deterministic, "cudnn_benchmark": '
                           'torch.backends.cudnn.benchmark},\n'
                           '            "protocol": {"epoch": epoch, "method": method, "sample_size": 1, '
                           '"obs_length": obs, "pred_length": sample_args.pred_length, "seed": seed,\n'
                           '                         "seed_applies_to": "python random, numpy, torch, '
                           'torch.cuda (set before each pass; not bitwise-deterministic across hardware)",\n'
                           '                         "warmup_trajectories_excluded": warmup, '
                           '"timed_trajectories": n, "trajectories_available": n_total, "eval_mode": '
                           'eval_mode,\n'
                           '                         "net_eval_called": eval_mode, '
                           '"dropout_active_in_prediction_path": (not eval_mode) and '
                           'float(getattr(saved_args, "dropout", 0.0)) > 0.0,\n'
                           '                         "dropout_probability": getattr(saved_args, "dropout", '
                           'None), "ground_truth_scoring_included": False, "oracle_selection_included": '
                           'False,\n'
                           '                         "batch_size_of_input": 1, '
                           '"saved_args_use_cuda_overridden_to_false_in_memory": cpu_override},\n'
                           '            "hashes": hashes, "setup_s": setup_s, "test_set_batch_creation_s": '
                           'batch_creation_s,\n'
                           '            "latency_s": {"pipeline": describe([r["pipeline_s"] for r in rows]), '
                           '"prep": describe([r["prep_s"] for r in rows]),\n'
                           '                          "transfer": describe([r["transfer_s"] for r in rows]), '
                           '"rollout_sample_call": describe([r["rollout_s"] for r in rows]),\n'
                           '                          "post": describe([r["post_s"] for r in rows]), '
                           '"model_forward_only_sum_per_trajectory": describe(fwd),\n'
                           '                          "model_forward_calls_per_trajectory": {"min": '
                           'min(calls), "max": max(calls), "mean": sum(calls) / len(calls)}},\n'
                           '            "throughput_sequences_per_s": {"pipeline_pass": (n / loop_wall) if '
                           'loop_wall > 0 else None, "definition": "timed trajectories / wall time of the '
                           'timed loop (pass 1)"},\n'
                           '            "peak_gpu_memory_mib": {"max_allocated": peak_alloc, "max_reserved": '
                           'peak_res, "note": "pass 1 after warm-up; None on CPU"},\n'
                           '            "output_check": {"non_finite_outputs": sum(1 for r in rows if not '
                           'r["output_finite"]), "shapes_seen": sorted({tuple(r["output_shape"]) for r in '
                           'rows})[:5]},\n'
                           '            "limitations": ["batch size 1, as in test.py; no batching study", '
                           '"future actor information is still consumed by the baseline path",\n'
                           '                            "model-only pass uses forward hooks that '
                           'synchronise, so it is separate from the pipeline pass",\n'
                           '                            "one dataset (HBS test split, 1081 sequences) and '
                           'one device; no claim beyond this"]}\n'
                           '        with open(os.path.join(out, "inference_benchmark_summary.json"), "x", '
                           'encoding="utf-8") as fh:\n'
                           '            json.dump(summary, fh, indent=2, sort_keys=True, default=str)\n'
                           '        return summary\n'
                           '    finally:\n'
                           '        os.chdir(old_cwd)\n'
                           '        if sys.path and sys.path[0] == repo:\n'
                           '            sys.path.pop(0)\n'
                           '\n'
                           '\n'
                           'def main(argv: Optional[Sequence[str]] = None) -> int:\n'
                           '    ap = argparse.ArgumentParser(description="Single-sample PCG predictor '
                           'benchmark (isolated; never modifies the repository or checkpoints).")\n'
                           '    ap.add_argument("--repo", required=True)\n'
                           '    ap.add_argument("--workdir", required=True)\n'
                           '    ap.add_argument("--run-id", required=True)\n'
                           '    ap.add_argument("--checkpoint", required=True)\n'
                           '    ap.add_argument("--config", required=True)\n'
                           '    ap.add_argument("--epoch", type=int, required=True)\n'
                           '    ap.add_argument("--warmup", type=int, default=20)\n'
                           '    ap.add_argument("--max-trajectories", type=int, default=None)\n'
                           '    ap.add_argument("--seed", type=int, default=12345)\n'
                           '    ap.add_argument("--cpu", action="store_true")\n'
                           '    ap.add_argument("--eval-mode", action="store_true")\n'
                           '    ap.add_argument("--hardware-label", default=None)\n'
                           '    ap.add_argument("--expected-checkpoint-sha256", default=None)\n'
                           '    ap.add_argument("--expected-config-sha256", default=None)\n'
                           '    a = ap.parse_args(argv)\n'
                           '    try:\n'
                           '        s = run_benchmark(a.repo, a.workdir, a.run_id, a.checkpoint, a.config, '
                           'epoch=a.epoch, warmup=a.warmup, max_trajectories=a.max_trajectories, '
                           'seed=a.seed,\n'
                           '                          use_cuda=False if a.cpu else None, '
                           'eval_mode=a.eval_mode, expected_checkpoint_sha256=a.expected_checkpoint_sha256,\n'
                           '                          expected_config_sha256=a.expected_config_sha256, '
                           'hardware_label=a.hardware_label)\n'
                           '    except RuntimeError as exc:\n'
                           '        print(str(exc), file=sys.stderr)\n'
                           '        return 2\n'
                           '    print(json.dumps({"timed": s["protocol"]["timed_trajectories"], '
                           '"pipeline_p50_s": s["latency_s"]["pipeline"].get("p50_s")}, indent=2))\n'
                           '    return 0\n'
                           '\n'
                           '\n'
                           'if __name__ == "__main__":\n'
                           '    sys.exit(main())\n'}

HELPER_SHA256 = {}
for _name, _src in HELPER_SOURCES.items():
    _p = os.path.join(TOOLS_DIR, _name)
    with open(_p, "x", encoding="utf-8", newline="\n") as _f:
        _f.write(_src)
    HELPER_SHA256[_name] = hashlib.sha256(_src.encode("utf-8")).hexdigest()
sys.path.insert(0, TOOLS_DIR)
import measurement_lib as ml
import inference_benchmark as bench_mod

print("Measurement run id :", MEAS_RUN_ID)
print("Output directory   :", MEAS_DIR)
print("Kernel interpreter :", sys.executable, "| Python", platform.python_version())
print("Repository present :", os.path.isdir(REPO), "| Drive backup present:", os.path.isdir(DRIVE_BACKUP))
print("Helper SHA-256     :", json.dumps(HELPER_SHA256, indent=1))
print("\nBenchmark is controlled in the dedicated control cell below. No training is available in this trimmed notebook.")

Measurement run id : meas_20261114T105102Z
Output directory   : /content/drive/MyDrive/PCG_Experiments/measurements/meas_20261114T105102Z
Kernel interpreter : /usr/bin/python3 | Python 3.13.16
Repository present : True | Drive backup present: True
Helper SHA-256     : {
 "measurement_lib.py": "711044d6befd4f5592b9b07220991f87d314e9dcf662817bfd81bb282c253c7b",
 "inference_benchmark.py": "85184647c778322844d917cdf777c73243f7f770aff01ea1e029f1f3552d90ee"
}

Benchmark is controlled in the dedicated control cell below. No training is available in this trimmed notebook.


In [5]:
# ==============================================================================================
# PART B - EXPERIMENT FINGERPRINT  ->  experiment_environment.json   (read-only on the experiment)
# ==============================================================================================
import re, glob, random

def _run(cmd, timeout=15):
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        return r.stdout.strip() if r.returncode == 0 else "ERROR(%d): %s" % (r.returncode, (r.stderr or r.stdout).strip()[:300])
    except Exception as exc:
        return "UNAVAILABLE: %s" % exc

def _version(mod):
    try:
        m = __import__(mod)
        return getattr(m, "__version__", "unknown")
    except Exception as exc:
        return "not importable: %s" % type(exc).__name__

def _probe_other_interpreter(py):
    """Ask ANOTHER interpreter (e.g. the pcg-env venv) what it really is. Read-only."""
    if not py or not os.path.isfile(py):
        return {"path": py, "exists": False}
    code = ("import sys,json,platform\nd={'executable':sys.executable,'python':sys.version,'platform':platform.platform()}\n"
            "for m in ('torch','torchvision','numpy','pandas','scipy','sklearn','matplotlib','psutil'):\n"
            "    try:\n        mod=__import__(m); d[m]=getattr(mod,'__version__','unknown')\n    except Exception as e:\n        d[m]='not importable: '+type(e).__name__\n"
            "try:\n    import torch; d['cuda_available']=torch.cuda.is_available(); d['torch_cuda']=torch.version.cuda\n"
            "except Exception: pass\nprint(json.dumps(d))")
    out = _run([py, "-c", code], timeout=120)
    try:
        d = json.loads(out.splitlines()[-1]); d["path"] = py; d["exists"] = True
        return d
    except Exception:
        return {"path": py, "exists": True, "probe_output": out[:500]}

def _cfg_dict(path):
    if not os.path.isfile(path):
        return {"path": path, "exists": False}
    with open(path, "rb") as fh:
        ns = pickle.load(fh)                      # a Namespace written by train.py (our own artifact)
    return {"path": path, "exists": True, "sha256": ml.sha256_file(path), "args": {k: (v if isinstance(v, (int, float, str, bool, list, type(None))) else str(v)) for k, v in vars(ns).items()}}

def collect_fingerprint():
    fp = {"fingerprint_version": "pcg-experiment-fingerprint-1.0", "measurement_run_id": MEAS_RUN_ID, "collected_utc": ml.utc_now_iso()}
    # --- interpreters (the historical notebook used two)
    fp["kernel_interpreter"] = {"executable": sys.executable, "python": sys.version, "implementation": platform.python_implementation()}
    fp["pcg_env_interpreter"] = _probe_other_interpreter("/content/pcg-env/bin/python")
    fp["interpreter_for_new_runs"] = PYTHON_FOR_NEW_RUNS or sys.executable
    # --- OS / hardware
    osr = {}
    try:
        with open("/etc/os-release") as fh:
            osr = dict(l.strip().split("=", 1) for l in fh if "=" in l)
    except Exception:
        pass
    fp["os"] = {"platform": platform.platform(), "system": platform.system(), "release": platform.release(), "os_release": {k: v.strip('"') for k, v in osr.items() if k in ("PRETTY_NAME", "VERSION_ID")}}
    cpu = _run(["lscpu"])
    fp["cpu"] = {"logical_cores": os.cpu_count(), "lscpu_model_name": next((l.split(":", 1)[1].strip() for l in cpu.splitlines() if l.startswith("Model name")), None), "platform_processor": platform.processor()}
    try:
        import psutil
        vm = psutil.virtual_memory()
        fp["ram"] = {"total_gib": round(vm.total / 2 ** 30, 2), "available_gib_at_collection": round(vm.available / 2 ** 30, 2)}
    except Exception as exc:
        fp["ram"] = {"unavailable": str(exc)}
    # --- libraries as seen by the KERNEL interpreter
    libs = {m: _version(m) for m in ("torch", "torchvision", "numpy", "pandas", "scipy", "sklearn", "matplotlib", "psutil")}
    fp["kernel_libraries"] = libs
    try:
        import torch
        fp["cuda"] = {"cuda_available": torch.cuda.is_available(), "torch_cuda_version": torch.version.cuda, "cudnn_version": (torch.backends.cudnn.version() if torch.cuda.is_available() else None)}
        if torch.cuda.is_available():
            p = torch.cuda.get_device_properties(0)
            fp["gpu"] = {"name": p.name, "total_vram_mib": p.total_memory // 2 ** 20, "compute_capability": "%d.%d" % (p.major, p.minor), "multiprocessors": p.multi_processor_count}
        fp["determinism_settings_in_kernel"] = {"cudnn_deterministic": torch.backends.cudnn.deterministic, "cudnn_benchmark": torch.backends.cudnn.benchmark,
                                                  "deterministic_algorithms_enabled": torch.are_deterministic_algorithms_enabled()}
    except Exception as exc:
        fp["cuda"] = {"error": str(exc)}
    smi = _run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"])
    fp["nvidia_smi_query"] = smi
    fp["gpu_driver_version"] = smi.split(",")[-1].strip() if smi and not smi.startswith(("ERROR", "UNAVAILABLE")) else None
    # --- git
    fp["git"] = {"commit": _run(["git", "-C", REPO, "rev-parse", "HEAD"]), "branch": _run(["git", "-C", REPO, "rev-parse", "--abbrev-ref", "HEAD"]),
                 "status_porcelain": _run(["git", "-C", REPO, "status", "--porcelain"]).splitlines()}
    fp["git"]["working_tree_clean"] = (len(fp["git"]["status_porcelain"]) == 0)
    # --- training configuration (the NEW run's config and the ORIGINAL CollisionGrid config)
    fp["training_configurations"] = {"training_run_config": _cfg_dict(os.path.join(STORE, "model", "config.pkl")),
                                     "collisiongrid_config_in_repo": _cfg_dict(os.path.join(STORE, "model", "CollisionGrid", "config.pkl")),
                                     "drive_backup_config": _cfg_dict(os.path.join(DRIVE_BACKUP, "config.pkl")),
                                     "drive_backup_original_collisiongrid_config": _cfg_dict(os.path.join(DRIVE_BACKUP, "CollisionGrid_original_config.pkl"))}
    # --- datasets, splits
    data_files = ["Data/HBS/hbs.csv", "Data/train/trajectories.cpkl", "Data/train/trajectories_train.cpkl", "Data/test/trajectories_test.cpkl"]
    fp["datasets"] = [ml.file_record(os.path.join(REPO, f), label=f) for f in data_files]
    fp["split_files"] = {"train_split": "Data/train/trajectories_train.cpkl", "test_split": "Data/test/trajectories_test.cpkl",
                         "note": "the historical loader has no separate validation split (see Phase 2 audit)"}
    # --- source files that define the experiment
    src_files = ["train.py", "test.py", "utils/DataLoader.py", "utils/helper.py", "utils/Interaction.py", "utils/grid.py", "model/model_CollisionGrid.py", "visualization.py", "requirements.txt"]
    fp["source_files"] = [ml.file_record(os.path.join(REPO, f), label=f) for f in src_files]
    # --- seeds: look for seeding in the code instead of assuming
    hits = []
    for f in glob.glob(os.path.join(REPO, "**", "*.py"), recursive=True):
        try:
            for n, line in enumerate(open(f, encoding="utf-8", errors="replace"), 1):
                if re.search(r"manual_seed|np\.random\.seed|random\.seed|set_seed|cudnn\.deterministic|use_deterministic_algorithms", line):
                    hits.append("%s:%d: %s" % (os.path.relpath(f, REPO), n, line.strip()[:120]))
        except OSError:
            pass
    fp["seeds"] = {"seed_calls_found_in_repository_python_files": hits, "conclusion": ("no seeding or determinism call found: the historical training and test were NOT seeded"
                   if not hits else "seeding calls exist; inspect them"), "PYTHONHASHSEED": os.environ.get("PYTHONHASHSEED"), "seed_used_by_this_notebook": None,
                   "note": "new instrumented runs record any seed they set; the monitored training runner sets none (it runs unmodified train.py)"}
    fp["helpers_sha256"] = HELPER_SHA256
    return fp

FINGERPRINT = collect_fingerprint()
_sha = ml.write_new_json(os.path.join(MEAS_DIR, "experiment_environment.json"), FINGERPRINT)
print("wrote experiment_environment.json  sha256 =", _sha)
print("kernel   :", FINGERPRINT["kernel_interpreter"]["executable"], "| torch", FINGERPRINT["kernel_libraries"]["torch"])
print("pcg-env  :", {k: FINGERPRINT["pcg_env_interpreter"].get(k) for k in ("exists", "executable", "torch", "torch_cuda")})
print("git      :", FINGERPRINT["git"]["commit"], "| clean:", FINGERPRINT["git"]["working_tree_clean"])
print("seeds    :", FINGERPRINT["seeds"]["conclusion"])


wrote experiment_environment.json  sha256 = 6228231245ae37a89bfb255abcfed7158fa539900e832f07f7e131c7dd3047f0
kernel   : /usr/bin/python3 | torch 2.14.1+cu130
pcg-env  : {'exists': False, 'executable': None, 'torch': None, 'torch_cuda': None}
git      : 0735a0f3b9f2186cb4e2be614eaf9c98d25f0de5 | clean: True
seeds    : no seeding or determinism call found: the historical training and test were NOT seeded


In [6]:
# ==============================================================================================
# PART C - ARTIFACT INTEGRITY MANIFEST  ->  artifact_manifest.json   (hashes only; nothing is written to or replaced in the artifacts)
# ==============================================================================================
# Previously recorded hashes. Source A = archived experiment manifest (experiments/PCG_T4_5_Epoch_Run/manifest.json in the project repository).
# Source B = hash of the git-tracked local copy of the historical files, computed on 2026-11-14 (no earlier recorded hash existed for these two).
_A = "archived experiment manifest.json (project repository)"
_B = "git-tracked local copy hashed 2026-11-14 (no earlier recorded hash)"
PREV = {
    "ckpt0": ("876D017CD6D381FBBDD1A9976A5A2843FDCEAEBA0FB45560DB406DF410C825CE", _A),
    "ckpt1": ("EBEE25A064E6DA5D235C5B23ECF8C1E55489EF8A5EEA654A279A6008D90D43F8", _A),
    "ckpt2": ("3ACF2412362C342E3426A7BE4C8B44558241ED3302970CF60554D832E5D0EC74", _A),
    "ckpt3": ("3A4EBD8BC4BFF5B960B9090ACB8EBD16640A01E5CB834B7DBFAA07D42F06B0EF", _A),
    "ckpt4": ("AC9F5462EC741B91541FA6BE74F07167060D63A469F88936C9D5FBC833387D48", _A),
    "experiment_config": ("9C994A14A5F6519B2C78256B3603C2E26975DDB647AC455BF13AECCD18485BA5", _A),
    "original_config": ("763E68C787524E836DD20DC833F6BF949DF488591F01BC74BDAB375ECB8E43D9", _A),
    "ckpt181": ("A01B3C59C425E070E7E25DB47C692226304E83C1F285C35D00AFA4BF1101775F", _B),
    "test_results": ("119E9BB665E2168035062FA075343ABA46CC9B2F2F8810054C16A996579192DE", _A),
    "log_curve": ("89BD93363B9D829B9876ACCCCC242951DB1E00ED6D91561304226466535D84C1", _A),
    "loss_plot_batch": ("7EC8BC95896EC75AA1746BCFD8C7BAD45DB7957E992FA32D81C57E19F9C77205", _A),
}

def _entry(label, path, key=None, note=None):
    h, src = PREV.get(key, (None, None))
    rec = ml.file_record(path, label=label, expected_sha256=h, expected_source=src)
    if note:
        rec["note"] = note
    return rec

ENTRIES = []
for e in range(5):                                   # epoch 0-4 checkpoints: working copy on the Colab disk/Drive link, and the Drive backup
    ENTRIES.append(_entry("checkpoint epoch %d (Store_Results/model)" % e, os.path.join(STORE, "model", "SOCIALLSTM_lstm_model_%d.tar" % e), "ckpt%d" % e))
    ENTRIES.append(_entry("checkpoint epoch %d (Drive backup)" % e, os.path.join(DRIVE_BACKUP, "SOCIALLSTM_lstm_model_%d.tar" % e), "ckpt%d" % e))
ENTRIES += [
    _entry("training configuration (Store_Results/model/config.pkl)", os.path.join(STORE, "model", "config.pkl"), "experiment_config"),
    _entry("training configuration (Drive backup config.pkl)", os.path.join(DRIVE_BACKUP, "config.pkl"), "experiment_config"),
    _entry("CollisionGrid config in repo (STAGED copy of the new training config)", os.path.join(STORE, "model", "CollisionGrid", "config.pkl"), "experiment_config",
           "the evaluation cell copied the new run's config over this file; it is EXPECTED to equal the training config, not the original"),
    _entry("ORIGINAL CollisionGrid config (Drive backup)", os.path.join(DRIVE_BACKUP, "CollisionGrid_original_config.pkl"), "original_config"),
    _entry("historical epoch-181 checkpoint (Store_Results/model/CollisionGrid)", os.path.join(STORE, "model", "CollisionGrid", "SOCIALLSTM_lstm_model_181.tar"), "ckpt181"),
    _entry("historical epoch-181 checkpoint (Drive backup)", os.path.join(DRIVE_BACKUP, "SOCIALLSTM_lstm_model_181.tar"), "ckpt181"),
    _entry("staged epoch-4 checkpoint (Store_Results/model/CollisionGrid)", os.path.join(STORE, "model", "CollisionGrid", "SOCIALLSTM_lstm_model_4.tar"), "ckpt4"),
    _entry("test prediction pickle (Store_Results/plot/test)", os.path.join(STORE, "plot", "test", "test_results.pkl"), "test_results",
           "test.py rewrites this file on every evaluation run; a mismatch can mean it was overwritten after the archived copy was made"),
    _entry("test prediction pickle (Drive backup test_results_epoch4.pkl)", os.path.join(DRIVE_BACKUP, "test_results_epoch4.pkl"), "test_results"),
    _entry("training log (Store_Results/log/log_curve.txt)", os.path.join(STORE, "log", "log_curve.txt"), "log_curve"),
    _entry("batch loss plot (Store_Results/plot/train)", os.path.join(STORE, "plot", "train", "loss_plot_batch.png"), "loss_plot_batch"),
    _entry("epoch loss plot (Store_Results/plot/train)", os.path.join(STORE, "plot", "train", "loss_plot_epoch.png")),
]
# consistency between a working copy and its Drive backup (same content?)
PAIRS = []
for e in range(5):
    a, b = ENTRIES[2 * e], ENTRIES[2 * e + 1]
    PAIRS.append({"pair": "epoch %d checkpoint: working copy vs Drive backup" % e, "both_exist": a["exists"] and b["exists"],
                  "identical": (a["sha256"] == b["sha256"]) if (a["exists"] and b["exists"]) else None})
MANIFEST = {
    "manifest_version": "pcg-artifact-manifest-1.0", "measurement_run_id": MEAS_RUN_ID, "created_utc": ml.utc_now_iso(),
    "policy": "read-only: files are hashed, never modified, moved or replaced",
    "previous_hash_sources": {"A": _A, "B": _B},
    "counts": {"entries": len(ENTRIES), "exist": sum(1 for r in ENTRIES if r["exists"]), "missing": sum(1 for r in ENTRIES if not r["exists"]),
               "match_previous": sum(1 for r in ENTRIES if r["matches_previous_hash"] is True),
               "mismatch_previous": sum(1 for r in ENTRIES if r["matches_previous_hash"] is False),
               "no_previous_hash": sum(1 for r in ENTRIES if r["exists"] and r["matches_previous_hash"] is None)},
    "pair_consistency": PAIRS, "artifacts": ENTRIES}
_sha = ml.write_new_json(os.path.join(MEAS_DIR, "artifact_manifest.json"), MANIFEST)
print("wrote artifact_manifest.json  sha256 =", _sha)
print(json.dumps(MANIFEST["counts"], indent=1))
for r in ENTRIES:
    flag = "MISSING" if not r["exists"] else ("MATCH" if r["matches_previous_hash"] is True else ("MISMATCH" if r["matches_previous_hash"] is False else "no previous hash"))
    print("%-9s %-72s %s" % (flag, r["label"][:72], (r["sha256"] or "")[:16]))


wrote artifact_manifest.json  sha256 = b4907abcc91e11a07a4675c0d9f691a028c32fe1043dbed9ebfd51745963c001
{
 "entries": 22,
 "exist": 22,
 "missing": 0,
 "match_previous": 21,
 "mismatch_previous": 0,
 "no_previous_hash": 1
}
MATCH     checkpoint epoch 0 (Store_Results/model)                                 876d017cd6d381fb
MATCH     checkpoint epoch 0 (Drive backup)                                        876d017cd6d381fb
MATCH     checkpoint epoch 1 (Store_Results/model)                                 ebee25a064e6da5d
MATCH     checkpoint epoch 1 (Drive backup)                                        ebee25a064e6da5d
MATCH     checkpoint epoch 2 (Store_Results/model)                                 3acf2412362c342e
MATCH     checkpoint epoch 2 (Drive backup)                                        3acf2412362c342e
MATCH     checkpoint epoch 3 (Store_Results/model)                                 3a4ebd8bc4bff5b9
MATCH     checkpoint epoch 3 (Drive backup)                                 

In [7]:
# ==============================================================================================
# PART F - RETROSPECTIVE ANALYSIS of what this notebook already recorded   (no new execution, no GPU)
# ==============================================================================================
# The data below were extracted programmatically from the SAVED OUTPUTS of the historical cells of this notebook
# (cell indices and the SHA-256 of the notebook they came from are inside the JSON). Nothing was typed in by hand
# and nothing was estimated. Everything computed here is labelled RETROSPECTIVE.
RETRO_JSON = r'''{"source_notebook_sha256":"b20af1a163b71a495dd3c37bd49fc150fb0b243de716df58173010c79dfeea10","extraction":"programmatic parse of saved cell outputs (cells 45, 49, 51, 54, 57, 60)","batch_size":10,"training_runs":{"one_epoch_cell45_pcg_env":{"cell_index":45,"execution_count":50,"command":"! /content/pcg-env/bin/python train.py --method 4 --num_epochs 1","interpreter_evidence":"explicit /content/pcg-env/bin/python in the command text (Python 3.11.18, PyTorch 2.2.1+cu121 per cells 19, 21, 28)","grid_calculation_s":31.914563417434692,"total_training_s":151.3521547317505,"batches":[[0,0,2.03,0.968],[1,0,1.012,0.743],[2,0,5.525,0.633],[3,0,1.509,0.648],[4,0,1.37,0.54],[5,0,1.007,0.59],[6,0,0.856,0.535],[7,0,1.655,0.619],[8,0,1.402,0.693],[9,0,0.993,0.562],[10,0,0.918,0.559],[11,0,0.671,0.651],[12,0,0.478,0.564],[13,0,0.373,0.555],[14,0,0.147,0.539],[15,0,0.778,0.56],[16,0,1.914,0.82],[17,0,0.821,0.799],[18,0,0.557,0.854],[19,0,0.417,0.742],[20,0,0.344,0.684],[21,0,0.047,0.766],[22,0,-0.02,0.484],[23,0,-0.055,0.511],[24,0,0.138,0.456],[25,0,0.288,0.488],[26,0,-0.01,0.589],[27,0,0.232,0.548],[28,0,-0.015,0.602],[29,0,0.01,0.548],[30,0,-0.26,0.664],[31,0,-0.182,0.623],[32,0,-0.08,0.559],[33,0,0.042,0.55],[34,0,0.308,0.528],[35,0,-0.133,0.609],[36,0,-0.392,0.524],[37,0,-0.54,0.508],[38,0,-0.661,0.693],[39,0,-0.513,0.551],[40,0,-0.453,0.657],[41,0,0.922,0.825],[42,0,0.446,0.809],[43,0,-0.185,0.95],[44,0,-0.406,0.884],[45,0,-0.453,0.704],[46,0,-0.651,0.541],[47,0,-0.581,0.599],[48,0,-0.44,0.696],[49,0,0.351,0.56],[50,0,0.191,0.555],[51,0,-0.55,0.552],[52,0,-0.567,0.567],[53,0,-0.606,0.552],[54,0,-0.801,0.565],[55,0,-0.773,0.648],[56,0,-0.843,0.589],[57,0,-0.962,0.473],[58,0,-1.096,0.506],[59,0,-0.645,0.551],[60,0,1.012,0.636],[61,0,-0.04,0.515],[62,0,-0.561,0.668],[63,0,-0.811,0.815],[64,0,-0.813,0.717],[65,0,-0.849,0.756],[66,0,-0.97,0.88],[67,0,-0.895,0.933],[68,0,-0.538,0.571],[69,0,0.371,0.688],[70,0,-0.204,0.475],[71,0,-0.755,0.535],[72,0,-0.887,0.618],[73,0,-0.911,0.519],[74,0,-1.059,0.534],[75,0,-1.062,0.493],[76,0,-1.196,0.548],[77,0,-0.974,0.526],[78,0,-0.957,0.574],[79,0,-0.886,0.508],[80,0,-0.422,0.595],[81,0,-0.611,0.563],[82,0,-0.915,0.638],[83,0,-0.977,0.604],[84,0,-0.979,0.515],[85,0,-1.045,0.527],[86,0,-0.889,0.741],[87,0,-0.96,0.655],[88,0,-1.249,0.748],[89,0,-0.955,0.751],[90,0,-0.873,1.033],[91,0,-0.853,0.649],[92,0,-0.924,0.458],[93,0,-1.1,0.552],[94,0,-1.187,0.691],[95,0,-1.259,0.556],[96,0,-1.248,0.616],[97,0,-1.313,0.563],[98,0,-1.083,0.664],[99,0,-1.22,0.547],[100,0,-0.999,0.554],[101,0,0.171,0.616],[102,0,0.446,0.592],[103,0,-0.807,0.681],[104,0,-0.916,0.549],[105,0,-0.928,0.701],[106,0,-0.837,0.548],[107,0,-1.104,0.495],[108,0,-1.051,0.713],[109,0,-1.154,0.759],[110,0,-1.165,0.698],[111,0,-1.104,0.803],[112,0,-1.14,0.758],[113,0,-0.758,0.815],[114,0,-0.769,0.62],[115,0,-1.116,0.489],[116,0,-1.317,0.557],[117,0,-1.339,0.587],[118,0,-1.259,0.476],[119,0,-0.998,0.506],[120,0,-0.554,0.57],[121,0,-0.471,0.546],[122,0,-1.055,0.613],[123,0,-1.218,0.565],[124,0,-1.273,0.574],[125,0,-1.336,0.535],[126,0,-1.414,0.616],[127,0,-1.479,0.595],[128,0,-1.403,0.543],[129,0,-1.517,0.53],[130,0,-1.086,0.657],[131,0,-1.362,0.818],[132,0,-1.231,0.768],[133,0,-1.038,0.926],[134,0,-0.807,0.687],[135,0,-1.213,0.741],[136,0,-1.224,0.928],[137,0,-1.144,0.708],[138,0,-1.147,0.745],[139,0,-1.339,0.679],[140,0,-1.393,0.619],[141,0,-1.294,0.58],[142,0,-1.428,0.555],[143,0,-1.406,0.624],[144,0,-1.237,0.547],[145,0,-1.331,0.579],[146,0,-1.263,0.582],[147,0,-0.865,0.605],[148,0,-0.23,0.542],[149,0,-0.808,0.626],[150,0,-1.294,0.483],[151,0,-1.254,0.581],[152,0,-1.451,0.601],[153,0,-1.419,0.643],[154,0,-1.224,0.821],[155,0,-1.099,0.858],[156,0,-1.145,0.768],[157,0,-1.436,0.953],[158,0,-1.442,0.785],[159,0,-1.51,0.541],[160,0,-1.502,0.522],[161,0,-0.713,0.543],[162,0,-0.678,0.601],[163,0,-0.975,0.542],[164,0,-1.395,0.608],[165,0,-1.381,0.628],[166,0,-1.264,0.6],[167,0,-1.344,0.61],[168,0,-1.369,0.613],[169,0,-1.547,0.552],[170,0,-1.237,0.613],[171,0,-1.297,0.576],[172,0,-1.128,0.554],[173,0,-1.411,0.601],[174,0,-1.337,0.675],[175,0,-1.209,0.51],[176,0,-0.813,0.708],[177,0,-0.69,0.82],[178,0,-1.2,0.679],[179,0,-1.325,0.74],[180,0,-0.915,0.702],[181,0,-1.393,0.682],[182,0,-1.496,0.622],[183,0,-1.515,0.556],[184,0,-1.444,0.517],[185,0,-1.264,0.532],[186,0,-1.05,0.582],[187,0,-1.271,0.654],[188,0,-1.274,0.506],[189,0,-1.399,0.581],[190,0,-1.481,0.541],[191,0,-1.458,0.617],[192,0,-1.315,0.465],[193,0,-1.59,0.558],[194,0,-1.557,0.493],[195,0,-1.524,0.517],[196,0,-0.775,0.551],[197,0,-1.092,0.63],[198,0,-1.275,0.682],[199,0,-1.566,0.631],[200,0,-1.605,0.895],[201,0,-1.288,0.893],[202,0,-1.536,0.715],[203,0,-1.416,0.773],[204,0,-1.239,0.863],[205,0,-1.069,0.774],[206,0,-1.532,0.525],[207,0,-1.535,0.684],[208,0,-1.155,0.509],[209,0,-1.48,0.552],[210,0,-1.566,0.517],[211,0,-1.664,0.559],[212,0,-0.878,0.609],[213,0,-1.578,0.59],[214,0,-1.593,0.644],[215,0,-1.579,0.462],[216,0,-1.567,0.65],[217,0,-1.618,0.602],[218,0,-1.063,0.706],[219,0,-1.265,0.527],[220,0,-1.346,0.709],[221,0,-1.377,0.648],[222,0,-1.21,0.787],[223,0,-1.562,0.712],[224,0,-1.523,0.808],[225,0,-1.68,0.879],[226,0,-1.433,0.807],[227,0,-1.471,0.871],[228,0,-1.492,0.651],[229,0,-1.534,0.536],[230,0,-1.654,0.571],[231,0,-1.64,0.559],[232,0,-1.297,0.598],[233,0,-1.403,0.545],[234,0,-0.935,0.501],[235,0,-1.243,0.589],[236,0,-1.613,0.604],[237,0,-1.638,0.575]]},"five_epoch_cell49_pcg_env":{"cell_index":49,"execution_count":54,"command":"!MPLBACKEND=Agg /content/pcg-env/bin/python train.py --method 4 --num_epochs 5","interpreter_evidence":"explicit /content/pcg-env/bin/python in the command text. The FIRST command of this cell was malformed ('5%cd ...') and exited with an argparse usage error; the SECOND command of the same cell ran a complete 5-epoch training","grid_calculation_s":30.574771642684937,"total_training_s":748.4588763713837,"batches":[[0,0,2.029,0.974],[1,0,1.475,0.573],[2,0,1.624,0.631],[3,0,1.84,0.535],[4,0,1.719,0.617],[5,0,1.465,0.758],[6,0,1.211,0.718],[7,0,0.921,0.846],[8,0,0.58,0.938],[9,0,0.647,0.799],[10,0,2.212,0.749],[11,0,1.037,0.467],[12,0,0.984,0.537],[13,0,0.822,0.578],[14,0,0.464,0.543],[15,0,0.537,0.671],[16,0,0.333,0.586],[17,0,0.113,0.591],[18,0,0.591,0.48],[19,0,0.117,0.53],[20,0,-0.012,0.589],[21,0,-0.089,0.629],[22,0,0.097,0.491],[23,0,0.372,0.538],[24,0,0.181,0.592],[25,0,-0.067,0.538],[26,0,-0.327,0.517],[27,0,-0.592,0.482],[28,0,-0.52,0.76],[29,0,1.432,0.823],[30,0,0.856,0.822],[31,0,0.314,0.771],[32,0,0.067,0.714],[33,0,-0.08,0.802],[34,0,-0.321,0.58],[35,0,-0.508,0.579],[36,0,-0.663,0.442],[37,0,-0.697,0.528],[38,0,-0.189,0.538],[39,0,0.396,0.621],[40,0,-0.278,0.56],[41,0,-0.586,0.501],[42,0,-0.725,0.702],[43,0,-0.838,0.543],[44,0,-0.891,0.424],[45,0,-0.608,0.513],[46,0,-0.501,0.617],[47,0,-0.741,0.517],[48,0,-0.948,0.487],[49,0,-0.862,0.578],[50,0,-0.928,0.55],[51,0,-0.729,0.663],[52,0,-0.569,0.77],[53,0,-0.823,0.742],[54,0,-0.926,0.67],[55,0,-0.607,0.847],[56,0,-0.83,0.923],[57,0,-0.926,0.622],[58,0,-1.101,0.583],[59,0,-0.943,0.465],[60,0,-0.842,0.496],[61,0,-0.64,0.553],[62,0,-0.903,0.631],[63,0,-1.131,0.549],[64,0,-1.124,0.519],[65,0,-1.011,0.587],[66,0,-0.488,0.591],[67,0,-0.431,0.514],[68,0,-0.845,0.445],[69,0,-0.994,0.607],[70,0,-1.169,0.594],[71,0,-1.119,0.544],[72,0,-1.091,0.51],[73,0,-1.144,0.517],[74,0,-0.841,0.47],[75,0,-0.469,0.621],[76,0,-0.966,0.79],[77,0,-0.919,0.84],[78,0,-1.165,0.863],[79,0,-1.153,0.685],[80,0,-1.271,0.852],[81,0,-1.11,0.64],[82,0,-1.277,0.547],[83,0,-1.091,0.422],[84,0,-0.981,0.561],[85,0,-0.882,0.481],[86,0,-0.786,0.629],[87,0,-1.031,0.529],[88,0,-1.279,0.621],[89,0,-1.239,0.528],[90,0,-1.113,0.642],[91,0,-1.044,0.545],[92,0,-1.076,0.576],[93,0,-0.555,0.557],[94,0,-0.858,0.478],[95,0,-1.376,0.551],[96,0,-1.269,0.623],[97,0,-1.261,0.586],[98,0,-1.068,0.618],[99,0,-1.027,0.671],[100,0,-1.277,0.913],[101,0,-1.426,0.778],[102,0,-1.373,0.837],[103,0,-1.147,1.013],[104,0,-1.412,0.597],[105,0,-1.488,0.586],[106,0,-1.091,0.603],[107,0,-0.729,0.63],[108,0,-0.951,0.546],[109,0,-1.177,0.585],[110,0,-1.131,0.57],[111,0,-1.057,0.556],[112,0,-1.148,0.574],[113,0,-0.783,0.479],[114,0,-1.104,0.582],[115,0,-0.904,0.602],[116,0,-1.162,0.661],[117,0,-1.046,0.623],[118,0,-1.4,0.542],[119,0,-1.445,0.5],[120,0,-1.529,0.456],[121,0,-1.434,0.555],[122,0,-1.034,0.93],[123,0,-0.78,0.823],[124,0,-0.865,0.986],[125,0,-1.235,0.731],[126,0,-1.057,0.899],[127,0,-1.206,0.492],[128,0,-1.343,0.544],[129,0,-1.499,0.637],[130,0,-1.636,0.494],[131,0,-1.127,0.548],[132,0,-0.597,0.475],[133,0,-1.22,0.553],[134,0,-1.275,0.52],[135,0,-1.353,0.567],[136,0,-1.338,0.596],[137,0,-1.37,0.595],[138,0,-1.436,0.571],[139,0,-1.176,0.601],[140,0,-1.155,0.56],[141,0,-1.185,0.599],[142,0,-1.149,0.556],[143,0,-1.152,0.561],[144,0,-1.361,0.849],[145,0,-1.416,0.897],[146,0,-1.156,0.813],[147,0,-1.23,0.91],[148,0,-1.374,0.814],[149,0,-1.332,0.854],[150,0,-1.551,0.485],[151,0,-1.501,0.681],[152,0,-1.588,0.514],[153,0,-1.365,0.571],[154,0,-1.101,0.554],[155,0,-1.262,0.492],[156,0,-0.776,0.511],[157,0,-0.785,0.548],[158,0,-1.328,0.642],[159,0,-1.421,0.59],[160,0,-1.577,0.555],[161,0,-1.63,0.561],[162,0,-1.449,0.615],[163,0,-1.433,0.558],[164,0,-1.287,0.497],[165,0,-1.372,0.525],[166,0,-1.309,0.46],[167,0,-1.237,0.743],[168,0,-1.411,0.764],[169,0,-1.543,0.812],[170,0,-1.244,0.791],[171,0,-1.228,0.672],[172,0,-1.316,0.784],[173,0,-1.408,0.658],[174,0,-1.621,0.5],[175,0,-1.492,0.575],[176,0,-1.471,0.467],[177,0,-1.28,0.55],[178,0,-1.508,0.548],[179,0,-1.493,0.561],[180,0,-1.519,0.61],[181,0,-1.645,0.529],[182,0,-1.391,0.566],[183,0,-1.44,0.659],[184,0,-1.348,0.601],[185,0,-1.39,0.591],[186,0,-1.311,0.587],[187,0,-1.279,0.576],[188,0,-1.149,0.51],[189,0,-1.407,0.515],[190,0,-1.556,0.587],[191,0,-1.541,0.737],[192,0,-1.447,0.761],[193,0,-1.484,0.769],[194,0,-1.499,0.775],[195,0,-1.614,0.735],[196,0,-1.606,0.773],[197,0,-0.901,0.582],[198,0,-1.235,0.674],[199,0,-1.494,0.578],[200,0,-1.49,0.675],[201,0,-1.629,0.505],[202,0,-1.41,0.526],[203,0,-1.343,0.583],[204,0,-1.606,0.501],[205,0,-1.545,0.68],[206,0,-1.326,0.647],[207,0,-1.098,0.517],[208,0,-1.459,0.637],[209,0,-1.418,0.515],[210,0,-1.671,0.489],[211,0,-1.428,0.556],[212,0,-1.209,0.668],[213,0,-1.316,0.885],[214,0,-1.535,0.716],[215,0,-1.485,0.719],[216,0,-1.502,0.806],[217,0,-1.574,0.739],[218,0,-1.437,0.919],[219,0,-1.401,0.515],[220,0,-1.567,0.484],[221,0,-1.416,0.642],[222,0,-1.52,0.535],[223,0,-1.495,0.525],[224,0,-1.625,0.473],[225,0,-1.47,0.607],[226,0,-1.318,0.696],[227,0,-1.526,0.575],[228,0,-1.617,0.572],[229,0,-1.723,0.648],[230,0,-1.195,0.531],[231,0,-1.5,0.586],[232,0,-1.594,0.567],[233,0,-1.467,0.504],[234,0,-1.628,0.64],[235,0,-1.664,0.461],[236,0,-1.571,0.667],[237,0,-1.61,0.862],[238,1,-1.284,0.713],[239,1,-1.426,1.054],[240,1,-1.616,0.885],[241,1,-1.324,0.648],[242,1,-1.471,0.545],[243,1,-1.314,0.532],[244,1,-1.614,0.59],[245,1,-1.64,0.568],[246,1,-1.43,0.627],[247,1,-1.59,0.543],[248,1,-1.446,0.581],[249,1,-1.591,0.5],[250,1,-1.602,0.533],[251,1,-1.597,0.613],[252,1,-1.52,0.631],[253,1,-1.672,0.696],[254,1,-1.47,0.533],[255,1,-1.484,0.569],[256,1,-1.703,0.652],[257,1,-1.79,0.551],[258,1,-1.713,0.792],[259,1,-1.714,0.727],[260,1,-1.768,0.865],[261,1,-1.693,0.812],[262,1,-1.367,0.736],[263,1,-1.233,0.821],[264,1,-1.063,0.545],[265,1,-1.528,0.558],[266,1,-1.317,0.491],[267,1,-1.506,0.48],[268,1,-1.694,0.469],[269,1,-1.767,0.692],[270,1,-1.53,0.657],[271,1,-1.574,0.525],[272,1,-1.272,0.495],[273,1,-1.212,0.488],[274,1,-1.614,0.501],[275,1,-1.569,0.627],[276,1,-1.701,0.607],[277,1,-1.724,0.494],[278,1,-1.533,0.648],[279,1,-1.621,0.466],[280,1,-1.541,0.437],[281,1,-1.562,0.545],[282,1,-1.734,0.847],[283,1,-1.406,0.797],[284,1,-1.634,0.708],[285,1,-1.765,0.929],[286,1,-1.581,0.818],[287,1,-1.504,0.796],[288,1,-1.571,0.565],[289,1,-1.632,0.528],[290,1,-1.647,0.581],[291,1,-1.281,0.55],[292,1,-1.505,0.576],[293,1,-1.245,0.642],[294,1,-1.634,0.546],[295,1,-1.658,0.618],[296,1,-1.606,0.592],[297,1,-1.384,0.526],[298,1,-1.308,0.554],[299,1,-1.794,0.516],[300,1,-1.678,0.514],[301,1,-1.621,0.468],[302,1,-1.718,0.647],[303,1,-1.611,0.556],[304,1,-0.945,0.741],[305,1,-1.235,0.675],[306,1,-1.654,0.934],[307,1,-1.525,0.807],[308,1,-1.489,0.827],[309,1,-1.685,0.861],[310,1,-1.585,0.669],[311,1,-1.617,0.552],[312,1,-1.659,0.597],[313,1,-1.83,0.516],[314,1,-1.861,0.583],[315,1,-1.823,0.484],[316,1,-1.54,0.595],[317,1,-1.597,0.559],[318,1,-1.689,0.613],[319,1,-2.041,0.465],[320,1,-1.585,0.655],[321,1,-1.968,0.57],[322,1,-1.023,0.506],[323,1,-1.476,0.643],[324,1,-1.601,0.58],[325,1,-1.855,0.486],[326,1,-1.734,0.643],[327,1,-1.377,0.674],[328,1,-1.246,0.911],[329,1,-1.627,0.797],[330,1,-1.057,0.678],[331,1,-1.657,0.738],[332,1,-1.705,0.911],[333,1,-1.652,0.676],[334,1,-1.863,0.55],[335,1,-1.508,0.495],[336,1,-1.332,0.526],[337,1,-1.519,0.5],[338,1,-1.838,0.594],[339,1,-1.687,0.462],[340,1,-1.759,0.663],[341,1,-1.684,0.546],[342,1,-1.889,0.5],[343,1,-1.566,0.615],[344,1,-1.736,0.63],[345,1,-1.623,0.625],[346,1,-1.545,0.565],[347,1,-1.785,0.525],[348,1,-1.49,0.675],[349,1,-1.335,0.647],[350,1,-1.631,0.549],[351,1,-1.772,0.824],[352,1,-1.663,0.782],[353,1,-1.915,0.726],[354,1,-1.868,0.809],[355,1,-1.893,0.85],[356,1,-1.816,0.755],[357,1,-1.786,0.605],[358,1,-1.513,0.583],[359,1,-1.944,0.529],[360,1,-1.947,0.671],[361,1,-1.767,0.613],[362,1,-1.777,0.619],[363,1,-1.382,0.659],[364,1,-1.03,0.69],[365,1,-1.604,0.54],[366,1,-1.698,0.482],[367,1,-1.876,0.525],[368,1,-1.683,0.625],[369,1,-1.723,0.642],[370,1,-1.533,0.658],[371,1,-1.56,0.634],[372,1,-1.728,0.51],[373,1,-1.746,0.745],[374,1,-1.667,0.768],[375,1,-1.93,0.71],[376,1,-1.849,0.858],[377,1,-1.976,0.908],[378,1,-1.915,0.746],[379,1,-1.302,0.578],[380,1,-1.287,0.527],[381,1,-1.668,0.523],[382,1,-1.7,0.554],[383,1,-1.795,0.609],[384,1,-1.619,0.564],[385,1,-1.847,0.661],[386,1,-1.755,0.611],[387,1,-1.838,0.566],[388,1,-1.888,0.454],[389,1,-1.521,0.553],[390,1,-1.593,0.625],[391,1,-1.594,0.477],[392,1,-1.492,0.482],[393,1,-1.861,0.591],[394,1,-1.704,0.538],[395,1,-1.743,0.682],[396,1,-1.755,0.728],[397,1,-1.725,0.71],[398,1,-1.924,0.806],[399,1,-1.741,0.77],[400,1,-1.791,0.868],[401,1,-1.588,0.938],[402,1,-1.523,0.585],[403,1,-1.701,0.472],[404,1,-1.66,0.429],[405,1,-1.713,0.551],[406,1,-1.579,0.553],[407,1,-1.746,0.527],[408,1,-1.718,0.454],[409,1,-1.388,0.553],[410,1,-1.744,0.563],[411,1,-1.781,0.59],[412,1,-1.766,0.461],[413,1,-1.893,0.534],[414,1,-1.883,0.667],[415,1,-1.785,0.542],[416,1,-1.717,0.626],[417,1,-1.405,0.62],[418,1,-1.4,0.625],[419,1,-1.893,0.581],[420,1,-1.625,0.827],[421,1,-1.949,0.824],[422,1,-1.929,0.738],[423,1,-1.889,0.722],[424,1,-1.679,0.735],[425,1,-1.559,0.758],[426,1,-1.392,0.635],[427,1,-1.777,0.578],[428,1,-1.863,0.495],[429,1,-1.9,0.568],[430,1,-1.81,0.58],[431,1,-1.919,0.65],[432,1,-1.789,0.613],[433,1,-1.725,0.58],[434,1,-1.978,0.591],[435,1,-1.777,0.586],[436,1,-2.145,0.607],[437,1,-1.825,0.533],[438,1,-2.094,0.554],[439,1,-1.656,0.51],[440,1,-1.576,0.535],[441,1,-1.786,0.604],[442,1,-1.604,0.497],[443,1,-1.724,0.789],[444,1,-1.795,0.753],[445,1,-1.838,0.7],[446,1,-1.949,0.681],[447,1,-1.833,0.784],[448,1,-1.907,0.808],[449,1,-1.474,0.642],[450,1,-1.821,0.511],[451,1,-1.54,0.687],[452,1,-1.988,0.615],[453,1,-1.885,0.544],[454,1,-1.933,0.527],[455,1,-1.75,0.578],[456,1,-1.601,0.49],[457,1,-1.816,0.606],[458,1,-1.725,0.52],[459,1,-2.043,0.549],[460,1,-1.843,0.523],[461,1,-1.828,0.518],[462,1,-1.533,0.551],[463,1,-1.875,0.456],[464,1,-1.942,0.609],[465,1,-1.633,0.597],[466,1,-1.578,0.704],[467,1,-1.548,0.764],[468,1,-1.779,0.7],[469,1,-1.615,0.753],[470,1,-1.989,0.77],[471,1,-1.692,0.74],[472,1,-2.117,0.642],[473,1,-1.334,0.52],[474,1,-1.761,0.547],[475,1,-1.683,0.528],[476,2,-1.756,0.534],[477,2,-1.844,0.602],[478,2,-1.651,0.67],[479,2,-1.84,0.539],[480,2,-1.611,0.513],[481,2,-1.843,0.588],[482,2,-1.515,0.443],[483,2,-1.995,0.454],[484,2,-1.965,0.575],[485,2,-1.169,0.529],[486,2,-1.725,0.582],[487,2,-1.774,0.651],[488,2,-1.824,0.533],[489,2,-1.411,0.849],[490,2,-1.835,0.75],[491,2,-1.879,0.906],[492,2,-1.867,0.761],[493,2,-1.934,0.823],[494,2,-1.689,0.858],[495,2,-1.947,0.541],[496,2,-1.49,0.702],[497,2,-1.812,0.494],[498,2,-1.812,0.567],[499,2,-2.055,0.557],[500,2,-1.945,0.462],[501,2,-2.169,0.683],[502,2,-2.055,0.529],[503,2,-2.051,0.562],[504,2,-1.676,0.613],[505,2,-1.454,0.524],[506,2,-1.657,0.569],[507,2,-1.976,0.562],[508,2,-1.897,0.536],[509,2,-1.93,0.594],[510,2,-2.099,0.483],[511,2,-1.958,0.658],[512,2,-1.838,0.745],[513,2,-2.118,0.81],[514,2,-1.819,0.683],[515,2,-1.814,0.677],[516,2,-1.574,0.895],[517,2,-1.827,0.846],[518,2,-1.827,0.66],[519,2,-2.022,0.562],[520,2,-2.099,0.584],[521,2,-1.857,0.525],[522,2,-1.909,0.582],[523,2,-2.012,0.529],[524,2,-1.96,0.7],[525,2,-1.85,0.74],[526,2,-2.146,0.536],[527,2,-1.931,0.601],[528,2,-1.615,0.618],[529,2,-1.545,0.52],[530,2,-1.585,0.651],[531,2,-1.969,0.546],[532,2,-1.2,0.534],[533,2,-1.857,0.619],[534,2,-1.803,0.559],[535,2,-1.923,0.78],[536,2,-1.687,0.682],[537,2,-1.919,0.834],[538,2,-1.76,0.694],[539,2,-1.846,0.823],[540,2,-1.763,0.841],[541,2,-1.842,0.514],[542,2,-1.918,0.549],[543,2,-1.787,0.551],[544,2,-1.798,0.548],[545,2,-2.131,0.474],[546,2,-1.891,0.728],[547,2,-1.884,0.533],[548,2,-1.898,0.557],[549,2,-2.102,0.553],[550,2,-2.103,0.527],[551,2,-2.231,0.546],[552,2,-1.628,0.631],[553,2,-1.397,0.624],[554,2,-1.905,0.586],[555,2,-2.155,0.489],[556,2,-1.842,0.689],[557,2,-2.121,0.457],[558,2,-2.033,0.731],[559,2,-1.866,0.721],[560,2,-1.591,0.706],[561,2,-1.897,0.718],[562,2,-1.73,1.109],[563,2,-1.846,0.816],[564,2,-2.134,0.562],[565,2,-1.899,0.469],[566,2,-1.892,0.539],[567,2,-1.89,0.525],[568,2,-1.453,0.596],[569,2,-1.635,0.539],[570,2,-1.813,0.484],[571,2,-1.9,0.629],[572,2,-1.822,0.622],[573,2,-1.79,0.642],[574,2,-1.934,0.567],[575,2,-2.005,0.578],[576,2,-2.103,0.579],[577,2,-1.834,0.515],[578,2,-1.319,0.512],[579,2,-1.894,0.575],[580,2,-2.036,0.662],[581,2,-1.436,0.799],[582,2,-2.052,0.708],[583,2,-1.752,0.78],[584,2,-1.389,0.627],[585,2,-1.496,0.929],[586,2,-1.989,0.685],[587,2,-2.087,0.646],[588,2,-1.988,0.515],[589,2,-2.166,0.462],[590,2,-1.939,0.677],[591,2,-1.851,0.562],[592,2,-1.64,0.562],[593,2,-1.973,0.609],[594,2,-1.926,0.515],[595,2,-2.013,0.593],[596,2,-1.569,0.536],[597,2,-1.525,0.481],[598,2,-1.801,0.6],[599,2,-1.901,0.544],[600,2,-1.894,0.632],[601,2,-1.875,0.601],[602,2,-1.799,0.554],[603,2,-2.099,0.694],[604,2,-2.227,0.754],[605,2,-1.62,0.73],[606,2,-1.604,0.721],[607,2,-1.734,0.8],[608,2,-2.071,0.733],[609,2,-1.613,0.879],[610,2,-2.083,0.608],[611,2,-2.176,0.563],[612,2,-1.901,0.565],[613,2,-1.774,0.481],[614,2,-1.863,0.53],[615,2,-1.687,0.462],[616,2,-1.967,0.569],[617,2,-1.819,0.548],[618,2,-1.837,0.623],[619,2,-1.897,0.539],[620,2,-2.17,0.671],[621,2,-1.926,0.576],[622,2,-2.219,0.586],[623,2,-2.06,0.594],[624,2,-1.954,0.506],[625,2,-1.222,0.651],[626,2,-1.398,0.584],[627,2,-1.687,0.589],[628,2,-1.774,0.679],[629,2,-2.009,0.73],[630,2,-2.163,0.771],[631,2,-1.962,0.83],[632,2,-1.734,0.929],[633,2,-1.575,0.62],[634,2,-1.527,0.495],[635,2,-1.898,0.602],[636,2,-1.87,0.675],[637,2,-2.164,0.661],[638,2,-2.13,0.486],[639,2,-1.759,0.496],[640,2,-1.591,0.548],[641,2,-2.024,0.571],[642,2,-2.224,0.563],[643,2,-2.046,0.515],[644,2,-1.994,0.515],[645,2,-1.92,0.565],[646,2,-1.975,0.497],[647,2,-1.775,0.571],[648,2,-1.531,0.514],[649,2,-2.083,0.533],[650,2,-2.121,0.503],[651,2,-2.068,0.783],[652,2,-1.9,0.827],[653,2,-2.236,0.78],[654,2,-2.073,0.689],[655,2,-1.968,0.938],[656,2,-1.848,0.717],[657,2,-2.104,0.521],[658,2,-1.69,0.527],[659,2,-1.893,0.58],[660,2,-2.202,0.591],[661,2,-1.562,0.676],[662,2,-1.895,0.51],[663,2,-1.896,0.542],[664,2,-1.996,0.707],[665,2,-1.887,0.476],[666,2,-2.035,0.582],[667,2,-1.969,0.454],[668,2,-2.037,0.598],[669,2,-1.891,0.58],[670,2,-1.553,0.589],[671,2,-1.867,0.574],[672,2,-1.968,0.527],[673,2,-1.999,0.508],[674,2,-2.016,0.822],[675,2,-1.887,0.902],[676,2,-1.904,0.77],[677,2,-2.042,0.762],[678,2,-1.772,0.706],[679,2,-1.914,0.786],[680,2,-2.001,0.486],[681,2,-1.827,0.559],[682,2,-1.845,0.598],[683,2,-1.923,0.523],[684,2,-2.025,0.583],[685,2,-1.877,0.51],[686,2,-2.033,0.641],[687,2,-2.015,0.573],[688,2,-1.902,0.611],[689,2,-1.567,0.553],[690,2,-1.794,0.562],[691,2,-1.846,0.519],[692,2,-2.012,0.689],[693,2,-1.679,0.607],[694,2,-2.046,0.596],[695,2,-1.847,0.484],[696,2,-1.698,0.507],[697,2,-1.836,0.795],[698,2,-1.759,0.791],[699,2,-2.103,0.748],[700,2,-1.96,0.919],[701,2,-2.152,0.72],[702,2,-1.947,0.676],[703,2,-1.999,0.576],[704,2,-1.831,0.53],[705,2,-1.93,0.565],[706,2,-2.093,0.543],[707,2,-1.54,0.746],[708,2,-1.869,0.612],[709,2,-2.109,0.485],[710,2,-1.877,0.519],[711,2,-1.857,0.452],[712,2,-1.682,0.548],[713,2,-1.894,0.538],[714,3,-2.056,0.585],[715,3,-1.921,0.597],[716,3,-1.757,0.501],[717,3,-2.178,0.654],[718,3,-2.463,0.528],[719,3,-2.067,0.523],[720,3,-1.477,0.94],[721,3,-1.539,0.711],[722,3,-2.053,0.794],[723,3,-2.253,0.707],[724,3,-1.867,0.922],[725,3,-2.034,0.627],[726,3,-1.936,0.597],[727,3,-1.853,0.669],[728,3,-1.812,0.508],[729,3,-2.175,0.495],[730,3,-2.119,0.586],[731,3,-1.664,0.522],[732,3,-1.812,0.54],[733,3,-2.229,0.522],[734,3,-2.177,0.601],[735,3,-2.063,0.515],[736,3,-1.92,0.51],[737,3,-1.878,0.541],[738,3,-2.114,0.531],[739,3,-2.171,0.417],[740,3,-2.024,0.627],[741,3,-1.981,0.573],[742,3,-1.968,0.481],[743,3,-2.133,0.668],[744,3,-1.954,0.744],[745,3,-1.871,0.668],[746,3,-1.84,0.8],[747,3,-2.001,0.744],[748,3,-1.909,0.844],[749,3,-2.058,0.786],[750,3,-2.173,0.6],[751,3,-1.824,0.579],[752,3,-2.006,0.63],[753,3,-2.108,0.526],[754,3,-2.161,0.52],[755,3,-1.169,0.466],[756,3,-1.901,0.507],[757,3,-2.279,0.598],[758,3,-2.043,0.56],[759,3,-1.886,0.556],[760,3,-2.058,0.605],[761,3,-1.811,0.46],[762,3,-1.824,0.496],[763,3,-1.926,0.511],[764,3,-2.261,0.615],[765,3,-1.964,0.578],[766,3,-1.839,0.672],[767,3,-1.929,0.881],[768,3,-1.887,0.793],[769,3,-1.963,0.6],[770,3,-2.006,0.847],[771,3,-2.141,0.625],[772,3,-1.995,0.708],[773,3,-2.045,0.584],[774,3,-2.119,0.507],[775,3,-1.36,0.523],[776,3,-1.293,0.523],[777,3,-1.923,0.485],[778,3,-1.793,0.61],[779,3,-1.914,0.725],[780,3,-1.943,0.543],[781,3,-1.881,0.482],[782,3,-1.887,0.543],[783,3,-1.953,0.536],[784,3,-1.949,0.478],[785,3,-2.05,0.543],[786,3,-2.04,0.578],[787,3,-2.121,0.513],[788,3,-2.218,0.6],[789,3,-2.053,0.495],[790,3,-1.673,0.485],[791,3,-1.853,0.872],[792,3,-1.928,0.763],[793,3,-1.777,0.782],[794,3,-1.952,0.733],[795,3,-1.802,0.877],[796,3,-1.961,0.737],[797,3,-2.108,0.553],[798,3,-1.989,0.537],[799,3,-2.176,0.604],[800,3,-2.095,0.522],[801,3,-1.806,0.509],[802,3,-2.07,0.542],[803,3,-2.056,0.52],[804,3,-1.656,0.575],[805,3,-1.87,0.506],[806,3,-1.882,0.545],[807,3,-2.085,0.681],[808,3,-1.878,0.52],[809,3,-2.02,0.577],[810,3,-1.988,0.526],[811,3,-1.869,0.58],[812,3,-1.908,0.549],[813,3,-2.169,0.558],[814,3,-2.04,0.802],[815,3,-2.14,0.96],[816,3,-2.03,0.671],[817,3,-2.136,0.679],[818,3,-1.979,0.756],[819,3,-1.826,0.796],[820,3,-2.161,0.516],[821,3,-1.877,0.48],[822,3,-1.823,0.496],[823,3,-1.719,0.425],[824,3,-1.996,0.557],[825,3,-1.909,0.598],[826,3,-2.092,0.589],[827,3,-2.273,0.566],[828,3,-1.694,0.443],[829,3,-1.707,0.591],[830,3,-1.726,0.548],[831,3,-2.076,0.564],[832,3,-2.09,0.56],[833,3,-2.267,0.678],[834,3,-1.8,0.535],[835,3,-1.835,0.616],[836,3,-2.04,0.547],[837,3,-1.853,0.501],[838,3,-1.902,0.698],[839,3,-1.978,0.713],[840,3,-1.938,0.863],[841,3,-1.972,0.679],[842,3,-2.12,0.949],[843,3,-1.946,0.802],[844,3,-2.127,0.648],[845,3,-2.149,0.667],[846,3,-2.108,0.597],[847,3,-1.976,0.687],[848,3,-2.153,0.504],[849,3,-1.913,0.738],[850,3,-1.666,0.696],[851,3,-1.936,0.668],[852,3,-1.734,0.589],[853,3,-2.039,0.668],[854,3,-2.146,0.574],[855,3,-2.255,0.598],[856,3,-2.066,0.548],[857,3,-2.225,0.496],[858,3,-2.066,0.631],[859,3,-1.723,0.551],[860,3,-2.166,0.787],[861,3,-2.041,0.695],[862,3,-2.209,0.811],[863,3,-1.902,1.039],[864,3,-2.056,0.814],[865,3,-1.912,0.713],[866,3,-2.001,0.592],[867,3,-2.336,0.515],[868,3,-2.4,0.518],[869,3,-1.883,0.564],[870,3,-1.79,0.57],[871,3,-1.919,0.578],[872,3,-1.859,0.565],[873,3,-2.095,0.628],[874,3,-2.104,0.542],[875,3,-2.071,0.561],[876,3,-2.269,0.659],[877,3,-1.768,0.491],[878,3,-1.84,0.625],[879,3,-1.757,0.57],[880,3,-1.992,0.635],[881,3,-2.306,0.59],[882,3,-2.0,0.646],[883,3,-2.027,0.771],[884,3,-2.154,0.821],[885,3,-2.358,0.91],[886,3,-2.191,0.737],[887,3,-2.167,0.902],[888,3,-1.857,0.593],[889,3,-1.721,0.567],[890,3,-1.994,0.583],[891,3,-1.914,0.554],[892,3,-1.6,0.6],[893,3,-2.053,0.579],[894,3,-1.521,0.527],[895,3,-2.137,0.617],[896,3,-2.458,0.572],[897,3,-2.071,0.618],[898,3,-2.1,0.67],[899,3,-2.105,0.469],[900,3,-2.204,0.503],[901,3,-2.359,0.507],[902,3,-1.551,0.61],[903,3,-2.098,0.537],[904,3,-2.204,0.532],[905,3,-1.918,0.644],[906,3,-1.964,0.726],[907,3,-2.224,0.623],[908,3,-2.055,0.94],[909,3,-2.015,0.906],[910,3,-1.839,0.89],[911,3,-1.824,0.462],[912,3,-2.179,0.506],[913,3,-1.788,0.547],[914,3,-2.013,0.65],[915,3,-1.989,0.528],[916,3,-2.043,0.58],[917,3,-2.223,0.483],[918,3,-2.03,0.564],[919,3,-1.557,0.456],[920,3,-2.233,0.636],[921,3,-2.143,0.488],[922,3,-1.95,0.639],[923,3,-1.677,0.589],[924,3,-1.853,0.534],[925,3,-2.011,0.67],[926,3,-2.074,0.449],[927,3,-1.774,0.608],[928,3,-2.196,0.587],[929,3,-2.285,0.725],[930,3,-2.141,0.741],[931,3,-2.034,0.73],[932,3,-1.723,0.685],[933,3,-1.873,0.805],[934,3,-2.232,0.849],[935,3,-2.16,0.57],[936,3,-2.176,0.526],[937,3,-1.955,0.615],[938,3,-1.971,0.659],[939,3,-2.147,0.524],[940,3,-2.154,0.636],[941,3,-1.562,0.482],[942,3,-2.105,0.599],[943,3,-1.953,0.76],[944,3,-2.164,0.575],[945,3,-1.811,0.477],[946,3,-1.823,0.498],[947,3,-2.006,0.643],[948,3,-2.254,0.57],[949,3,-1.766,0.526],[950,3,-2.119,0.58],[951,3,-1.89,0.697],[952,4,-1.911,0.934],[953,4,-2.076,0.87],[954,4,-2.063,0.815],[955,4,-2.081,0.682],[956,4,-2.047,0.827],[957,4,-1.297,0.532],[958,4,-1.905,0.59],[959,4,-2.072,0.597],[960,4,-2.185,0.641],[961,4,-2.378,0.578],[962,4,-2.297,0.546],[963,4,-1.995,0.501],[964,4,-2.055,0.565],[965,4,-1.797,0.773],[966,4,-2.199,0.552],[967,4,-1.889,0.504],[968,4,-1.918,0.666],[969,4,-1.842,0.581],[970,4,-2.117,0.677],[971,4,-2.477,0.538],[972,4,-2.356,0.663],[973,4,-2.022,0.783],[974,4,-2.426,0.767],[975,4,-1.882,0.862],[976,4,-1.994,0.719],[977,4,-2.021,0.768],[978,4,-1.86,0.737],[979,4,-1.927,0.551],[980,4,-2.089,0.704],[981,4,-2.242,0.524],[982,4,-2.011,0.53],[983,4,-2.124,0.576],[984,4,-2.36,0.735],[985,4,-2.16,0.56],[986,4,-1.951,0.541],[987,4,-2.147,0.634],[988,4,-2.039,0.598],[989,4,-1.859,0.686],[990,4,-2.127,0.621],[991,4,-2.217,0.514],[992,4,-2.106,0.507],[993,4,-2.434,0.498],[994,4,-2.029,0.558],[995,4,-1.921,0.568],[996,4,-1.215,0.768],[997,4,-1.941,0.781],[998,4,-2.224,0.758],[999,4,-2.294,0.609],[1000,4,-2.316,0.8],[1001,4,-2.071,0.839],[1002,4,-1.911,0.572],[1003,4,-2.094,0.597],[1004,4,-2.37,0.388],[1005,4,-2.209,0.508],[1006,4,-1.738,0.528],[1007,4,-2.137,0.568],[1008,4,-2.082,0.54],[1009,4,-2.034,0.531],[1010,4,-2.103,0.497],[1011,4,-2.4,0.567],[1012,4,-2.076,0.607],[1013,4,-2.184,0.583],[1014,4,-2.141,0.504],[1015,4,-2.23,0.505],[1016,4,-2.115,0.591],[1017,4,-2.244,0.51],[1018,4,-2.107,0.566],[1019,4,-2.38,0.577],[1020,4,-2.321,0.783],[1021,4,-2.169,0.781],[1022,4,-2.225,0.676],[1023,4,-1.983,0.778],[1024,4,-1.948,0.929],[1025,4,-2.017,0.762],[1026,4,-1.835,0.596],[1027,4,-2.099,0.554],[1028,4,-2.241,0.54],[1029,4,-1.946,0.521],[1030,4,-2.008,0.62],[1031,4,-1.988,0.609],[1032,4,-2.016,0.552],[1033,4,-1.976,0.634],[1034,4,-1.946,0.546],[1035,4,-2.036,0.542],[1036,4,-2.239,0.546],[1037,4,-2.282,0.564],[1038,4,-2.227,0.571],[1039,4,-1.942,0.574],[1040,4,-2.166,0.578],[1041,4,-1.84,0.61],[1042,4,-1.883,0.662],[1043,4,-2.089,0.776],[1044,4,-2.014,0.832],[1045,4,-1.831,0.752],[1046,4,-2.026,0.807],[1047,4,-2.152,0.936],[1048,4,-1.569,0.641],[1049,4,-1.964,0.53],[1050,4,-2.545,0.538],[1051,4,-1.936,0.532],[1052,4,-1.339,0.533],[1053,4,-1.775,0.572],[1054,4,-1.991,0.538],[1055,4,-1.979,0.531],[1056,4,-2.176,0.621],[1057,4,-2.223,0.522],[1058,4,-2.087,0.639],[1059,4,-2.145,0.638],[1060,4,-1.94,0.56],[1061,4,-2.337,0.552],[1062,4,-2.131,0.565],[1063,4,-1.727,0.521],[1064,4,-2.173,0.533],[1065,4,-1.921,0.668],[1066,4,-2.3,0.732],[1067,4,-2.242,0.789],[1068,4,-2.235,0.778],[1069,4,-1.855,0.929],[1070,4,-1.942,0.655],[1071,4,-2.169,0.784],[1072,4,-2.341,0.629],[1073,4,-2.196,0.587],[1074,4,-2.281,0.49],[1075,4,-2.024,0.587],[1076,4,-2.19,0.649],[1077,4,-1.202,0.556],[1078,4,-1.989,0.506],[1079,4,-1.662,0.693],[1080,4,-1.974,0.714],[1081,4,-2.076,0.55],[1082,4,-2.304,0.481],[1083,4,-2.181,0.565],[1084,4,-1.962,0.538],[1085,4,-2.277,0.545],[1086,4,-2.154,0.685],[1087,4,-2.355,0.58],[1088,4,-2.166,0.56],[1089,4,-2.039,0.798],[1090,4,-2.162,0.74],[1091,4,-2.192,0.641],[1092,4,-2.013,0.813],[1093,4,-2.16,0.672],[1094,4,-1.927,0.825],[1095,4,-1.939,0.505],[1096,4,-2.247,0.602],[1097,4,-2.41,0.47],[1098,4,-2.164,0.599],[1099,4,-2.263,0.607],[1100,4,-2.036,0.584],[1101,4,-2.094,0.706],[1102,4,-1.707,0.645],[1103,4,-2.16,0.586],[1104,4,-2.105,0.668],[1105,4,-2.029,0.6],[1106,4,-2.236,0.608],[1107,4,-1.883,0.642],[1108,4,-2.021,0.48],[1109,4,-1.892,0.586],[1110,4,-2.097,0.581],[1111,4,-1.916,0.792],[1112,4,-2.09,0.895],[1113,4,-2.164,0.829],[1114,4,-2.126,0.678],[1115,4,-2.159,0.81],[1116,4,-2.133,0.867],[1117,4,-2.127,0.569],[1118,4,-2.083,0.636],[1119,4,-2.009,0.543],[1120,4,-2.05,0.534],[1121,4,-2.114,0.514],[1122,4,-1.805,0.673],[1123,4,-1.53,0.563],[1124,4,-1.934,0.626],[1125,4,-2.022,0.587],[1126,4,-2.111,0.526],[1127,4,-2.174,0.582],[1128,4,-1.871,0.68],[1129,4,-2.125,0.64],[1130,4,-2.353,0.565],[1131,4,-2.199,0.643],[1132,4,-2.211,0.551],[1133,4,-2.167,0.618],[1134,4,-2.251,0.768],[1135,4,-2.145,0.772],[1136,4,-2.308,0.757],[1137,4,-2.108,0.899],[1138,4,-2.157,0.798],[1139,4,-1.954,0.856],[1140,4,-2.172,0.536],[1141,4,-2.124,0.563],[1142,4,-2.056,0.536],[1143,4,-2.116,0.626],[1144,4,-2.206,0.519],[1145,4,-2.258,0.728],[1146,4,-2.175,0.54],[1147,4,-1.808,0.602],[1148,4,-2.015,0.52],[1149,4,-2.049,0.466],[1150,4,-2.247,0.553],[1151,4,-2.391,0.531],[1152,4,-1.899,0.589],[1153,4,-1.928,0.695],[1154,4,-1.819,0.519],[1155,4,-2.285,0.547],[1156,4,-1.924,0.728],[1157,4,-2.335,0.814],[1158,4,-2.376,0.675],[1159,4,-2.224,0.786],[1160,4,-2.339,0.872],[1161,4,-2.201,0.834],[1162,4,-1.996,0.691],[1163,4,-2.097,0.581],[1164,4,-2.05,0.511],[1165,4,-2.068,0.521],[1166,4,-2.068,0.504],[1167,4,-1.807,0.475],[1168,4,-1.762,0.511],[1169,4,-1.831,0.682],[1170,4,-2.433,0.627],[1171,4,-2.112,0.597],[1172,4,-1.924,0.599],[1173,4,-2.04,0.611],[1174,4,-2.112,0.594],[1175,4,-2.234,0.562],[1176,4,-2.301,0.561],[1177,4,-1.952,0.701],[1178,4,-2.065,0.498],[1179,4,-2.334,0.615],[1180,4,-2.043,0.797],[1181,4,-2.178,0.774],[1182,4,-2.153,0.743],[1183,4,-2.095,0.9],[1184,4,-1.779,0.772],[1185,4,-1.308,0.659],[1186,4,-1.825,0.628],[1187,4,-2.001,0.706],[1188,4,-2.037,0.539],[1189,4,-2.286,0.631]]},"five_epoch_cell51_default_python":{"cell_index":51,"execution_count":56,"command":"!python train.py --method 4 --num_epochs 5","interpreter_evidence":"plain `!python`: resolved through the shell PATH, which the notebook does not record. The kernel itself was /usr/bin/python3 3.13.16 with PyTorch 2.14.1+cu130 (cells 0, 11). Grid time 13.4 s vs about 31 s in both pcg-env runs is consistent with a different environment but does not prove it","grid_calculation_s":13.412188291549683,"total_training_s":748.938973903656,"batches":[[0,0,1.917,1.737],[1,0,2.132,0.691],[2,0,1.754,0.867],[3,0,1.335,0.649],[4,0,1.235,0.594],[5,0,1.51,0.63],[6,0,1.166,0.603],[7,0,1.007,0.557],[8,0,0.841,0.642],[9,0,0.577,0.594],[10,0,0.409,0.527],[11,0,1.221,0.532],[12,0,0.668,0.592],[13,0,0.41,0.522],[14,0,0.088,0.642],[15,0,0.486,0.475],[16,0,0.521,0.525],[17,0,0.543,0.556],[18,0,0.159,0.517],[19,0,-0.208,0.478],[20,0,-0.311,0.587],[21,0,-0.217,0.597],[22,0,3.064,0.712],[23,0,0.496,0.766],[24,0,0.299,0.673],[25,0,0.15,0.778],[26,0,-0.072,0.803],[27,0,-0.146,0.642],[28,0,-0.32,0.555],[29,0,-0.378,0.477],[30,0,-0.245,0.558],[31,0,0.726,0.551],[32,0,0.248,0.59],[33,0,-0.152,0.642],[34,0,-0.2,0.667],[35,0,-0.319,0.603],[36,0,-0.483,0.549],[37,0,-0.649,0.602],[38,0,-0.743,0.597],[39,0,-0.9,0.643],[40,0,-0.887,0.665],[41,0,0.646,0.6],[42,0,0.933,0.486],[43,0,-0.356,0.54],[44,0,-0.542,0.82],[45,0,-0.619,0.672],[46,0,-0.705,0.689],[47,0,-0.806,0.795],[48,0,-0.88,0.999],[49,0,-0.921,0.541],[50,0,-0.971,0.577],[51,0,-0.543,0.573],[52,0,-0.909,0.552],[53,0,-0.805,0.573],[54,0,-0.778,0.559],[55,0,-0.663,0.624],[56,0,-0.774,0.608],[57,0,-0.485,0.597],[58,0,-0.882,0.568],[59,0,-0.999,0.59],[60,0,-0.928,0.525],[61,0,-0.357,0.517],[62,0,-0.654,0.545],[63,0,-0.97,0.569],[64,0,-1.161,0.552],[65,0,-1.335,0.479],[66,0,-1.231,0.755],[67,0,-1.402,0.675],[68,0,-0.877,0.64],[69,0,-1.013,0.77],[70,0,-0.983,0.82],[71,0,-1.019,0.826],[72,0,-0.997,0.717],[73,0,-0.993,0.66],[74,0,-1.07,0.619],[75,0,-1.1,0.587],[76,0,-1.075,0.536],[77,0,-0.84,0.621],[78,0,-0.794,0.563],[79,0,-0.893,0.611],[80,0,-1.222,0.614],[81,0,-1.173,0.611],[82,0,-1.174,0.705],[83,0,-1.341,0.559],[84,0,-1.222,0.688],[85,0,-1.107,0.655],[86,0,-1.294,0.563],[87,0,-1.138,0.627],[88,0,-1.4,0.714],[89,0,-1.47,0.755],[90,0,-1.315,0.766],[91,0,-1.035,0.785],[92,0,0.092,0.697],[93,0,-0.111,0.674],[94,0,-1.078,0.526],[95,0,-1.133,0.63],[96,0,-1.192,0.484],[97,0,-1.163,0.566],[98,0,-0.862,0.563],[99,0,-1.201,0.578],[100,0,-1.233,0.634],[101,0,-1.268,0.552],[102,0,-0.928,0.652],[103,0,-0.778,0.663],[104,0,-0.828,0.536],[105,0,-1.133,0.614],[106,0,-1.225,0.486],[107,0,-1.371,0.626],[108,0,-1.477,0.549],[109,0,-1.261,0.538],[110,0,-1.452,0.573],[111,0,-1.241,0.701],[112,0,-1.368,0.743],[113,0,-1.264,0.701],[114,0,-1.258,0.827],[115,0,-0.969,0.823],[116,0,-1.071,0.679],[117,0,-1.267,0.723],[118,0,-1.35,0.582],[119,0,-1.376,0.688],[120,0,-1.398,0.652],[121,0,-1.392,0.598],[122,0,-1.201,0.618],[123,0,-1.537,0.615],[124,0,-1.429,0.666],[125,0,-0.903,0.5],[126,0,-0.831,0.613],[127,0,-1.202,0.595],[128,0,-1.373,0.661],[129,0,-1.415,0.54],[130,0,-1.38,0.633],[131,0,-1.292,0.572],[132,0,-1.107,0.784],[133,0,-0.93,0.927],[134,0,-0.954,0.655],[135,0,-1.349,0.64],[136,0,-1.243,0.776],[137,0,-1.464,0.851],[138,0,-1.501,0.575],[139,0,-1.575,0.575],[140,0,-1.096,0.522],[141,0,-1.506,0.662],[142,0,-1.519,0.581],[143,0,-1.456,0.56],[144,0,-1.41,0.453],[145,0,-0.987,0.487],[146,0,-0.806,0.786],[147,0,-1.432,0.567],[148,0,-1.475,0.523],[149,0,-1.354,0.603],[150,0,-1.351,0.587],[151,0,-1.439,0.565],[152,0,-1.558,0.545],[153,0,-1.64,0.556],[154,0,-1.259,0.602],[155,0,-1.093,0.831],[156,0,-0.912,0.726],[157,0,-1.239,0.707],[158,0,-1.572,0.67],[159,0,-1.104,0.733],[160,0,-1.35,0.723],[161,0,-1.512,0.605],[162,0,-1.444,0.553],[163,0,-1.581,0.494],[164,0,-1.496,0.642],[165,0,-1.24,0.566],[166,0,-1.321,0.555],[167,0,-1.291,0.507],[168,0,-1.563,0.614],[169,0,-1.452,0.543],[170,0,-1.467,0.566],[171,0,-1.419,0.556],[172,0,-1.293,0.493],[173,0,-0.812,0.63],[174,0,-1.122,0.545],[175,0,-1.424,0.658],[176,0,-1.547,0.621],[177,0,-1.207,0.677],[178,0,-1.49,0.864],[179,0,-1.448,0.736],[180,0,-1.552,0.768],[181,0,-1.522,0.746],[182,0,-1.491,0.886],[183,0,-1.414,0.537],[184,0,-1.361,0.537],[185,0,-1.366,0.591],[186,0,-1.574,0.579],[187,0,-1.594,0.6],[188,0,-1.498,0.51],[189,0,-1.249,0.586],[190,0,-1.379,0.589],[191,0,-1.06,0.493],[192,0,-1.267,0.579],[193,0,-1.622,0.674],[194,0,-1.59,0.575],[195,0,-1.579,0.586],[196,0,-1.297,0.563],[197,0,-1.524,0.589],[198,0,-1.334,0.611],[199,0,-1.388,0.643],[200,0,-1.562,0.817],[201,0,-1.387,0.919],[202,0,-1.229,1.021],[203,0,-1.489,0.63],[204,0,-1.52,0.984],[205,0,-1.456,0.485],[206,0,-1.232,0.63],[207,0,-1.473,0.54],[208,0,-1.389,0.603],[209,0,-1.261,0.595],[210,0,-1.353,0.518],[211,0,-1.48,0.528],[212,0,-1.508,0.658],[213,0,-1.504,0.611],[214,0,-1.424,0.645],[215,0,-1.445,0.46],[216,0,-1.369,0.535],[217,0,-1.585,0.571],[218,0,-1.697,0.499],[219,0,-1.46,0.549],[220,0,-1.663,0.506],[221,0,-1.483,0.609],[222,0,-1.37,0.636],[223,0,-1.476,0.834],[224,0,-1.413,0.721],[225,0,-1.698,0.795],[226,0,-1.464,0.801],[227,0,-1.483,0.675],[228,0,-1.467,0.545],[229,0,-1.542,0.538],[230,0,-1.474,0.569],[231,0,-1.217,0.57],[232,0,-0.701,0.597],[233,0,-1.446,0.574],[234,0,-1.637,0.6],[235,0,-1.672,0.59],[236,0,-1.613,0.736],[237,0,-1.685,0.528],[238,1,-1.807,0.606],[239,1,-1.644,0.482],[240,1,-1.446,0.592],[241,1,-1.152,0.636],[242,1,-1.239,0.58],[243,1,-1.43,0.567],[244,1,-1.45,0.771],[245,1,-1.743,0.794],[246,1,-1.586,0.788],[247,1,-1.683,0.659],[248,1,-1.501,0.713],[249,1,-1.404,0.966],[250,1,-1.454,0.532],[251,1,-1.45,0.625],[252,1,-1.738,0.583],[253,1,-1.693,0.663],[254,1,-1.652,0.531],[255,1,-1.582,0.561],[256,1,-1.342,0.547],[257,1,-0.889,0.624],[258,1,-1.035,0.517],[259,1,-1.511,0.637],[260,1,-1.471,0.573],[261,1,-1.513,0.561],[262,1,-1.721,0.7],[263,1,-1.952,0.555],[264,1,-1.194,0.586],[265,1,-1.214,0.584],[266,1,-1.42,0.687],[267,1,-1.473,0.821],[268,1,-1.653,0.796],[269,1,-1.367,0.93],[270,1,-1.664,0.981],[271,1,-1.624,0.662],[272,1,-1.514,0.444],[273,1,-1.55,0.542],[274,1,-1.623,0.609],[275,1,-1.53,0.604],[276,1,-1.454,0.735],[277,1,-1.581,0.56],[278,1,-1.535,0.504],[279,1,-1.646,0.496],[280,1,-1.61,0.546],[281,1,-1.436,0.688],[282,1,-1.271,0.605],[283,1,-1.471,0.49],[284,1,-1.444,0.605],[285,1,-1.49,0.599],[286,1,-1.578,0.524],[287,1,-1.681,0.528],[288,1,-1.767,0.546],[289,1,-1.818,0.807],[290,1,-1.701,0.878],[291,1,-1.485,0.862],[292,1,-1.151,0.892],[293,1,-1.288,0.746],[294,1,-0.898,0.571],[295,1,-1.554,0.589],[296,1,-1.779,0.512],[297,1,-1.722,0.477],[298,1,-1.719,0.725],[299,1,-1.412,0.584],[300,1,-1.769,0.665],[301,1,-1.645,0.585],[302,1,-1.883,0.611],[303,1,-1.571,0.554],[304,1,-1.458,0.474],[305,1,-1.647,0.675],[306,1,-1.767,0.675],[307,1,-1.619,0.546],[308,1,-1.72,0.554],[309,1,-1.667,0.605],[310,1,-1.819,0.617],[311,1,-1.826,0.863],[312,1,-1.443,0.757],[313,1,-1.223,0.694],[314,1,-1.678,0.685],[315,1,-1.719,0.771],[316,1,-1.627,0.705],[317,1,-1.633,0.532],[318,1,-1.561,0.629],[319,1,-1.956,0.495],[320,1,-1.559,0.584],[321,1,-1.228,0.535],[322,1,-0.908,0.493],[323,1,-1.394,0.637],[324,1,-1.536,0.516],[325,1,-1.486,0.673],[326,1,-1.81,0.587],[327,1,-1.842,0.492],[328,1,-1.889,0.568],[329,1,-1.183,0.55],[330,1,-1.743,0.54],[331,1,-1.625,0.529],[332,1,-1.65,0.558],[333,1,-1.482,0.526],[334,1,-1.691,0.741],[335,1,-1.567,0.745],[336,1,-1.628,0.748],[337,1,-1.557,0.842],[338,1,-1.453,0.906],[339,1,-1.622,0.574],[340,1,-1.606,0.646],[341,1,-1.903,0.698],[342,1,-1.775,0.618],[343,1,-1.752,0.53],[344,1,-1.435,0.616],[345,1,-1.648,0.563],[346,1,-1.727,0.653],[347,1,-1.744,0.615],[348,1,-1.687,0.566],[349,1,-1.652,0.663],[350,1,-1.879,0.628],[351,1,-1.862,0.626],[352,1,-1.551,0.543],[353,1,-1.411,0.506],[354,1,-1.632,0.503],[355,1,-1.866,0.798],[356,1,-1.837,0.706],[357,1,-1.726,0.758],[358,1,-1.666,0.757],[359,1,-1.649,0.912],[360,1,-1.276,0.95],[361,1,-1.467,0.542],[362,1,-1.711,0.501],[363,1,-1.454,0.558],[364,1,-1.128,0.513],[365,1,-1.509,0.694],[366,1,-1.871,0.567],[367,1,-1.606,0.501],[368,1,-1.568,0.564],[369,1,-1.593,0.658],[370,1,-1.636,0.539],[371,1,-1.442,0.568],[372,1,-1.781,0.535],[373,1,-1.854,0.563],[374,1,-1.908,0.584],[375,1,-1.79,0.558],[376,1,-1.898,0.535],[377,1,-1.647,0.602],[378,1,-1.02,0.61],[379,1,-1.413,0.698],[380,1,-1.706,0.692],[381,1,-1.612,0.776],[382,1,-1.18,0.759],[383,1,-1.477,0.837],[384,1,-1.781,0.539],[385,1,-1.902,0.651],[386,1,-1.869,0.579],[387,1,-2.018,0.577],[388,1,-1.61,0.614],[389,1,-1.461,0.55],[390,1,-1.553,0.631],[391,1,-1.658,0.545],[392,1,-1.492,0.519],[393,1,-1.577,0.706],[394,1,-1.515,0.562],[395,1,-1.664,0.62],[396,1,-1.687,0.596],[397,1,-1.556,0.611],[398,1,-1.761,0.518],[399,1,-1.531,0.545],[400,1,-1.311,0.608],[401,1,-1.58,0.706],[402,1,-1.667,0.694],[403,1,-1.833,0.623],[404,1,-1.767,0.793],[405,1,-1.69,0.802],[406,1,-1.868,0.637],[407,1,-1.166,0.595],[408,1,-1.644,0.634],[409,1,-1.482,0.551],[410,1,-1.698,0.571],[411,1,-1.637,0.523],[412,1,-1.823,0.473],[413,1,-1.718,0.546],[414,1,-1.772,0.569],[415,1,-1.666,0.522],[416,1,-1.502,0.61],[417,1,-1.423,0.549],[418,1,-1.771,0.546],[419,1,-1.521,0.647],[420,1,-1.826,0.563],[421,1,-1.769,0.504],[422,1,-1.561,0.515],[423,1,-1.448,0.581],[424,1,-1.457,0.829],[425,1,-1.828,0.782],[426,1,-1.868,0.66],[427,1,-1.432,0.69],[428,1,-1.687,0.843],[429,1,-1.875,0.678],[430,1,-1.814,0.602],[431,1,-1.712,0.573],[432,1,-2.069,0.508],[433,1,-1.673,0.58],[434,1,-1.664,0.5],[435,1,-1.688,0.692],[436,1,-1.616,0.615],[437,1,-1.762,0.603],[438,1,-2.01,0.571],[439,1,-1.577,0.58],[440,1,-1.663,0.597],[441,1,-1.535,0.564],[442,1,-1.658,0.532],[443,1,-1.921,0.577],[444,1,-1.83,0.614],[445,1,-1.584,0.602],[446,1,-1.832,0.723],[447,1,-1.641,0.646],[448,1,-1.865,0.628],[449,1,-1.701,0.749],[450,1,-1.892,0.699],[451,1,-1.377,0.931],[452,1,-1.573,0.55],[453,1,-1.704,0.668],[454,1,-1.206,0.577],[455,1,-1.579,0.483],[456,1,-1.766,0.541],[457,1,-1.918,0.554],[458,1,-1.675,0.566],[459,1,-1.68,0.553],[460,1,-1.789,0.52],[461,1,-1.701,0.617],[462,1,-1.739,0.59],[463,1,-1.962,0.512],[464,1,-1.685,0.583],[465,1,-1.598,0.456],[466,1,-1.443,0.604],[467,1,-1.674,0.598],[468,1,-2.04,0.626],[469,1,-1.661,0.925],[470,1,-1.959,0.732],[471,1,-1.947,0.737],[472,1,-1.885,0.829],[473,1,-1.489,0.88],[474,1,-1.637,0.781],[475,1,-1.732,0.539],[476,2,-1.802,0.603],[477,2,-2.106,0.523],[478,2,-1.305,0.465],[479,2,-1.532,0.502],[480,2,-1.798,0.555],[481,2,-1.707,0.707],[482,2,-1.425,0.554],[483,2,-1.588,0.564],[484,2,-1.871,0.507],[485,2,-1.732,0.484],[486,2,-1.916,0.596],[487,2,-1.825,0.638],[488,2,-1.819,0.496],[489,2,-1.884,0.596],[490,2,-1.92,0.533],[491,2,-1.926,0.59],[492,2,-1.608,0.789],[493,2,-2.087,0.762],[494,2,-1.6,0.736],[495,2,-1.441,0.745],[496,2,-1.814,0.693],[497,2,-1.667,0.732],[498,2,-1.89,0.547],[499,2,-1.708,0.488],[500,2,-1.916,0.641],[501,2,-1.929,0.565],[502,2,-1.68,0.498],[503,2,-1.478,0.524],[504,2,-1.633,0.624],[505,2,-1.936,0.603],[506,2,-1.923,0.595],[507,2,-1.918,0.575],[508,2,-1.982,0.535],[509,2,-1.887,0.601],[510,2,-1.919,0.505],[511,2,-0.835,0.545],[512,2,-1.376,0.515],[513,2,-1.791,0.547],[514,2,-1.691,0.556],[515,2,-1.867,0.752],[516,2,-1.885,0.715],[517,2,-1.605,0.716],[518,2,-1.376,0.702],[519,2,-1.66,0.878],[520,2,-2.081,0.619],[521,2,-2.026,0.52],[522,2,-1.739,0.593],[523,2,-1.838,0.642],[524,2,-1.909,0.603],[525,2,-1.802,0.698],[526,2,-2.01,0.594],[527,2,-1.658,0.513],[528,2,-1.965,0.644],[529,2,-1.737,0.583],[530,2,-1.53,0.535],[531,2,-1.856,0.697],[532,2,-1.964,0.515],[533,2,-2.183,0.549],[534,2,-1.939,0.548],[535,2,-1.632,0.666],[536,2,-1.765,0.605],[537,2,-1.313,0.744],[538,2,-1.406,0.766],[539,2,-1.856,0.621],[540,2,-1.82,0.867],[541,2,-1.864,0.809],[542,2,-1.944,0.828],[543,2,-1.556,0.617],[544,2,-1.473,0.589],[545,2,-1.702,0.589],[546,2,-1.741,0.496],[547,2,-1.911,0.641],[548,2,-1.978,0.526],[549,2,-2.01,0.62],[550,2,-2.023,0.63],[551,2,-1.688,0.626],[552,2,-1.83,0.667],[553,2,-1.831,0.551],[554,2,-1.827,0.547],[555,2,-1.781,0.649],[556,2,-1.984,0.61],[557,2,-1.921,0.564],[558,2,-1.884,0.475],[559,2,-1.431,0.829],[560,2,-1.502,0.628],[561,2,-1.689,0.765],[562,2,-1.863,0.66],[563,2,-1.871,0.803],[564,2,-1.777,0.751],[565,2,-1.858,0.501],[566,2,-1.883,0.58],[567,2,-1.568,0.642],[568,2,-1.778,0.556],[569,2,-1.628,0.599],[570,2,-1.686,0.561],[571,2,-1.774,0.535],[572,2,-1.939,0.508],[573,2,-1.894,0.575],[574,2,-1.38,0.531],[575,2,-1.83,0.533],[576,2,-1.967,0.513],[577,2,-1.686,0.506],[578,2,-1.837,0.602],[579,2,-2.059,0.652],[580,2,-1.778,0.554],[581,2,-2.062,0.608],[582,2,-1.897,0.759],[583,2,-1.298,0.616],[584,2,-1.754,0.834],[585,2,-1.885,0.789],[586,2,-1.761,1.029],[587,2,-1.372,0.608],[588,2,-1.761,0.64],[589,2,-1.859,0.607],[590,2,-1.837,0.547],[591,2,-1.72,0.579],[592,2,-1.924,0.59],[593,2,-1.83,0.569],[594,2,-2.094,0.539],[595,2,-1.581,0.664],[596,2,-1.659,0.542],[597,2,-1.975,0.518],[598,2,-1.926,0.573],[599,2,-1.968,0.589],[600,2,-2.105,0.547],[601,2,-1.175,0.568],[602,2,-1.262,0.475],[603,2,-1.694,0.63],[604,2,-1.894,0.8],[605,2,-1.88,0.732],[606,2,-1.79,0.745],[607,2,-1.726,0.636],[608,2,-1.865,0.807],[609,2,-1.961,0.778],[610,2,-1.565,0.525],[611,2,-1.643,0.6],[612,2,-1.763,0.523],[613,2,-1.743,0.609],[614,2,-1.91,0.529],[615,2,-1.9,0.565],[616,2,-1.746,0.552],[617,2,-1.265,0.467],[618,2,-1.863,0.609],[619,2,-1.939,0.526],[620,2,-1.816,0.612],[621,2,-1.815,0.514],[622,2,-1.848,0.521],[623,2,-1.756,0.563],[624,2,-1.9,0.585],[625,2,-2.093,0.554],[626,2,-1.971,0.487],[627,2,-1.907,0.615],[628,2,-1.766,0.744],[629,2,-1.999,0.724],[630,2,-1.821,0.664],[631,2,-1.816,0.81],[632,2,-1.918,0.783],[633,2,-2.063,0.603],[634,2,-1.848,0.623],[635,2,-1.589,0.599],[636,2,-1.754,0.574],[637,2,-1.932,0.587],[638,2,-1.745,0.655],[639,2,-2.002,0.569],[640,2,-1.902,0.539],[641,2,-2.013,0.662],[642,2,-1.875,0.597],[643,2,-1.946,0.428],[644,2,-2.086,0.506],[645,2,-1.704,0.539],[646,2,-0.923,0.529],[647,2,-1.575,0.692],[648,2,-1.632,0.597],[649,2,-1.723,0.54],[650,2,-1.912,0.686],[651,2,-1.794,0.785],[652,2,-1.987,0.76],[653,2,-1.892,0.834],[654,2,-1.761,0.683],[655,2,-2.123,0.767],[656,2,-2.082,0.565],[657,2,-2.019,0.595],[658,2,-1.939,0.53],[659,2,-1.868,0.663],[660,2,-1.991,0.535],[661,2,-1.743,0.6],[662,2,-1.795,0.547],[663,2,-1.962,0.576],[664,2,-1.345,0.598],[665,2,-1.829,0.536],[666,2,-1.868,0.623],[667,2,-1.901,0.498],[668,2,-1.99,0.568],[669,2,-1.865,0.601],[670,2,-1.963,0.594],[671,2,-1.858,0.496],[672,2,-2.002,0.597],[673,2,-1.405,0.808],[674,2,-1.645,0.736],[675,2,-1.847,0.923],[676,2,-1.801,0.803],[677,2,-1.538,0.915],[678,2,-1.873,0.606],[679,2,-2.071,0.524],[680,2,-2.12,0.596],[681,2,-2.136,0.599],[682,2,-2.006,0.54],[683,2,-1.513,0.555],[684,2,-0.692,0.596],[685,2,-1.799,0.615],[686,2,-1.771,0.529],[687,2,-1.898,0.583],[688,2,-1.702,0.703],[689,2,-1.919,0.603],[690,2,-1.808,0.585],[691,2,-1.587,0.529],[692,2,-1.962,0.647],[693,2,-1.79,0.544],[694,2,-1.518,0.507],[695,2,-1.67,0.812],[696,2,-1.755,0.7],[697,2,-1.833,0.697],[698,2,-1.863,0.682],[699,2,-1.941,0.814],[700,2,-1.991,0.792],[701,2,-2.215,0.617],[702,2,-1.991,0.58],[703,2,-1.909,0.605],[704,2,-2.141,0.592],[705,2,-2.183,0.571],[706,2,-1.896,0.497],[707,2,-1.991,0.532],[708,2,-1.695,0.528],[709,2,-1.746,0.504],[710,2,-1.897,0.544],[711,2,-1.686,0.543],[712,2,-1.931,0.616],[713,2,-2.071,0.592],[714,3,-2.083,0.545],[715,3,-1.783,0.647],[716,3,-1.929,0.713],[717,3,-1.726,0.803],[718,3,-1.96,0.658],[719,3,-2.009,0.8],[720,3,-2.064,0.736],[721,3,-1.868,0.731],[722,3,-1.615,0.73],[723,3,-2.233,0.606],[724,3,-1.847,0.731],[725,3,-1.525,0.493],[726,3,-1.829,0.622],[727,3,-1.648,0.522],[728,3,-1.835,0.494],[729,3,-1.805,0.54],[730,3,-1.869,0.547],[731,3,-2.159,0.648],[732,3,-1.656,0.601],[733,3,-1.514,0.594],[734,3,-1.94,0.645],[735,3,-1.764,0.534],[736,3,-2.019,0.589],[737,3,-2.066,0.618],[738,3,-1.934,0.678],[739,3,-2.061,0.646],[740,3,-2.056,0.836],[741,3,-2.102,0.737],[742,3,-1.721,0.76],[743,3,-1.797,0.683],[744,3,-2.049,0.757],[745,3,-1.801,0.511],[746,3,-1.861,0.537],[747,3,-2.079,0.489],[748,3,-1.844,0.693],[749,3,-2.197,0.581],[750,3,-2.38,0.605],[751,3,-1.816,0.613],[752,3,-2.052,0.601],[753,3,-2.075,0.606],[754,3,-1.816,0.523],[755,3,-1.814,0.525],[756,3,-1.891,0.628],[757,3,-1.902,0.54],[758,3,-1.86,0.577],[759,3,-1.817,0.586],[760,3,-1.703,0.549],[761,3,-1.598,0.515],[762,3,-1.974,0.883],[763,3,-1.84,0.828],[764,3,-1.943,0.808],[765,3,-1.973,0.719],[766,3,-2.062,0.907],[767,3,-2.128,0.732],[768,3,-2.281,0.546],[769,3,-1.956,0.557],[770,3,-1.564,0.552],[771,3,-1.519,0.684],[772,3,-1.982,0.571],[773,3,-1.903,0.519],[774,3,-2.04,0.503],[775,3,-1.965,0.574],[776,3,-2.12,0.639],[777,3,-1.91,0.596],[778,3,-1.115,0.582],[779,3,-1.576,0.49],[780,3,-1.797,0.514],[781,3,-1.977,0.559],[782,3,-1.717,0.543],[783,3,-1.973,0.606],[784,3,-1.714,0.609],[785,3,-1.85,0.697],[786,3,-2.142,0.709],[787,3,-2.059,0.803],[788,3,-2.018,0.831],[789,3,-2.019,0.809],[790,3,-2.308,0.497],[791,3,-1.819,0.538],[792,3,-1.657,0.528],[793,3,-1.829,0.597],[794,3,-2.072,0.504],[795,3,-1.821,0.475],[796,3,-1.836,0.482],[797,3,-2.253,0.624],[798,3,-2.154,0.665],[799,3,-1.9,0.509],[800,3,-2.033,0.528],[801,3,-1.901,0.508],[802,3,-2.047,0.618],[803,3,-1.752,0.536],[804,3,-1.964,0.648],[805,3,-1.894,0.511],[806,3,-2.054,0.511],[807,3,-1.926,0.678],[808,3,-2.199,0.645],[809,3,-2.097,0.746],[810,3,-1.494,0.722],[811,3,-1.902,0.775],[812,3,-2.121,0.665],[813,3,-2.1,0.518],[814,3,-2.246,0.594],[815,3,-2.188,0.562],[816,3,-1.444,0.531],[817,3,-1.732,0.545],[818,3,-1.82,0.645],[819,3,-1.797,0.631],[820,3,-2.221,0.489],[821,3,-1.82,0.719],[822,3,-1.571,0.519],[823,3,-1.812,0.59],[824,3,-2.05,0.536],[825,3,-1.924,0.581],[826,3,-1.815,0.64],[827,3,-1.787,0.486],[828,3,-1.915,0.565],[829,3,-2.008,0.509],[830,3,-2.331,0.579],[831,3,-1.91,0.733],[832,3,-1.977,0.701],[833,3,-1.88,0.758],[834,3,-2.206,0.833],[835,3,-2.074,0.749],[836,3,-1.901,0.537],[837,3,-2.076,0.443],[838,3,-1.934,0.603],[839,3,-1.91,0.541],[840,3,-2.031,0.554],[841,3,-1.615,0.505],[842,3,-1.804,0.593],[843,3,-1.869,0.516],[844,3,-1.833,0.473],[845,3,-2.03,0.604],[846,3,-2.036,0.532],[847,3,-2.003,0.635],[848,3,-2.169,0.512],[849,3,-2.199,0.509],[850,3,-1.8,0.516],[851,3,-2.039,0.511],[852,3,-1.99,0.489],[853,3,-1.807,0.539],[854,3,-1.728,0.698],[855,3,-1.767,0.679],[856,3,-1.448,0.821],[857,3,-1.959,0.805],[858,3,-2.101,0.741],[859,3,-2.314,0.67],[860,3,-1.309,0.543],[861,3,-1.813,0.714],[862,3,-2.209,0.504],[863,3,-2.042,0.563],[864,3,-2.161,0.607],[865,3,-1.93,0.582],[866,3,-1.952,0.617],[867,3,-2.042,0.555],[868,3,-1.642,0.614],[869,3,-2.031,0.671],[870,3,-2.223,0.585],[871,3,-2.034,0.589],[872,3,-1.664,0.507],[873,3,-2.072,0.616],[874,3,-1.826,0.582],[875,3,-2.101,0.605],[876,3,-1.943,0.76],[877,3,-1.942,0.841],[878,3,-2.244,0.785],[879,3,-2.066,0.806],[880,3,-2.141,0.724],[881,3,-1.797,0.711],[882,3,-2.246,0.572],[883,3,-2.139,0.479],[884,3,-1.954,0.556],[885,3,-2.059,0.602],[886,3,-1.749,0.509],[887,3,-2.043,0.47],[888,3,-1.968,0.562],[889,3,-2.067,0.497],[890,3,-2.013,0.622],[891,3,-1.986,0.611],[892,3,-2.027,0.519],[893,3,-1.784,0.65],[894,3,-1.666,0.55],[895,3,-1.837,0.632],[896,3,-1.903,0.65],[897,3,-2.106,0.502],[898,3,-2.141,0.568],[899,3,-2.289,0.598],[900,3,-1.515,0.7],[901,3,-2.155,0.788],[902,3,-1.995,0.691],[903,3,-2.096,0.779],[904,3,-2.043,0.694],[905,3,-1.973,0.544],[906,3,-1.113,0.523],[907,3,-1.892,0.451],[908,3,-2.07,0.569],[909,3,-2.055,0.531],[910,3,-1.988,0.634],[911,3,-1.913,0.571],[912,3,-1.98,0.526],[913,3,-1.569,0.568],[914,3,-2.199,0.546],[915,3,-1.746,0.6],[916,3,-1.994,0.613],[917,3,-2.201,0.673],[918,3,-1.838,0.587],[919,3,-2.041,0.533],[920,3,-1.735,0.581],[921,3,-1.396,0.553],[922,3,-1.805,0.76],[923,3,-1.928,0.695],[924,3,-1.976,0.783],[925,3,-2.199,0.841],[926,3,-2.084,0.8],[927,3,-2.17,0.668],[928,3,-1.966,0.693],[929,3,-1.99,0.645],[930,3,-2.008,0.606],[931,3,-2.003,0.513],[932,3,-2.237,0.547],[933,3,-1.92,0.486],[934,3,-2.153,0.584],[935,3,-2.177,0.529],[936,3,-1.796,0.598],[937,3,-1.822,0.499],[938,3,-2.076,0.617],[939,3,-2.147,0.589],[940,3,-2.005,0.599],[941,3,-2.133,0.718],[942,3,-2.104,0.553],[943,3,-2.129,0.609],[944,3,-1.823,0.696],[945,3,-2.09,0.759],[946,3,-2.282,0.713],[947,3,-2.215,0.818],[948,3,-1.79,0.874],[949,3,-1.757,0.792],[950,3,-1.97,0.484],[951,3,-1.969,0.672],[952,4,-2.112,0.584],[953,4,-2.218,0.584],[954,4,-2.023,0.593],[955,4,-1.99,0.62],[956,4,-2.08,0.631],[957,4,-1.909,0.713],[958,4,-2.019,0.568],[959,4,-1.996,0.475],[960,4,-1.78,0.517],[961,4,-2.082,0.457],[962,4,-2.073,0.661],[963,4,-2.052,0.496],[964,4,-1.653,0.585],[965,4,-2.041,0.553],[966,4,-2.403,0.631],[967,4,-2.093,0.846],[968,4,-1.69,0.71],[969,4,-1.439,0.857],[970,4,-1.827,0.931],[971,4,-1.952,0.851],[972,4,-2.098,0.746],[973,4,-2.103,0.452],[974,4,-2.197,0.604],[975,4,-1.947,0.621],[976,4,-1.709,0.602],[977,4,-1.974,0.512],[978,4,-1.908,0.531],[979,4,-2.194,0.45],[980,4,-2.398,0.576],[981,4,-2.094,0.71],[982,4,-2.097,0.643],[983,4,-1.763,0.638],[984,4,-1.774,0.6],[985,4,-1.873,0.537],[986,4,-1.847,0.601],[987,4,-1.966,0.658],[988,4,-2.028,0.741],[989,4,-2.279,0.67],[990,4,-1.997,0.77],[991,4,-2.343,0.702],[992,4,-2.016,0.829],[993,4,-1.57,0.824],[994,4,-1.797,0.521],[995,4,-1.934,0.607],[996,4,-2.285,0.635],[997,4,-2.016,0.6],[998,4,-2.027,0.589],[999,4,-1.826,0.526],[1000,4,-2.179,0.536],[1001,4,-1.659,0.663],[1002,4,-1.899,0.534],[1003,4,-1.9,0.629],[1004,4,-2.17,0.525],[1005,4,-2.005,0.51],[1006,4,-1.804,0.594],[1007,4,-2.127,0.572],[1008,4,-2.02,0.535],[1009,4,-2.218,0.511],[1010,4,-2.093,0.629],[1011,4,-2.166,0.857],[1012,4,-2.27,0.785],[1013,4,-2.217,0.788],[1014,4,-1.91,0.748],[1015,4,-1.697,0.754],[1016,4,-1.888,0.676],[1017,4,-2.15,0.555],[1018,4,-1.923,0.564],[1019,4,-1.535,0.598],[1020,4,-2.008,0.566],[1021,4,-2.014,0.645],[1022,4,-2.179,0.565],[1023,4,-1.975,0.68],[1024,4,-2.104,0.573],[1025,4,-2.274,0.634],[1026,4,-2.056,0.593],[1027,4,-1.934,0.628],[1028,4,-2.045,0.632],[1029,4,-1.996,0.528],[1030,4,-1.767,0.547],[1031,4,-2.093,0.656],[1032,4,-2.401,0.516],[1033,4,-1.786,0.788],[1034,4,-2.133,0.819],[1035,4,-2.164,0.749],[1036,4,-2.323,0.758],[1037,4,-2.044,0.784],[1038,4,-1.888,0.89],[1039,4,-1.668,0.606],[1040,4,-1.639,0.665],[1041,4,-1.818,0.552],[1042,4,-2.122,0.628],[1043,4,-1.859,0.563],[1044,4,-1.918,0.553],[1045,4,-2.078,0.512],[1046,4,-2.19,0.546],[1047,4,-2.113,0.569],[1048,4,-1.957,0.661],[1049,4,-1.94,0.572],[1050,4,-2.173,0.504],[1051,4,-1.951,0.523],[1052,4,-1.772,0.597],[1053,4,-2.261,0.61],[1054,4,-1.814,0.541],[1055,4,-2.259,0.61],[1056,4,-1.955,0.907],[1057,4,-2.007,0.89],[1058,4,-2.221,0.832],[1059,4,-2.152,0.771],[1060,4,-2.208,0.779],[1061,4,-2.176,0.598],[1062,4,-2.047,0.518],[1063,4,-2.029,0.542],[1064,4,-2.16,0.522],[1065,4,-1.875,0.545],[1066,4,-2.125,0.572],[1067,4,-2.056,0.587],[1068,4,-2.138,0.584],[1069,4,-2.0,0.552],[1070,4,-1.852,0.576],[1071,4,-1.978,0.547],[1072,4,-2.135,0.567],[1073,4,-2.147,0.572],[1074,4,-1.975,0.521],[1075,4,-2.186,0.56],[1076,4,-1.938,0.554],[1077,4,-1.979,0.701],[1078,4,-1.887,0.775],[1079,4,-2.121,0.793],[1080,4,-1.833,0.851],[1081,4,-1.829,0.842],[1082,4,-2.039,0.891],[1083,4,-2.299,0.705],[1084,4,-2.081,0.513],[1085,4,-2.046,0.556],[1086,4,-1.982,0.567],[1087,4,-2.16,0.558],[1088,4,-2.041,0.591],[1089,4,-1.962,0.55],[1090,4,-1.731,0.591],[1091,4,-2.108,0.543],[1092,4,-1.839,0.572],[1093,4,-2.127,0.436],[1094,4,-2.073,0.553],[1095,4,-1.973,0.574],[1096,4,-1.816,0.609],[1097,4,-1.883,0.619],[1098,4,-1.594,0.587],[1099,4,-2.07,0.513],[1100,4,-2.16,0.735],[1101,4,-2.276,0.653],[1102,4,-1.93,0.718],[1103,4,-1.796,0.679],[1104,4,-1.981,0.771],[1105,4,-2.105,0.856],[1106,4,-2.284,0.573],[1107,4,-2.076,0.631],[1108,4,-2.294,0.608],[1109,4,-2.045,0.774],[1110,4,-2.052,0.623],[1111,4,-2.061,0.502],[1112,4,-2.139,0.513],[1113,4,-2.135,0.471],[1114,4,-2.109,0.613],[1115,4,-2.04,0.587],[1116,4,-2.106,0.554],[1117,4,-1.959,0.511],[1118,4,-2.195,0.611],[1119,4,-2.097,0.581],[1120,4,-2.2,0.694],[1121,4,-2.113,0.537],[1122,4,-2.258,0.701],[1123,4,-2.253,0.793],[1124,4,-1.813,0.633],[1125,4,-1.868,0.709],[1126,4,-1.977,0.723],[1127,4,-1.759,0.955],[1128,4,-2.181,0.704],[1129,4,-2.169,0.617],[1130,4,-2.109,0.657],[1131,4,-2.248,0.548],[1132,4,-2.249,0.596],[1133,4,-2.286,0.547],[1134,4,-2.044,0.637],[1135,4,-1.818,0.634],[1136,4,-1.946,0.611],[1137,4,-1.852,0.577],[1138,4,-1.886,0.514],[1139,4,-2.084,0.696],[1140,4,-1.777,0.673],[1141,4,-2.048,0.615],[1142,4,-2.179,0.565],[1143,4,-2.026,0.477],[1144,4,-1.901,0.702],[1145,4,-2.086,0.816],[1146,4,-2.167,0.773],[1147,4,-2.367,0.867],[1148,4,-1.948,0.795],[1149,4,-1.791,0.763],[1150,4,-2.071,0.561],[1151,4,-2.02,0.63],[1152,4,-2.456,0.504],[1153,4,-2.185,0.637],[1154,4,-2.118,0.6],[1155,4,-2.325,0.601],[1156,4,-2.172,0.572],[1157,4,-2.099,0.572],[1158,4,-2.015,0.681],[1159,4,-2.127,0.563],[1160,4,-2.138,0.582],[1161,4,-2.215,0.492],[1162,4,-2.149,0.591],[1163,4,-2.323,0.547],[1164,4,-2.138,0.654],[1165,4,-2.502,0.554],[1166,4,-1.853,0.55],[1167,4,-1.904,0.835],[1168,4,-2.165,0.811],[1169,4,-2.152,0.784],[1170,4,-2.037,0.814],[1171,4,-2.338,0.874],[1172,4,-1.969,0.541],[1173,4,-1.998,0.595],[1174,4,-1.988,0.617],[1175,4,-1.928,0.586],[1176,4,-1.924,0.663],[1177,4,-1.876,0.654],[1178,4,-2.22,0.618],[1179,4,-2.128,0.544],[1180,4,-2.09,0.598],[1181,4,-1.839,0.561],[1182,4,-2.039,0.495],[1183,4,-2.079,0.645],[1184,4,-2.113,0.533],[1185,4,-2.358,0.492],[1186,4,-1.845,0.635],[1187,4,-2.045,0.533],[1188,4,-1.826,0.476],[1189,4,-1.934,0.775]]}},"archived_epoch_losses_log_curve":[-0.936250077671915,-1.613831212391447,-1.7974503563302218,-1.9391287906934749,-2.0328173953199715],"archived_epoch_losses_source":"experiments/PCG_T4_5_Epoch_Run/logs/log_curve.txt of the project repository (SHA-256 recorded in its archived manifest)","checkpoint_mtimes_cell54":{"epoch_0_to_4":["2026-11-09 15:25:43","2026-11-09 15:28:13","2026-11-09 15:30:41","2026-11-09 15:33:09","2026-11-09 15:35:40"],"source":"cell 54 output, Store_Results/model checkpoint listing (Colab local time as printed)"},"test_pass":{"command":"!python test.py --method 4 --epoch 4 --sample_size 5","interpreter_evidence":"plain `!python` (see the cell-51 note; not recorded)","per_trajectory":[[1,0.37364697456359863],[2,0.2578461170196533],[3,0.26996755599975586],[4,0.2660355567932129],[5,0.2916259765625],[6,0.26781153678894043],[7,0.2783491611480713],[8,0.31917524337768555],[9,0.5508687496185303],[10,0.5385119915008545],[11,0.5244061946868896],[12,0.46898317337036133],[13,0.5004546642303467],[14,0.537506103515625],[15,0.6411800384521484],[16,0.5905232429504395],[17,0.28664231300354004],[18,0.33423852920532227],[19,0.37310290336608887],[20,0.30478906631469727],[21,0.30272459983825684],[22,0.37893199920654297],[23,0.4079270362854004],[24,0.4347565174102783],[25,0.4167921543121338],[26,0.394787073135376],[27,0.3562159538269043],[28,0.42688512802124023],[29,0.35923266410827637],[30,0.3168821334838867],[31,0.32494568824768066],[32,0.31369757652282715],[33,0.2718949317932129],[34,0.27946972846984863],[35,0.2641124725341797],[36,0.27518630027770996],[37,0.2238314151763916],[38,0.28400754928588867],[39,0.22064661979675293],[40,0.1856076717376709],[41,0.18405842781066895],[42,0.14870452880859375],[43,0.1796729564666748],[44,0.19587063789367676],[45,0.18207359313964844],[46,0.1841411590576172],[47,0.1799020767211914],[48,0.18212151527404785],[49,0.1921389102935791],[50,0.13270354270935059],[51,0.1334998607635498],[52,0.18376874923706055],[53,0.17355132102966309],[54,0.3233809471130371],[55,0.3232564926147461],[56,0.34778618812561035],[57,0.34714365005493164],[58,0.36229896545410156],[59,0.40276408195495605],[60,0.40601372718811035],[61,0.41420745849609375],[62,0.42371153831481934],[63,0.49468064308166504],[64,0.49988770484924316],[65,0.351459264755249],[66,0.3572232723236084],[67,0.3299076557159424],[68,0.3033263683319092],[69,0.30715346336364746],[70,0.29703307151794434],[71,0.24475955963134766],[72,0.1839597225189209],[73,0.18293428421020508],[74,0.28440356254577637],[75,0.28833818435668945],[76,0.2850518226623535],[77,0.31967878341674805],[78,0.27969956398010254],[79,0.2348616123199463],[80,0.28008484840393066],[81,0.24737143516540527],[82,0.26947617530822754],[83,0.2622506618499756],[84,0.2763063907623291],[85,0.2365703582763672],[86,0.19913220405578613],[87,0.15958404541015625],[88,0.1605548858642578],[89,0.1630697250366211],[90,0.18612933158874512],[91,0.16176271438598633],[92,0.16437578201293945],[93,0.15999865531921387],[94,0.1599714756011963],[95,0.2018899917602539],[96,0.21421289443969727],[97,0.2346937656402588],[98,0.23404145240783691],[99,0.253448486328125],[100,0.2657146453857422],[101,0.2482132911682129],[102,0.24080157279968262],[103,0.2491302490234375],[104,0.24303364753723145],[105,0.3125782012939453],[106,0.356121301651001],[107,0.3647193908691406],[108,0.3557736873626709],[109,0.2820899486541748],[110,0.28719091415405273],[111,0.29943346977233887],[112,0.3022806644439697],[113,0.2946186065673828],[114,0.2229633331298828],[115,0.16974425315856934],[116,0.21035146713256836],[117,0.19519281387329102],[118,0.1920163631439209],[119,0.2400531768798828],[120,0.3988916873931885],[121,0.25878047943115234],[122,0.248185396194458],[123,0.2393805980682373],[124,0.21997857093811035],[125,0.23613882064819336],[126,0.31067419052124023],[127,0.3093299865722656],[128,0.36416006088256836],[129,0.34147047996520996],[130,0.23529648780822754],[131,0.28774023056030273],[132,0.27890682220458984],[133,0.27086710929870605],[134,0.26276397705078125],[135,0.24250364303588867],[136,0.23806262016296387],[137,0.26572418212890625],[138,0.2655024528503418],[139,0.28977036476135254],[140,0.28061795234680176],[141,0.23325085639953613],[142,0.2638254165649414],[143,0.28267502784729004],[144,0.26932215690612793],[145,0.30141758918762207],[146,0.32064247131347656],[147,0.31771373748779297],[148,0.30921411514282227],[149,0.22729778289794922],[150,0.16972112655639648],[151,0.18179965019226074],[152,0.16612863540649414],[153,0.27066755294799805],[154,0.2746446132659912],[155,0.3221135139465332],[156,0.34311485290527344],[157,0.3761773109436035],[158,0.5147848129272461],[159,0.4600961208343506],[160,0.49376630783081055],[161,0.4506373405456543],[162,0.4751100540161133],[163,0.4726371765136719],[164,0.5040798187255859],[165,0.5079946517944336],[166,0.40100717544555664],[167,0.25879549980163574],[168,0.2778172492980957],[169,0.2631392478942871],[170,0.23614764213562012],[171,0.233506441116333],[172,0.17823195457458496],[173,0.16931509971618652],[174,0.16082382202148438],[175,0.17201638221740723],[176,0.1269698143005371],[177,0.13382577896118164],[178,0.12416863441467285],[179,0.14482593536376953],[180,0.12705302238464355],[181,0.12393403053283691],[182,0.20080852508544922],[183,0.20794177055358887],[184,0.16220808029174805],[185,0.1716916561126709],[186,0.15700721740722656],[187,0.15614676475524902],[188,0.16481232643127441],[189,0.15579676628112793],[190,0.16460537910461426],[191,0.16039061546325684],[192,0.17489194869995117],[193,0.1957712173461914],[194,0.26706981658935547],[195,0.26140546798706055],[196,0.28530001640319824],[197,0.2607574462890625],[198,0.2680239677429199],[199,0.2668437957763672],[200,0.28833675384521484],[201,0.31743431091308594],[202,0.26378607749938965],[203,0.27846741676330566],[204,0.20889520645141602],[205,0.18151021003723145],[206,0.18392467498779297],[207,0.17351913452148438],[208,0.16339111328125],[209,0.18796324729919434],[210,0.16892409324645996],[211,0.13484525680541992],[212,0.13600587844848633],[213,0.19188332557678223],[214,0.12595033645629883],[215,0.13262367248535156],[216,0.1764373779296875],[217,0.1693415641784668],[218,0.28471875190734863],[219,0.2824559211730957],[220,0.3047921657562256],[221,0.3426072597503662],[222,0.30977654457092285],[223,0.3090076446533203],[224,0.3667881488800049],[225,0.32070016860961914],[226,0.3276851177215576],[227,0.334531307220459],[228,0.37676382064819336],[229,0.3680565357208252],[230,0.47748446464538574],[231,0.3070831298828125],[232,0.3081545829772949],[233,0.32207417488098145],[234,0.34027528762817383],[235,0.34392333030700684],[236,0.3555281162261963],[237,0.30309462547302246],[238,0.347639799118042],[239,0.36269545555114746],[240,0.34950709342956543],[241,0.31469273567199707],[242,0.320728063583374],[243,0.26937413215637207],[244,0.2688171863555908],[245,0.3049275875091553],[246,0.3586690425872803],[247,0.3845705986022949],[248,0.38510966300964355],[249,0.3592526912689209],[250,0.4250211715698242],[251,0.4272582530975342],[252,0.41319775581359863],[253,0.3869619369506836],[254,0.40374040603637695],[255,0.34362125396728516],[256,0.3354032039642334],[257,0.34577250480651855],[258,0.35233521461486816],[259,0.39739251136779785],[260,0.48560476303100586],[261,0.43102574348449707],[262,0.43182849884033203],[263,0.4209325313568115],[264,0.42714786529541016],[265,0.539959192276001],[266,0.5922834873199463],[267,0.509568452835083],[268,0.41594862937927246],[269,0.3548917770385742],[270,0.33626699447631836],[271,0.38045167922973633],[272,0.27953529357910156],[273,0.27233266830444336],[274,0.22771739959716797],[275,0.2671196460723877],[276,0.2372121810913086],[277,0.2533721923828125],[278,0.294874906539917],[279,0.2977285385131836],[280,0.3070216178894043],[281,0.25324082374572754],[282,0.2618696689605713],[283,0.2623627185821533],[284,0.25604772567749023],[285,0.3265416622161865],[286,0.3639676570892334],[287,0.3881347179412842],[288,0.3530690670013428],[289,0.3750112056732178],[290,0.358379602432251],[291,0.24972224235534668],[292,0.24036383628845215],[293,0.2651503086090088],[294,0.24315190315246582],[295,0.34743762016296387],[296,0.3831050395965576],[297,0.3842043876647949],[298,0.39633989334106445],[299,0.3410334587097168],[300,0.3930213451385498],[301,0.5351767539978027],[302,0.6246788501739502],[303,0.6049416065216064],[304,0.6032900810241699],[305,0.558934211730957],[306,0.584916353225708],[307,0.6035845279693604],[308,0.4312126636505127],[309,0.3135490417480469],[310,0.3383805751800537],[311,0.3340601921081543],[312,0.2844052314758301],[313,0.34697866439819336],[314,0.276033878326416],[315,0.24365496635437012],[316,0.23846888542175293],[317,0.2982480525970459],[318,0.27434444427490234],[319,0.28160595893859863],[320,0.2898719310760498],[321,0.2857186794281006],[322,0.3254432678222656],[323,0.3226463794708252],[324,0.33981847763061523],[325,0.36282992362976074],[326,0.3560190200805664],[327,0.36792922019958496],[328,0.35596251487731934],[329,0.3135528564453125],[330,0.29296445846557617],[331,0.28231215476989746],[332,0.2866396903991699],[333,0.25664210319519043],[334,0.24869585037231445],[335,0.24404644966125488],[336,0.2476794719696045],[337,0.21598339080810547],[338,0.22592782974243164],[339,0.17599964141845703],[340,0.17571496963500977],[341,0.17615938186645508],[342,0.18236875534057617],[343,0.28960204124450684],[344,0.2995419502258301],[345,0.3448505401611328],[346,0.3596017360687256],[347,0.28038454055786133],[348,0.28752994537353516],[349,0.29195475578308105],[350,0.3898196220397949],[351,0.39386487007141113],[352,0.4267880916595459],[353,0.43732666969299316],[354,0.47100329399108887],[355,0.3269951343536377],[356,0.3607757091522217],[357,0.3495981693267822],[358,0.36425089836120605],[359,0.3899199962615967],[360,0.38804101943969727],[361,0.4053504467010498],[362,0.38736724853515625],[363,0.44107890129089355],[364,0.38178324699401855],[365,0.3909306526184082],[366,0.3109171390533447],[367,0.3079545497894287],[368,0.30701422691345215],[369,0.3302736282348633],[370,0.31161069869995117],[371,0.3064887523651123],[372,0.31655430793762207],[373,0.33843374252319336],[374,0.31473684310913086],[375,0.32276058197021484],[376,0.29074931144714355],[377,0.22907233238220215],[378,0.27694153785705566],[379,0.23864459991455078],[380,0.25751733779907227],[381,0.22826886177062988],[382,0.22510886192321777],[383,0.2689805030822754],[384,0.2856605052947998],[385,0.30890417098999023],[386,0.43586111068725586],[387,0.4386568069458008],[388,0.4322361946105957],[389,0.423811674118042],[390,0.43352651596069336],[391,0.410921573638916],[392,0.46871304512023926],[393,0.46212100982666016],[394,0.359194278717041],[395,0.4261128902435303],[396,0.26643848419189453],[397,0.26623106002807617],[398,0.2880685329437256],[399,0.30542993545532227],[400,0.26887011528015137],[401,0.1317751407623291],[402,0.13123536109924316],[403,0.1506819725036621],[404,0.13332080841064453],[405,0.13790154457092285],[406,0.17135334014892578],[407,0.18564605712890625],[408,0.17008304595947266],[409,0.20774054527282715],[410,0.17492938041687012],[411,0.17426276206970215],[412,0.17006564140319824],[413,0.1818094253540039],[414,0.1297624111175537],[415,0.13422632217407227],[416,0.14763164520263672],[417,0.12845969200134277],[418,0.13195514678955078],[419,0.1296072006225586],[420,0.1907033920288086],[421,0.19065189361572266],[422,0.23957204818725586],[423,0.28920984268188477],[424,0.32527828216552734],[425,0.4250216484069824],[426,0.40253448486328125],[427,0.4059743881225586],[428,0.409313440322876],[429,0.4033207893371582],[430,0.3827517032623291],[431,0.35475635528564453],[432,0.3145413398742676],[433,0.31442809104919434],[434,0.3277890682220459],[435,0.31377172470092773],[436,0.2725188732147217],[437,0.26645946502685547],[438,0.295623779296875],[439,0.2745335102081299],[440,0.27117085456848145],[441,0.34006214141845703],[442,0.2807350158691406],[443,0.26801633834838867],[444,0.18449687957763672],[445,0.1838977336883545],[446,0.1673870086669922],[447,0.16185784339904785],[448,0.16437935829162598],[449,0.15771150588989258],[450,0.183685302734375],[451,0.18612265586853027],[452,0.18792438507080078],[453,0.16065526008605957],[454,0.18417024612426758],[455,0.18906331062316895],[456,0.16267061233520508],[457,0.12354612350463867],[458,0.18509888648986816],[459,0.16410589218139648],[460,0.16640186309814453],[461,0.2181847095489502],[462,0.20760154724121094],[463,0.2162647247314453],[464,0.20391416549682617],[465,0.20293688774108887],[466,0.20803570747375488],[467,0.20448565483093262],[468,0.2528529167175293],[469,0.20483112335205078],[470,0.1987912654876709],[471,0.20321393013000488],[472,0.2714424133300781],[473,0.2964944839477539],[474,0.26872777938842773],[475,0.26947808265686035],[476,0.28992390632629395],[477,0.31226682662963867],[478,0.2792527675628662],[479,0.28045058250427246],[480,0.2185964584350586],[481,0.20454192161560059],[482,0.2053546905517578],[483,0.20575761795043945],[484,0.2101905345916748],[485,0.2263319492340088],[486,0.24632549285888672],[487,0.23918867111206055],[488,0.2527048587799072],[489,0.30704164505004883],[490,0.23598861694335938],[491,0.24760079383850098],[492,0.2407970428466797],[493,0.2624778747558594],[494,0.2393953800201416],[495,0.24366402626037598],[496,0.23647069931030273],[497,0.24852967262268066],[498,0.3319571018218994],[499,0.48964881896972656],[500,0.45273804664611816],[501,0.4390277862548828],[502,0.544337272644043],[503,0.708418607711792],[504,0.7287957668304443],[505,0.7688643932342529],[506,0.6449224948883057],[507,0.5173516273498535],[508,0.5462558269500732],[509,0.5606567859649658],[510,0.5500504970550537],[511,0.5725021362304688],[512,0.45063138008117676],[513,0.46909403800964355],[514,0.4148569107055664],[515,0.4056112766265869],[516,0.29898953437805176],[517,0.3257296085357666],[518,0.2862205505371094],[519,0.3842463493347168],[520,0.40531396865844727],[521,0.39327049255371094],[522,0.37109375],[523,0.422593355178833],[524,0.43494725227355957],[525,0.4396648406982422],[526,0.4548921585083008],[527,0.40890932083129883],[528,0.420881986618042],[529,0.5800793170928955],[530,0.5078015327453613],[531,0.4758634567260742],[532,0.4356703758239746],[533,0.4510056972503662],[534,0.4987452030181885],[535,0.5565037727355957],[536,0.4622533321380615],[537,0.4963796138763428],[538,0.28202152252197266],[539,0.3291501998901367],[540,0.3256094455718994],[541,0.3220665454864502],[542,0.30849575996398926],[543,0.3166239261627197],[544,0.2575342655181885],[545,0.2325453758239746],[546,0.23828744888305664],[547,0.3625447750091553],[548,0.3665347099304199],[549,0.37090015411376953],[550,0.3698923587799072],[551,0.3643617630004883],[552,0.35710740089416504],[553,0.36147165298461914],[554,0.36544132232666016],[555,0.35503458976745605],[556,0.2743337154388428],[557,0.21160483360290527],[558,0.20039582252502441],[559,0.2061448097229004],[560,0.20669269561767578],[561,0.20434045791625977],[562,0.25844907760620117],[563,0.20312786102294922],[564,0.17458748817443848],[565,0.17553305625915527],[566,0.17131900787353516],[567,0.22558975219726562],[568,0.2426457405090332],[569,0.24839091300964355],[570,0.24054288864135742],[571,0.25203418731689453],[572,0.27687692642211914],[573,0.29151058197021484],[574,0.3115518093109131],[575,0.346005916595459],[576,0.3402066230773926],[577,0.348130464553833],[578,0.35303544998168945],[579,0.21836233139038086],[580,0.24611401557922363],[581,0.2315828800201416],[582,0.2297656536102295],[583,0.2438945770263672],[584,0.2366795539855957],[585,0.2555711269378662],[586,0.2398216724395752],[587,0.26513147354125977],[588,0.26744604110717773],[589,0.26772475242614746],[590,0.21233510971069336],[591,0.2242109775543213],[592,0.2198963165283203],[593,0.2874181270599365],[594,0.2616727352142334],[595,0.21720314025878906],[596,0.23563432693481445],[597,0.2567436695098877],[598,0.25434398651123047],[599,0.2697455883026123],[600,0.2891428470611572],[601,0.25173163414001465],[602,0.261171817779541],[603,0.273712158203125],[604,0.32372283935546875],[605,0.30925869941711426],[606,0.3190336227416992],[607,0.3314931392669678],[608,0.30449795722961426],[609,0.31597065925598145],[610,0.32474231719970703],[611,0.1752324104309082],[612,0.13302373886108398],[613,0.13111495971679688],[614,0.13408613204956055],[615,0.1826176643371582],[616,0.2369556427001953],[617,0.2234210968017578],[618,0.26217222213745117],[619,0.26744508743286133],[620,0.2851133346557617],[621,0.2662835121154785],[622,0.2539083957672119],[623,0.18891453742980957],[624,0.18511199951171875],[625,0.2299489974975586],[626,0.25412845611572266],[627,0.21040678024291992],[628,0.25859689712524414],[629,0.35834431648254395],[630,0.43475937843322754],[631,0.3512253761291504],[632,0.32916855812072754],[633,0.29931020736694336],[634,0.23365545272827148],[635,0.1733262538909912],[636,0.26030898094177246],[637,0.3034803867340088],[638,0.29769110679626465],[639,0.28081798553466797],[640,0.48429369926452637],[641,0.3736433982849121],[642,0.5112550258636475],[643,0.40616846084594727],[644,0.3892202377319336],[645,0.4508194923400879],[646,0.47589993476867676],[647,0.5009937286376953],[648,0.4945366382598877],[649,0.5600576400756836],[650,0.513606071472168],[651,0.5034878253936768],[652,0.4242539405822754],[653,0.49647998809814453],[654,0.37667107582092285],[655,0.33550262451171875],[656,0.372997522354126],[657,0.3383965492248535],[658,0.2977027893066406],[659,0.30941009521484375],[660,0.21960926055908203],[661,0.1778888702392578],[662,0.17262697219848633],[663,0.2064952850341797],[664,0.2323319911956787],[665,0.31568074226379395],[666,0.347684383392334],[667,0.4210388660430908],[668,0.3816819190979004],[669,0.5434277057647705],[670,0.5421123504638672],[671,0.49202847480773926],[672,0.4624035358428955],[673,0.47999024391174316],[674,0.3891286849975586],[675,0.31122708320617676],[676,0.36092638969421387],[677,0.36167216300964355],[678,0.291301965713501],[679,0.35280513763427734],[680,0.2178359031677246],[681,0.16948962211608887],[682,0.16253066062927246],[683,0.1616373062133789],[684,0.17202448844909668],[685,0.16056275367736816],[686,0.16621637344360352],[687,0.17837190628051758],[688,0.17058396339416504],[689,0.19697356224060059],[690,0.2075653076171875],[691,0.19966602325439453],[692,0.22737932205200195],[693,0.1636524200439453],[694,0.16317367553710938],[695,0.17354178428649902],[696,0.1714620590209961],[697,0.18950295448303223],[698,0.27317214012145996],[699,0.24094057083129883],[700,0.25840044021606445],[701,0.24698138236999512],[702,0.257007360458374],[703,0.20950531959533691],[704,0.2514004707336426],[705,0.24744915962219238],[706,0.223524808883667],[707,0.21080517768859863],[708,0.2130434513092041],[709,0.1800525188446045],[710,0.1716299057006836],[711,0.1709423065185547],[712,0.14707636833190918],[713,0.130171537399292],[714,0.18437814712524414],[715,0.2508430480957031],[716,0.24694132804870605],[717,0.26613354682922363],[718,0.26014137268066406],[719,0.25528931617736816],[720,0.25844407081604004],[721,0.27738499641418457],[722,0.2716519832611084],[723,0.30882906913757324],[724,0.31288671493530273],[725,0.3044872283935547],[726,0.4355282783508301],[727,0.43894243240356445],[728,0.44075560569763184],[729,0.4291708469390869],[730,0.3930368423461914],[731,0.36948275566101074],[732,0.3830857276916504],[733,0.39414477348327637],[734,0.402660608291626],[735,0.4795355796813965],[736,0.37344908714294434],[737,0.2674081325531006],[738,0.25894856452941895],[739,0.2743246555328369],[740,0.195328950881958],[741,0.18576908111572266],[742,0.22389888763427734],[743,0.23608183860778809],[744,0.26328468322753906],[745,0.2597024440765381],[746,0.22489142417907715],[747,0.22317862510681152],[748,0.19420313835144043],[749,0.22340941429138184],[750,0.27275824546813965],[751,0.26856565475463867],[752,0.2954831123352051],[753,0.26243042945861816],[754,0.28015565872192383],[755,0.26909613609313965],[756,0.28334617614746094],[757,0.26602602005004883],[758,0.30933117866516113],[759,0.35945701599121094],[760,0.3084394931793213],[761,0.26908445358276367],[762,0.30129241943359375],[763,0.3175344467163086],[764,0.30868959426879883],[765,0.2607254981994629],[766,0.2727549076080322],[767,0.22170662879943848],[768,0.22288775444030762],[769,0.2232506275177002],[770,0.2933766841888428],[771,0.2552616596221924],[772,0.2639622688293457],[773,0.25929975509643555],[774,0.27256011962890625],[775,0.3886542320251465],[776,0.3511228561401367],[777,0.2902710437774658],[778,0.29326963424682617],[779,0.28902649879455566],[780,0.2844200134277344],[781,0.30673694610595703],[782,0.2873671054840088],[783,0.22769689559936523],[784,0.17120933532714844],[785,0.17744708061218262],[786,0.20597219467163086],[787,0.2573716640472412],[788,0.24228310585021973],[789,0.28328466415405273],[790,0.2451176643371582],[791,0.17729902267456055],[792,0.1861720085144043],[793,0.17176318168640137],[794,0.13147306442260742],[795,0.12627363204956055],[796,0.17760992050170898],[797,0.22602105140686035],[798,0.21442651748657227],[799,0.21489262580871582],[800,0.20734477043151855],[801,0.26294612884521484],[802,0.2547447681427002],[803,0.2064979076385498],[804,0.20913362503051758],[805,0.20409393310546875],[806,0.2706880569458008],[807,0.29010844230651855],[808,0.2863776683807373],[809,0.31320953369140625],[810,0.29085874557495117],[811,0.35034918785095215],[812,0.42519688606262207],[813,0.3842306137084961],[814,0.3853623867034912],[815,0.4401683807373047],[816,0.41191887855529785],[817,0.3737375736236572],[818,0.3934199810028076],[819,0.3695824146270752],[820,0.29496192932128906],[821,0.3619992733001709],[822,0.3240845203399658],[823,0.32906627655029297],[824,0.37756943702697754],[825,0.41887903213500977],[826,0.48586177825927734],[827,0.41597604751586914],[828,0.4160490036010742],[829,0.39486217498779297],[830,0.2772047519683838],[831,0.2681286334991455],[832,0.3335263729095459],[833,0.32113099098205566],[834,0.36502838134765625],[835,0.3577556610107422],[836,0.2598686218261719],[837,0.2573831081390381],[838,0.16387009620666504],[839,0.17151498794555664],[840,0.16269803047180176],[841,0.13910174369812012],[842,0.12044262886047363],[843,0.13165950775146484],[844,0.13141679763793945],[845,0.1301712989807129],[846,0.12759780883789062],[847,0.12599778175354004],[848,0.12292122840881348],[849,0.13836884498596191],[850,0.12596511840820312],[851,0.13179445266723633],[852,0.1329634189605713],[853,0.130568265914917],[854,0.134721040725708],[855,0.17605948448181152],[856,0.19582009315490723],[857,0.17856097221374512],[858,0.17752981185913086],[859,0.17682099342346191],[860,0.24952292442321777],[861,0.2996799945831299],[862,0.29088425636291504],[863,0.2796008586883545],[864,0.27544426918029785],[865,0.2931218147277832],[866,0.3231687545776367],[867,0.3401341438293457],[868,0.3778359889984131],[869,0.3606603145599365],[870,0.3802809715270996],[871,0.41097426414489746],[872,0.4307417869567871],[873,0.4422633647918701],[874,0.47777509689331055],[875,0.4928708076477051],[876,0.40518832206726074],[877,0.3862309455871582],[878,0.5323548316955566],[879,0.5469167232513428],[880,0.510305643081665],[881,0.5019829273223877],[882,0.5363624095916748],[883,0.5401434898376465],[884,0.4549415111541748],[885,0.4766349792480469],[886,0.5129740238189697],[887,0.33646178245544434],[888,0.36112093925476074],[889,0.41844749450683594],[890,0.380756139755249],[891,0.38592529296875],[892,0.40790557861328125],[893,0.35376572608947754],[894,0.4112513065338135],[895,0.3441281318664551],[896,0.3621392250061035],[897,0.30832982063293457],[898,0.3288846015930176],[899,0.3336336612701416],[900,0.3273892402648926],[901,0.3384420871734619],[902,0.40560197830200195],[903,0.42940211296081543],[904,0.44449377059936523],[905,0.3726208209991455],[906,0.35112786293029785],[907,0.3572700023651123],[908,0.298520565032959],[909,0.3457655906677246],[910,0.32911205291748047],[911,0.3775749206542969],[912,0.4304618835449219],[913,0.4366605281829834],[914,0.6133637428283691],[915,0.5839848518371582],[916,0.5456051826477051],[917,0.5304741859436035],[918,0.5487878322601318],[919,0.5747568607330322],[920,0.587836503982544],[921,0.47876667976379395],[922,0.4871227741241455],[923,0.45310139656066895],[924,0.47563815116882324],[925,0.4681689739227295],[926,0.45889830589294434],[927,0.5085787773132324],[928,0.4581465721130371],[929,0.486431360244751],[930,0.4527580738067627],[931,0.41948747634887695],[932,0.36123108863830566],[933,0.3645768165588379],[934,0.3320951461791992],[935,0.27097034454345703],[936,0.31800150871276855],[937,0.32703733444213867],[938,0.23625683784484863],[939,0.2813992500305176],[940,0.3497471809387207],[941,0.36783814430236816],[942,0.3672068119049072],[943,0.3790915012359619],[944,0.36480236053466797],[945,0.4133014678955078],[946,0.556344747543335],[947,0.7755434513092041],[948,0.6651053428649902],[949,0.7626008987426758],[950,0.5520651340484619],[951,0.6613571643829346],[952,0.5383892059326172],[953,0.4247004985809326],[954,0.3140444755554199],[955,0.315091609954834],[956,0.33115053176879883],[957,0.2782621383666992],[958,0.2848019599914551],[959,0.2527313232421875],[960,0.16927433013916016],[961,0.2050611972808838],[962,0.1720142364501953],[963,0.16199588775634766],[964,0.1702876091003418],[965,0.1802663803100586],[966,0.16694879531860352],[967,0.27675843238830566],[968,0.2806375026702881],[969,0.3327052593231201],[970,0.3209550380706787],[971,0.2839088439941406],[972,0.2805180549621582],[973,0.29881834983825684],[974,0.2844853401184082],[975,0.24906396865844727],[976,0.31122803688049316],[977,0.33690953254699707],[978,0.3386549949645996],[979,0.3478100299835205],[980,0.3415381908416748],[981,0.33393144607543945],[982,0.3561825752258301],[983,0.342360258102417],[984,0.3469045162200928],[985,0.3487098217010498],[986,0.33512210845947266],[987,0.3305537700653076],[988,0.6056602001190186],[989,0.6371912956237793],[990,0.6521992683410645],[991,0.6523206233978271],[992,0.6931705474853516],[993,0.7172420024871826],[994,0.6423063278198242],[995,0.4641275405883789],[996,0.4813346862792969],[997,0.45780038833618164],[998,0.4857954978942871],[999,0.4180920124053955],[1000,0.3620467185974121],[1001,0.4376637935638428],[1002,0.44721341133117676],[1003,0.44261860847473145],[1004,0.45639705657958984],[1005,0.44079113006591797],[1006,0.4508340358734131],[1007,0.44539880752563477],[1008,0.4387509822845459],[1009,0.518218994140625],[1010,0.5198240280151367],[1011,0.5436971187591553],[1012,0.5157971382141113],[1013,0.4575042724609375],[1014,0.3637685775756836],[1015,0.24284839630126953],[1016,0.22869277000427246],[1017,0.16550612449645996],[1018,0.20193219184875488],[1019,0.1800062656402588],[1020,0.22617268562316895],[1021,0.33301734924316406],[1022,0.26451587677001953],[1023,0.26864194869995117],[1024,0.26706528663635254],[1025,0.39147377014160156],[1026,0.35983848571777344],[1027,0.3514988422393799],[1028,0.3147611618041992],[1029,0.3470799922943115],[1030,0.3252429962158203],[1031,0.36834239959716797],[1032,0.1987311840057373],[1033,0.17437052726745605],[1034,0.16909265518188477],[1035,0.20615029335021973],[1036,0.21306109428405762],[1037,0.1979081630706787],[1038,0.2695152759552002],[1039,0.3547189235687256],[1040,0.364032506942749],[1041,0.3521230220794678],[1042,0.2835094928741455],[1043,0.33132410049438477],[1044,0.3373403549194336],[1045,0.3581831455230713],[1046,0.3252449035644531],[1047,0.31906723976135254],[1048,0.3123505115509033],[1049,0.25443601608276367],[1050,0.24361824989318848],[1051,0.21010231971740723],[1052,0.20673108100891113],[1053,0.20455527305603027],[1054,0.2583498954772949],[1055,0.25158071517944336],[1056,0.250821590423584],[1057,0.24371767044067383],[1058,0.22491669654846191],[1059,0.2155609130859375],[1060,0.17650365829467773],[1061,0.17456531524658203],[1062,0.17766475677490234],[1063,0.17633748054504395],[1064,0.19259142875671387],[1065,0.17812061309814453],[1066,0.18605279922485352],[1067,0.17725849151611328],[1068,0.1810932159423828],[1069,0.13229966163635254],[1070,0.19167351722717285],[1071,0.1801929473876953],[1072,0.1885206699371338],[1073,0.21320843696594238],[1074,0.22295427322387695],[1075,0.3281822204589844],[1076,0.30757594108581543],[1077,0.2913084030151367],[1078,0.325411319732666],[1079,0.2927238941192627],[1080,0.28997206687927246],[1081,0.29116368293762207]],"reported_metrics":{"ADE":0.5587965606209528,"FDE":1.1381949186325073,"MHD":3.071232318878174,"speed_error":0.37326878044215545,"heading_error":40.81245962143868,"collision_pct":2.17,"collision_pct_ped_ped":5.91,"collision_pct_ped_veh":0.21},"protocol":{"sample_size":5,"selection":"best of five by lowest ADE computed against ground truth (ORACLE selection)","model_mode":"net.eval() is never called in test.py; the model contains nn.Dropout(0.5), so dropout is active in the prediction path","eval_called_in_printed_test_py":false,"future_information":"baseline sample() receives ground-truth sequences and future actor lists (Phase 2 Task 4 audit)","checkpoint":"epoch 4 of the cell-51 run, staged in Store_Results/model/CollisionGrid (cell 59)"},"metric_note":"cell 60 prints speed error 0.1393 and heading error 1665.6569 in the per-iteration lines but 0.37326878044215545 and 40.81245962143868 in the final summary; the archived/reported values are the final-summary ones. The difference is recorded, not reconciled."},"environment_findings":["Two Python environments exist: the Colab default kernel (Python 3.13.16, PyTorch 2.14.1+cu130; cells 0, 11) and /content/pcg-env (Python 3.11.18, PyTorch 2.2.1+cu121; cells 19, 21, 28). Installing torch 2.2.1 into the default environment failed (cell 10).","TWO complete 5-epoch trainings are in the saved outputs: cell 49 (execution 54, explicit pcg-env) and cell 51 (execution 56, plain !python). Both write the same Store_Results file names, so the later run overwrote the earlier run's checkpoints and log. The archived log_curve.txt matches the cell-51 run, so the archived checkpoints come from the default-interpreter run, not from pcg-env.","The first command in cell 49 was malformed ('--num_epochs 5%cd ...') and failed with an argparse usage error; only its second command trained.","Cell 41 claims 'DataLoader works inside Python 3.11!' but its shell command failed (unmatched quote) and the remaining lines ran in the KERNEL (Python 3.13.16, PyTorch 2.14.1+cu130): that check does not show anything about pcg-env.","The 1-epoch run (cell 45) used pcg-env; the 5-epoch archived run and the test (cell 60) used plain !python, whose interpreter is not recorded. The grid-calculation time is 31.9 s (cell 45) and 30.6 s (cell 49) in pcg-env versus 13.4 s in cell 51.","The two 5-epoch runs are not identical (unseeded): first-batch loss 2.029 vs 1.917 and different epoch mean losses, so they are two independent samples of the same procedure.","Store_Results was replaced by a symlink to Drive (cell 47), so training wrote its checkpoints through the Drive mount."],"not_reconstructable":["GPU utilisation, GPU memory (allocated, reserved, peak), CPU utilisation, RAM and process memory during training or testing: never recorded (only two idle nvidia-smi snapshots exist, cells 1 and 53)","per-epoch wall-clock time (only an estimate from checkpoint mtimes and from summed batch times)","data-loading time per epoch and checkpoint-save time","CUDA-synchronised per-batch time (the printed time/batch comes from time.time() around the batch, which includes several host-device synchronisations but was not an explicit measurement)","single-sample inference latency, model-only latency, p50/p90/p95/p99 of one prediction, inference throughput and inference peak memory (the historical test time covers 5 samples + scoring + oracle selection)","the interpreter actually used by plain `!python` in cells 51 and 60","checkpoints, config and log of the overwritten cell-49 run (only its printed output survives)","any random seed (none was set; runs are not reproducible)"]}'''
RETRO = json.loads(RETRO_JSON)

def _per_epoch(rows):
    by = {}
    for gb, ep, loss, t in rows:
        by.setdefault(ep, []).append((loss, t))
    out = []
    for ep in sorted(by):
        l = [x[0] for x in by[ep]]; t = [x[1] for x in by[ep]]
        out.append({"epoch": ep, "n_batches": len(t), "mean_batch_loss": sum(l) / len(l), "sum_of_printed_batch_times_s": sum(t), "batch_time": ml.describe(t)})
    return out

RETRO_SUMMARY = {"label": "RETROSPECTIVE (extracted from saved notebook outputs; not a new measurement)", "source_notebook_sha256": RETRO["source_notebook_sha256"],
                 "runs": {}, "test_pass": {}, "not_reconstructable": []}
for key, run in RETRO["training_runs"].items():
    rows = run["batches"]
    times = [r[3] for r in rows]; losses = [r[2] for r in rows]
    s = {"notebook_cell_index": run["cell_index"], "execution_count": run["execution_count"], "command": run["command"], "interpreter_evidence": run["interpreter_evidence"],
         "batches_logged": len(rows), "epochs": sorted({r[1] for r in rows}), "grid_calculation_s_reported": run["grid_calculation_s"],
         "total_training_s_reported": run["total_training_s"], "sum_of_printed_batch_times_s": sum(times),
         "reported_total_minus_sum_of_batch_times_s": (run["total_training_s"] - sum(times)) if run["total_training_s"] is not None else None,
         "batch_time_printed": ml.describe(times), "batch_time_printed_excluding_first": ml.describe(times[1:]), "first_batch_time_s": times[0] if times else None,
         "loss": {"first": losses[0] if losses else None, "last": losses[-1] if losses else None}, "per_epoch": _per_epoch(rows),
         "throughput_batches_per_s_from_total": (len(rows) / run["total_training_s"]) if run["total_training_s"] else None,
         "throughput_sequences_per_s_from_total": (len(rows) * RETRO["batch_size"] / run["total_training_s"]) if run["total_training_s"] else None,
         "throughput_note": "total includes batch creation, plotting and checkpoint saving; batch_size=%d" % RETRO["batch_size"]}
    RETRO_SUMMARY["runs"][key] = s
    with open(os.path.join(MEAS_DIR, "retro_train_batches_%s.csv" % key), "x", newline="", encoding="utf-8") as fh:
        import csv as _csv
        w = _csv.writer(fh); w.writerow(["global_batch", "epoch", "train_loss", "printed_time_per_batch_s"]); w.writerows(rows)

# cross-check of the archived epoch losses (log_curve.txt) against the means of the printed batch losses of each run
ARCH = RETRO["archived_epoch_losses_log_curve"]
RETRO_SUMMARY["archived_log_curve_crosscheck"] = {"archived_epoch_losses": ARCH, "source": RETRO["archived_epoch_losses_source"], "max_abs_diff_to_run_mean_batch_loss": {}}
for key, s in RETRO_SUMMARY["runs"].items():
    if len(s["per_epoch"]) == len(ARCH):
        RETRO_SUMMARY["archived_log_curve_crosscheck"]["max_abs_diff_to_run_mean_batch_loss"][key] = max(abs(e["mean_batch_loss"] - a) for e, a in zip(s["per_epoch"], ARCH))
_d = RETRO_SUMMARY["archived_log_curve_crosscheck"]["max_abs_diff_to_run_mean_batch_loss"]
RETRO_SUMMARY["archived_log_curve_crosscheck"]["closest_run"] = min(_d, key=_d.get) if _d else None
RETRO_SUMMARY["archived_log_curve_crosscheck"]["note"] = ("printed batch losses have 3 decimals, so a difference below ~1e-3 means the archived log_curve.txt was produced by that run; "
                                                         "the other run's checkpoints/log were overwritten (same file names)")

# epoch durations derived from checkpoint modification times (cell 54 output) - an ESTIMATE with ~1 s resolution
mt = RETRO["checkpoint_mtimes_cell54"]
import datetime as _dt
_ts = [_dt.datetime.strptime(x, "%Y-%m-%d %H:%M:%S") for x in mt["epoch_0_to_4"]]
_gaps = [(_ts[i + 1] - _ts[i]).total_seconds() for i in range(len(_ts) - 1)]
RETRO_SUMMARY["epoch_duration_from_checkpoint_mtimes"] = {"label": "RETROSPECTIVE ESTIMATE", "source": mt["source"], "mtimes": mt["epoch_0_to_4"], "epoch_1_to_4_durations_s": _gaps,
    "epoch_0_duration_s_residual": (RETRO["training_runs"]["five_epoch_cell51_default_python"]["total_training_s"] - sum(_gaps)),
    "caveats": ["mtime resolution is 1 s and Drive/FUSE can delay mtimes", "epoch 0 is the total reported training time minus epochs 1-4 (it includes the first-batch warm-up)",
                "applies to the run whose checkpoints were archived (the cell-51 run, see the archived-log cross-check)"]}

# historical test pass: per-trajectory time (5 samples + all metric computations + selection)
tt = [r[1] for r in RETRO["test_pass"]["per_trajectory"]]
RETRO_SUMMARY["test_pass"] = {"command": RETRO["test_pass"]["command"], "interpreter_evidence": RETRO["test_pass"]["interpreter_evidence"], "trajectories": len(tt),
    "per_trajectory_time_s": ml.describe(tt), "sum_s": sum(tt), "throughput_trajectories_per_s": (len(tt) / sum(tt)) if tt else None,
    "WHAT_THIS_TIME_INCLUDES": "5 sampled rollouts + ground-truth scoring (ADE/FDE/MHD/speed/heading/collisions) + best-of-five selection + result bookkeeping, per trajectory",
    "NOT": "single-prediction inference latency; NOT video or real-time performance", "protocol": RETRO["test_pass"]["protocol"], "reported_metrics_final_summary": RETRO["test_pass"]["reported_metrics"],
    "metric_note": RETRO["test_pass"]["metric_note"]}
with open(os.path.join(MEAS_DIR, "retro_test_trajectory_times.csv"), "x", newline="", encoding="utf-8") as fh:
    import csv as _csv
    w = _csv.writer(fh); w.writerow(["trajectory", "time_s_5samples_plus_scoring"]); w.writerows(RETRO["test_pass"]["per_trajectory"])

RETRO_SUMMARY["environment_findings"] = RETRO["environment_findings"]
RETRO_SUMMARY["not_reconstructable"] = RETRO["not_reconstructable"]
_sha = ml.write_new_json(os.path.join(MEAS_DIR, "retrospective_summary.json"), RETRO_SUMMARY)
print("wrote retrospective_summary.json  sha256 =", _sha)
for key, s in RETRO_SUMMARY["runs"].items():
    print("\n[%s] cell %s | %d batches | total %.2f s | grid %.2f s | sum(batch times) %.2f s" % (key, s["notebook_cell_index"], s["batches_logged"], s["total_training_s_reported"],
          s["grid_calculation_s_reported"], s["sum_of_printed_batch_times_s"]))
    b = s["batch_time_printed"]
    print("   batch time: median %.3f  p90 %.3f  p95 %.3f  max %.3f  | first batch %.3f s" % (b["p50"], b["p90"], b["p95"], b["max"], s["first_batch_time_s"]))
    print("   per-epoch mean loss:", [round(e["mean_batch_loss"], 4) for e in s["per_epoch"]])
print("\narchived log_curve closest run:", RETRO_SUMMARY["archived_log_curve_crosscheck"]["closest_run"], _d)
print("epoch durations from checkpoint mtimes (epochs 1-4):", RETRO_SUMMARY["epoch_duration_from_checkpoint_mtimes"]["epoch_1_to_4_durations_s"])
t = RETRO_SUMMARY["test_pass"]["per_trajectory_time_s"]
print("test per-trajectory time (5 samples + scoring): median %.3f p90 %.3f p95 %.3f p99 %.3f s over %d trajectories" % (t["p50"], t["p90"], t["p95"], t["p99"], t["count"]))
print("\nThis is NOT single-sample inference latency. Not reconstructable from the saved outputs:")
for x in RETRO_SUMMARY["not_reconstructable"]:
    print("  -", x)


wrote retrospective_summary.json  sha256 = 4a20165ae759a1e1f0503512f642850b7db87443ac0e9b729f1258a87a1316f1

[one_epoch_cell45_pcg_env] cell 45 | 238 batches | total 151.35 s | grid 31.91 s | sum(batch times) 150.28 s
   batch time: median 0.601  p90 0.809  p95 0.879  max 1.033  | first batch 0.968 s
   per-epoch mean loss: [-0.7937]

[five_epoch_cell49_pcg_env] cell 49 | 1190 batches | total 748.46 s | grid 30.57 s | sum(batch times) 742.64 s
   batch time: median 0.590  p90 0.802  p95 0.858  max 1.109  | first batch 0.974 s
   per-epoch mean loss: [-0.9283, -1.6565, -1.866, -1.9841, -2.0715]

[five_epoch_cell51_default_python] cell 51 | 1190 batches | total 748.94 s | grid 13.41 s | sum(batch times) 743.62 s
   batch time: median 0.599  p90 0.785  p95 0.831  max 1.737  | first batch 1.737 s
   per-epoch mean loss: [-0.9362, -1.6138, -1.7975, -1.9391, -2.0328]

archived log_curve closest run: five_epoch_cell51_default_python {'five_epoch_cell49_pcg_env': 0.06851182854372784, 'five_epo

## Benchmark approval gate

The environment fingerprint, artifact manifest, and retrospective report should now exist under the new `MEAS_DIR` on Google Drive.

Review `artifact_manifest.json` before proceeding. In particular, verify:
- The epoch-4 checkpoint in the Drive backup is `MATCH`.
- The experiment configuration in the Drive backup is `MATCH`.
- The checkpoint/configuration selected by the benchmark are the expected archived files.
- No unexplained mismatch exists in the experiment artifacts.

If the critical files match, set the next cell to `True` and run the benchmark cell. If anything is missing or mismatched, stop and share the manifest instead.


In [8]:
RUN_INFERENCE_BENCHMARK = False  # Change to True ONLY after reviewing the artifact manifest.

In [9]:
# ==============================================================================================
# SINGLE-SAMPLE PREDICTOR BENCHMARK (runs only if RUN_INFERENCE_BENCHMARK = True)
# ==============================================================================================
# SCOPE: PCG trajectory-PREDICTOR-ONLY, sample_size = 1. NOT end-to-end video performance and NOT deployment-valid:
# the baseline path still consumes future actor information, and the historical test.py runs with dropout active (net.eval() is never called).
# Why a separate benchmark: the historical test.py timing (cell 60) covers 5 sampled rollouts + ground-truth scoring + best-of-five oracle
# selection per trajectory, so it cannot be read as the cost of ONE prediction. test.py is neither modified nor run here; the baseline
# `sample` function is imported from it unchanged and its per-trajectory preparation steps are replicated in the benchmark module.
BENCH_WORKDIR = os.path.join(WORK_ROOT, "inference")
_ck_candidates = [os.path.join(DRIVE_BACKUP, "SOCIALLSTM_lstm_model_4.tar"), os.path.join(STORE, "model", "SOCIALLSTM_lstm_model_4.tar")]
_cf_candidates = [os.path.join(DRIVE_BACKUP, "config.pkl"), os.path.join(STORE, "model", "config.pkl")]
BENCH_CKPT = next((p for p in _ck_candidates if os.path.isfile(p)), _ck_candidates[0])
BENCH_CONFIG = next((p for p in _cf_candidates if os.path.isfile(p)), _cf_candidates[0])
EXPECTED_CKPT = PREV["ckpt4"][0]
EXPECTED_CONFIG = PREV["experiment_config"][0]
_bp = bench_mod.preflight(REPO, BENCH_WORKDIR, BENCH_CKPT, BENCH_CONFIG)
# The exact files selected for benchmarking must match the archived hashes.
if os.path.isfile(BENCH_CKPT) and ml.sha256_file(BENCH_CKPT).lower() != EXPECTED_CKPT.lower():
    _bp.append("selected epoch-4 checkpoint does not match the archived SHA-256")
if os.path.isfile(BENCH_CONFIG) and ml.sha256_file(BENCH_CONFIG).lower() != EXPECTED_CONFIG.lower():
    _bp.append("selected experiment config does not match the archived SHA-256")
print("checkpoint :", BENCH_CKPT)
print("config     :", BENCH_CONFIG)
print("Pre-flight :", "OK" if not _bp else "PROBLEMS -> benchmark will NOT start")
for _p in _bp:
    print("   -", _p)

BENCH_SUMMARY = None
BENCH_OUTPUT_DIR = None
if not RUN_INFERENCE_BENCHMARK:
    print("\nRUN_INFERENCE_BENCHMARK is False: nothing was run.")
elif _bp:
    print("\nNOT started: safe isolation could not be guaranteed.")
else:
    import torch
    _hw = "%s | %s | torch %s" % (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only", platform.platform(), torch.__version__)
    os.makedirs(WORK_ROOT, exist_ok=True)
    # Run in THIS kernel (its interpreter and torch are what get recorded). To benchmark under /content/pcg-env instead, run
    # The benchmark uses the active Colab kernel and records its exact interpreter.
    BENCH_SUMMARY = bench_mod.run_benchmark(REPO, BENCH_WORKDIR, MEAS_RUN_ID, BENCH_CKPT, BENCH_CONFIG, epoch=4, warmup=BENCH_WARMUP,
                                            max_trajectories=BENCH_MAX_TRAJECTORIES, seed=BENCH_SEED, expected_checkpoint_sha256=EXPECTED_CKPT,
                                            expected_config_sha256=EXPECTED_CONFIG, hardware_label=_hw)
    BENCH_OUTPUT_DIR = os.path.join(MEAS_DIR, "inference_benchmark")
    shutil.copytree(os.path.join(BENCH_WORKDIR, "measurements"), BENCH_OUTPUT_DIR)
    L = BENCH_SUMMARY["latency_s"]
    print("\nSCOPE:", BENCH_SUMMARY["scope"])
    print("hashes match previously recorded: checkpoint =", BENCH_SUMMARY["hashes"]["checkpoint_matches_expected"], "| config =", BENCH_SUMMARY["hashes"]["config_matches_expected"])
    for k in ("pipeline", "model_forward_only_sum_per_trajectory"):
        d = L[k]
        print("%-42s p50 %.4f  p90 %.4f  p95 %.4f  p99 %.4f s  (n=%d)" % (k, d["p50_s"], d["p90_s"], d["p95_s"], d["p99_s"], d["count"]))
    print("throughput: %.2f sequences/s | peak GPU memory (MiB): %s" % (BENCH_SUMMARY["throughput_sequences_per_s"]["pipeline_pass"], BENCH_SUMMARY["peak_gpu_memory_mib"]))
    print("copied to:", BENCH_OUTPUT_DIR)


checkpoint : /content/drive/MyDrive/PCG_Experiments/PCG_T4_5_Epoch_Run/SOCIALLSTM_lstm_model_4.tar
config     : /content/drive/MyDrive/PCG_Experiments/PCG_T4_5_Epoch_Run/config.pkl
Pre-flight : OK

RUN_INFERENCE_BENCHMARK is False: nothing was run.


In [ ]:
# ==============================================================================================
# VISUALISATIONS (historical evidence plus any new benchmark data)
# ==============================================================================================
import csv as _csv
import matplotlib
import matplotlib.pyplot as plt

PLOT_DIR = os.path.join(MEAS_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=False)
PLOTS_WRITTEN, PLOTS_SKIPPED = [], {}
RUN_LABELS = {"five_epoch_cell49_pcg_env": "cell 49 run (pcg-env, overwritten later)", "five_epoch_cell51_default_python": "cell 51 run (default python, ARCHIVED)"}

def _save(fig, name):
    p = os.path.join(PLOT_DIR, name)
    fig.savefig(p, dpi=130, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    PLOTS_WRITTEN.append(name)

def _read_csv(path):
    with open(path, newline="", encoding="utf-8") as fh:
        return list(_csv.DictReader(fh))

def _num(rows, key):
    out = []
    for r in rows:
        try:
            v = float(r[key])
            if v == v:
                out.append(v)
        except (KeyError, TypeError, ValueError):
            pass
    return out

NEW_TRAIN = None  # no training is run by this measurement notebook
NEW_BENCH = BENCH_OUTPUT_DIR if "BENCH_OUTPUT_DIR" in globals() and BENCH_OUTPUT_DIR else None

# 1. training loss over batches --------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 4))
for key, run in RETRO["training_runs"].items():
    if key.startswith("five_epoch"):
        ax.plot([r[0] for r in run["batches"]], [r[2] for r in run["batches"]], lw=0.7, label="RETROSPECTIVE " + RUN_LABELS.get(key, key))
if NEW_TRAIN and os.path.isfile(os.path.join(NEW_TRAIN, "training_batch_log.csv")):
    rows = _read_csv(os.path.join(NEW_TRAIN, "training_batch_log.csv"))
    ax.plot(_num(rows, "global_batch")[:len(_num(rows, "train_loss"))], _num(rows, "train_loss"), lw=0.7, label="NEW monitored run")
ax.set_xlabel("global batch"); ax.set_ylabel("train loss (negative log-likelihood)"); ax.set_title("Training loss over batches (238 batches per epoch)"); ax.legend(fontsize=8)
_save(fig, "01_training_loss_over_batches.png")

# 2. batch-time distribution ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4))
for key, run in RETRO["training_runs"].items():
    if key.startswith("five_epoch"):
        ax.hist([r[3] for r in run["batches"]], bins=60, alpha=0.5, label="RETROSPECTIVE " + RUN_LABELS.get(key, key) + " (printed time/batch)")
if NEW_TRAIN and os.path.isfile(os.path.join(NEW_TRAIN, "training_batch_log.csv")):
    ax.hist(_num(_read_csv(os.path.join(NEW_TRAIN, "training_batch_log.csv")), "sync_batch_time_s"), bins=60, alpha=0.5, label="NEW run (CUDA-synchronised)")
ax.set_xlabel("seconds per batch"); ax.set_ylabel("batches"); ax.set_title("Training batch-time distribution"); ax.legend(fontsize=8)
_save(fig, "02_batch_time_distribution.png")

# 3. per-epoch duration -------------------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4))
w = 0.25; shown = 0
for i, (key, s) in enumerate(RETRO_SUMMARY["runs"].items()):
    if key.startswith("five_epoch"):
        ax.bar([e["epoch"] + (shown - 0.5) * w for e in s["per_epoch"]], [e["sum_of_printed_batch_times_s"] for e in s["per_epoch"]], width=w, label="RETRO sum of printed batch times: " + RUN_LABELS.get(key, key))
        shown += 1
_mt = RETRO_SUMMARY["epoch_duration_from_checkpoint_mtimes"]
ax.scatter([1, 2, 3, 4], _mt["epoch_1_to_4_durations_s"], color="k", zorder=3, label="RETRO estimate from checkpoint mtimes (epochs 1-4, ~1 s resolution)")
if NEW_TRAIN and os.path.isfile(os.path.join(NEW_TRAIN, "training_epoch_summary.csv")):
    er = _read_csv(os.path.join(NEW_TRAIN, "training_epoch_summary.csv"))
    ax.scatter(_num(er, "epoch"), _num(er, "epoch_wall_s"), marker="D", color="r", zorder=4, label="NEW run epoch wall-clock")
ax.set_xlabel("epoch"); ax.set_ylabel("seconds"); ax.set_title("Per-epoch training duration")
ax.set_ylim(0, ax.get_ylim()[1] * 1.5); ax.legend(fontsize=7, loc="upper center")
_save(fig, "03_per_epoch_duration.png")

# 4/5. GPU utilisation and memory over time: ONLY from a new monitored run's telemetry -----------------------
_rl = os.path.join(NEW_TRAIN, "training_resource_log.csv") if NEW_TRAIN else None
if _rl and os.path.isfile(_rl):
    rows = _read_csv(_rl)
    t = _num(rows, "t_s")
    util = _num(rows, "gpu_util_pct")
    if util:
        fig, ax = plt.subplots(figsize=(10, 3.5)); ax.plot([float(r["t_s"]) for r in rows if r.get("gpu_util_pct") not in (None, "")], util, lw=0.8)
        ax.set_xlabel("seconds since run start"); ax.set_ylabel("GPU utilisation (%)"); ax.set_title("GPU utilisation during training (device-wide samples)")
        _save(fig, "04_gpu_utilization_over_time.png")
    else:
        PLOTS_SKIPPED["04_gpu_utilization_over_time"] = "telemetry file has no GPU utilisation values"
    mem = _num(rows, "gpu_mem_used_mib")
    if mem or _num(rows, "torch_allocated_mib"):
        fig, ax = plt.subplots(figsize=(10, 3.5))
        for col, lab in (("gpu_mem_used_mib", "device memory used (nvidia-smi/NVML)"), ("torch_allocated_mib", "torch allocated"), ("torch_reserved_mib", "torch reserved")):
            ys = [(float(r["t_s"]), float(r[col])) for r in rows if r.get(col) not in (None, "")]
            if ys:
                ax.plot([a for a, _ in ys], [b for _, b in ys], lw=0.8, label=lab)
        ax.set_xlabel("seconds since run start"); ax.set_ylabel("MiB"); ax.set_title("GPU memory during training"); ax.legend(fontsize=8)
        _save(fig, "05_gpu_memory_over_time.png")
    else:
        PLOTS_SKIPPED["05_gpu_memory_over_time"] = "telemetry file has no memory values"
else:
    PLOTS_SKIPPED["04_gpu_utilization_over_time"] = "no GPU telemetry exists (the historical run recorded none); requires a NEW monitored run"
    PLOTS_SKIPPED["05_gpu_memory_over_time"] = "no GPU memory telemetry exists; requires a NEW monitored run"

# 6. inference latency distribution -----------------------------------------------------------------------------
if NEW_BENCH and os.path.isfile(os.path.join(NEW_BENCH, "inference_latency_samples.csv")):
    rows = _read_csv(os.path.join(NEW_BENCH, "inference_latency_samples.csv"))
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(_num(rows, "pipeline_s"), bins=60, alpha=0.6, label="predictor pipeline, 1 sample (NEW)")
    ax.hist(_num(rows, "model_forward_s"), bins=60, alpha=0.6, label="model forwards only (NEW)")
    ax.set_xlabel("seconds per trajectory"); ax.set_ylabel("trajectories"); ax.set_title("PCG predictor-only latency (NOT end-to-end video)"); ax.legend(fontsize=8)
    _save(fig, "06_inference_latency_distribution.png")
else:
    PLOTS_SKIPPED["06_inference_latency_distribution"] = "no single-sample benchmark has been run (set RUN_INFERENCE_BENCHMARK = True)"
# the historical per-trajectory test time is plotted separately and labelled, because it is NOT inference latency
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist([r[1] for r in RETRO["test_pass"]["per_trajectory"]], bins=60)
ax.set_xlabel("seconds per trajectory"); ax.set_ylabel("trajectories")
ax.set_title("RETROSPECTIVE historical test time per trajectory\n(5 samples + ground-truth scoring + best-of-5 oracle selection; NOT single-prediction latency)", fontsize=9)
_save(fig, "06b_retro_test_time_NOT_inference_latency.png")

print("plots written:", PLOTS_WRITTEN)
print("plots skipped (no data):")
for k, v in PLOTS_SKIPPED.items():
    print("  -", k, ":", v)


In [ ]:
# ==============================================================================================
# OUTPUT INDEX - what this measurement session produced (hashes of every file under MEAS_DIR)
# ==============================================================================================
INDEX = {"measurement_run_id": MEAS_RUN_ID, "created_utc": ml.utc_now_iso(), "output_directory": MEAS_DIR, "files": [],
         "ran_monitored_training": False,
         "ran_inference_benchmark": bool(RUN_INFERENCE_BENCHMARK),
         "plots_skipped": PLOTS_SKIPPED,
         "still_to_measure": [] }
INDEX["still_to_measure"].append("training-resource telemetry was intentionally not collected; this notebook does not retrain the baseline")
if not INDEX["ran_inference_benchmark"]:
    INDEX["still_to_measure"].append("single-sample predictor latency (p50/p90/p95/p99, throughput, peak GPU memory): requires Part E on the GPU runtime")
INDEX["still_to_measure"] += ["end-to-end video latency (detection, tracking, decoding, forecasting): outside this notebook",
                              "non-oracle accuracy (single sample, no future actor information): needs the corrected evaluator (Phase 3)"]
for d, _, fs in os.walk(MEAS_DIR):
    for f in sorted(fs):
        p = os.path.join(d, f)
        if os.path.basename(p) == "measurement_outputs_index.json":
            continue
        INDEX["files"].append({"path": os.path.relpath(p, MEAS_DIR), "bytes": os.path.getsize(p), "sha256": ml.sha256_file(p)})
INDEX["files"].sort(key=lambda r: r["path"])
_sha = ml.write_new_json(os.path.join(MEAS_DIR, "measurement_outputs_index.json"), INDEX)
print("Output directory:", MEAS_DIR)
for r in INDEX["files"]:
    print("  %-62s %10d bytes  %s" % (r["path"], r["bytes"], r["sha256"][:12]))
print("\nStill to measure:")
for x in INDEX["still_to_measure"]:
    print("  -", x)
